# p53 Rescue Mutation Discovery — Full Simulation (Google Colab)

This notebook runs the **p53-proteoMgCAD** computational protein design pipeline to discover
second-site rescue mutations that restore tumor suppressor function in mutant p53.

**Fully self-contained** — upload this notebook (plus data files) to Google Colab and run.

**Pipeline overview:**
1. Install dependencies & write package source code
2. Upload data files (DMS CSV + oracle weights)
3. Load models (ESM-2 protein language model + functional oracle)
4. Build scenario matrix (8 cancer hotspots × 3 delivery methods)
5. Run campaign (Pass A screening → Pass B deep refinement)
6. Analyze & visualize results (Top-30 shortlist, clinical impact, heatmaps)

**Required data files** (upload when prompted):
- `p53_DMS_Giacomelli_2018.csv` — Deep mutational scanning data (~1.9 MB)
- `functional_oracle.pt` — Trained oracle model weights (~29 MB)
- `p53_wt.pdb` — Wild-type structure (optional, for contact penalties)

## 0. Install Dependencies & Build Package

In [ ]:
# Install required packages (takes ~2 min on Colab)
!pip install -q torch transformers pandas numpy matplotlib seaborn scipy pyarrow

import os, sys

# Runtime guards
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Dependencies installed successfully")

### 0a. Write Package Source Code

The following cells write the `p53cad` package source files to disk using `%%writefile`.
You can collapse these cells — they just recreate the package structure.

In [ ]:
import os

# Create package directory structure
for d in [
    'p53cad', 'p53cad/core', 'p53cad/data', 'p53cad/engine',
    'p53cad/analysis', 'p53cad/results',
    'data/raw', 'data/models', 'data/campaigns',
]:
    os.makedirs(d, exist_ok=True)
    init_path = os.path.join(d, '__init__.py')
    if d.startswith('p53cad') and not os.path.exists(init_path):
        open(init_path, 'w').close()

print("Package directory structure created")

In [ ]:
%%writefile p53cad/core/logging.py
import logging
import os
import sys
from logging.handlers import RotatingFileHandler
from pathlib import Path
from typing import Optional, Union


DEFAULT_LOG_FILE = Path("logs/p53cad_workflow.log")


def setup_logging(level=logging.INFO, log_file: Optional[Union[Path, str]] = None) -> Path:
    """
    Sets up logging for the p53cad package.
    Always logs to console and a persistent workflow file.
    """
    env_log_file = os.getenv("P53CAD_LOG_FILE", "").strip()
    if env_log_file:
        resolved_log_file = Path(env_log_file).expanduser()
    elif log_file is not None:
        resolved_log_file = Path(log_file).expanduser()
    else:
        resolved_log_file = DEFAULT_LOG_FILE

    resolved_log_file.parent.mkdir(parents=True, exist_ok=True)

    handlers = [
        logging.StreamHandler(sys.stdout),
        RotatingFileHandler(
            resolved_log_file,
            maxBytes=10 * 1024 * 1024,
            backupCount=5,
            encoding="utf-8",
        ),
    ]

    logging.basicConfig(
        level=level,
        format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        handlers=handlers,
        force=True,
    )
    logging.captureWarnings(True)
    return resolved_log_file

def get_logger(name: str) -> logging.Logger:
    return logging.getLogger(name)


In [ ]:
%%writefile p53cad/core/runtime.py
from __future__ import annotations

import importlib.util
import os
import platform
import shutil
from typing import Any, Dict, Optional

from p53cad.core.logging import get_logger


def _has_module(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False


def bootstrap_runtime(seed: Optional[int] = None) -> None:
    """
    Apply process-level runtime guards before heavy ML/scientific imports.
    """
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    if seed is not None:
        os.environ.setdefault("PYTHONHASHSEED", str(int(seed)))


def get_runtime_capabilities() -> Dict[str, Any]:
    """
    Collect lightweight runtime capability probes without forcing heavy imports.
    """
    caps: Dict[str, Any] = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "kmp_duplicate_lib_ok": os.getenv("KMP_DUPLICATE_LIB_OK", ""),
        "mps_fallback_enabled": os.getenv("PYTORCH_ENABLE_MPS_FALLBACK", ""),
        "torch_installed": _has_module("torch"),
        "streamlit_installed": _has_module("streamlit"),
        "plotly_installed": _has_module("plotly"),
        "rdkit_installed": _has_module("rdkit"),
        "openmm_installed": _has_module("openmm"),
        "openff_installed": _has_module("openff.toolkit"),
        "openmmforcefields_installed": _has_module("openmmforcefields"),
        "meeko_installed": _has_module("meeko"),
        "vina_py_installed": _has_module("vina"),
        "vina_cli_available": shutil.which("vina") is not None,
    }

    if caps["torch_installed"]:
        try:
            import torch  # pylint: disable=import-outside-toplevel

            caps["torch_version"] = torch.__version__
            caps["mps_available"] = bool(torch.backends.mps.is_available())
            caps["cuda_available"] = bool(torch.cuda.is_available())
        except Exception as exc:  # pragma: no cover - defensive path
            caps["torch_probe_error"] = str(exc)
            caps["mps_available"] = False
            caps["cuda_available"] = False
    else:
        caps["mps_available"] = False
        caps["cuda_available"] = False

    if _has_module("transformers"):
        try:
            import transformers  # pylint: disable=import-outside-toplevel

            caps["transformers_version"] = transformers.__version__
        except Exception as exc:  # pragma: no cover - defensive path
            caps["transformers_probe_error"] = str(exc)

    return caps


def log_runtime_capabilities(logger_name: str = "p53cad.runtime") -> Dict[str, Any]:
    """
    Log a single compact capability line for workflow integrity debugging.
    """
    logger = get_logger(logger_name)
    caps = get_runtime_capabilities()
    logger.info(
        "Runtime capabilities | python=%s torch=%s mps=%s cuda=%s transformers=%s rdkit=%s "
        "vina_cli=%s openmm=%s openff=%s openmmforcefields=%s",
        caps.get("python"),
        caps.get("torch_version", "n/a"),
        caps.get("mps_available"),
        caps.get("cuda_available"),
        caps.get("transformers_version", "n/a"),
        caps.get("rdkit_installed"),
        caps.get("vina_cli_available"),
        caps.get("openmm_installed"),
        caps.get("openff_installed"),
        caps.get("openmmforcefields_installed"),
    )
    return caps


In [ ]:
%%writefile p53cad/data/dms.py
from __future__ import annotations

import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import Optional, Tuple, List, Dict
from p53cad.core.logging import get_logger

logger = get_logger(__name__)

# =============================================================================
# DATA SOURCES
# =============================================================================
# Full Giacomelli 2018 saturation mutagenesis data (~8260 variants)
# Paper: https://www.nature.com/articles/s41588-018-0204-y
#
# Score columns available:
# - A549_p53WT_Nutlin-3_Z-score: Main functional score (p53 activation assay)
# - A549_p53NULL_Nutlin-3_Z-score: Control (no p53)
# - A549_p53NULL_Etoposide_Z-score: DNA damage response
# =============================================================================

# Wild-type p53 sequence (393 amino acids)
P53_WT = (
    "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP"
    "DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAK"
    "SVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHE"
    "RCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNS"
    "SCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELP"
    "PGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPG"
    "GSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD"
)

# Default path to full saturation DMS data
DEFAULT_DMS_PATH = Path(__file__).parent.parent.parent / "data" / "raw" / "p53_DMS_Giacomelli_2018_FULL.csv"
# Fallback to old cell-line data if full data not available
LEGACY_DMS_PATH = Path(__file__).parent.parent.parent / "data" / "raw" / "p53_DMS_Giacomelli_2018.csv"


def parse_single_mutation(mutation_str: str) -> Optional[Tuple[str, int, str]]:
    """
    Parse a single mutation string like 'R175H' into (wt_aa, position, mut_aa).

    Returns None for complex mutations (frameshift, nonsense, splice, deletions).
    """
    mutation_str = mutation_str.strip()

    # Skip complex mutations
    if any(x in mutation_str for x in ['fs', '*', 'del', 'ins', 'splice', '>']):
        return None

    # Match standard format: A123B (single letter, number, single letter)
    match = re.match(r'^([A-Z])(\d+)([A-Z])$', mutation_str)
    if match:
        wt_aa, pos_str, mut_aa = match.groups()
        pos = int(pos_str)
        # Validate position is within p53 sequence
        if 1 <= pos <= len(P53_WT):
            return (wt_aa, pos, mut_aa)

    return None


def parse_mutation(mutation: str) -> List[Tuple[str, int, str]]:
    """
    Parse mutation string which may contain multiple mutations (comma-separated).

    Returns list of (wt_aa, position, mut_aa) tuples for valid point mutations.
    Returns empty list if any mutation is invalid/complex.
    """
    # Handle quoted strings from CSV
    mutation = mutation.strip('"').strip()

    # Split by comma for double/multiple mutants
    parts = [p.strip() for p in mutation.split(',')]

    parsed = []
    for part in parts:
        result = parse_single_mutation(part)
        if result is None:
            return []  # If any part is invalid, skip entire entry
        parsed.append(result)

    return parsed


def apply_mutation(wt_seq: str, mutation: str) -> Optional[str]:
    """
    Applies a mutation string (e.g. R175H or 'R175H, T211I') to WT sequence.

    Returns None if mutation is complex (fs, *, del) or invalid.
    """
    parsed = parse_mutation(mutation)
    if not parsed:
        return None

    seq = list(wt_seq)

    for wt_aa, pos, mut_aa in parsed:
        idx = pos - 1  # Convert 1-indexed to 0-indexed

        # Validate WT amino acid matches (with some tolerance for data errors)
        if idx < len(seq):
            # Note: Some mutations in the dataset may have WT mismatches due to
            # different reference sequences or data entry errors. We still apply them.
            seq[idx] = mut_aa
        else:
            return None

    return "".join(seq)


def load_dms_data(file_path: Optional[Path | str] = None, score_column: str = "A549_p53WT_Nutlin-3_Z-score") -> pd.DataFrame:
    """
    Load Giacomelli 2018 p53 DMS data.

    Supports two formats:
    1. Full saturation format (8260 entries): Allele, AA_wt, AA_variant, Position, Z-scores
    2. Legacy cell-line format (529 entries): mutation, score

    The main functional score is A549_p53WT_Nutlin-3_Z-score which measures
    p53 transcriptional activity. Negative = loss of function, positive = functional.

    Args:
        file_path: Path to DMS CSV. Uses default path if not provided.
        score_column: Which Z-score column to use (default: A549_p53WT_Nutlin-3_Z-score)

    Returns:
        DataFrame with columns: mutation, score, pos, wt, alt, sequence
    """
    path = Path(file_path) if file_path else DEFAULT_DMS_PATH

    # Try paths in order: specified > default full > legacy > parent dir
    if not path.exists():
        if LEGACY_DMS_PATH.exists():
            path = LEGACY_DMS_PATH
            logger.info("Using legacy DMS data (cell-line format)")
        else:
            alt_path = Path(__file__).parent.parent.parent.parent / "p53_DMS_Giacomelli_2018.csv"
            if alt_path.exists():
                path = alt_path
            else:
                raise FileNotFoundError(
                    f"DMS data not found. Searched: {DEFAULT_DMS_PATH}, {LEGACY_DMS_PATH}"
                )

    logger.info(f"Loading DMS data from {path}")
    df = pd.read_csv(path)

    # Detect format based on columns
    is_saturation_format = 'Allele' in df.columns and 'Position' in df.columns
    is_legacy_format = 'mutation' in df.columns and 'score' in df.columns

    if is_saturation_format:
        return _load_saturation_format(df, score_column)
    elif is_legacy_format:
        return _load_legacy_format(df)
    else:
        raise ValueError(f"Unknown DMS file format. Columns: {df.columns.tolist()}")


def _load_saturation_format(df: pd.DataFrame, score_column: str) -> pd.DataFrame:
    """Load full saturation mutagenesis format (8260 entries).

    NOTE: Nutlin-3 Z-scores are INVERTED - positive = loss of function.
    We negate scores so negative = pathogenic (matches clinical convention).
    """
    logger.info(f"Detected saturation format with {len(df)} entries")

    if score_column not in df.columns:
        # Try to find a suitable score column
        score_cols = [c for c in df.columns if 'Z-score' in c or 'score' in c.lower()]
        if score_cols:
            score_column = score_cols[0]
            logger.warning(f"Score column not found, using: {score_column}")
        else:
            raise ValueError(f"No score column found. Available: {df.columns.tolist()}")

    processed_data = []
    skipped = 0

    for _, row in df.iterrows():
        allele = str(row.get('Allele', ''))
        wt_aa = str(row.get('AA_wt', ''))
        mut_aa = str(row.get('AA_variant', ''))
        pos = int(row.get('Position', 0))
        raw_score = row.get(score_column)

        # Skip rows with missing data or stop codons (Z)
        if pd.isna(raw_score) or not allele or mut_aa == 'Z' or pos == 0:
            skipped += 1
            continue

        # NEGATE score: Nutlin-3 positive = loss of function, we want negative = pathogenic
        score = -float(raw_score)

        # Build mutation string (e.g., "R175H")
        mutation = f"{wt_aa}{pos}{mut_aa}"

        # Generate mutant sequence
        sequence = apply_mutation(P53_WT, mutation)
        if not sequence:
            skipped += 1
            continue

        processed_data.append({
            'mutation': mutation,
            'score': score,
            'pos': pos,
            'wt': wt_aa,
            'alt': mut_aa,
            'sequence': sequence,
            'n_mutations': 1,
            'is_hotspot': pos in [175, 248, 273, 245, 249, 282, 220],
            'allele': allele  # Keep original allele notation
        })

    result_df = pd.DataFrame(processed_data)
    logger.info(f"Loaded {len(result_df)} valid saturation DMS entries (skipped {skipped})")
    logger.info(f"  - Positions covered: {result_df['pos'].nunique()}")
    logger.info(f"  - Score range: [{result_df['score'].min():.2f}, {result_df['score'].max():.2f}]")

    return result_df


def _load_legacy_format(df: pd.DataFrame) -> pd.DataFrame:
    """Load legacy cell-line format (529 entries)."""
    logger.info(f"Detected legacy format with {len(df)} entries")

    processed_data = []
    skipped_complex = 0
    skipped_invalid = 0

    for _, row in df.iterrows():
        mutation = str(row['mutation'])
        score = float(row['score'])

        parsed = parse_mutation(mutation)
        if not parsed:
            skipped_complex += 1
            continue

        if len(parsed) == 1:
            wt_aa, pos, mut_aa = parsed[0]
            sequence = apply_mutation(P53_WT, mutation)

            if sequence:
                processed_data.append({
                    'mutation': mutation,
                    'score': score,
                    'pos': pos,
                    'wt': wt_aa,
                    'alt': mut_aa,
                    'sequence': sequence,
                    'n_mutations': 1,
                    'is_hotspot': pos in [175, 248, 273, 245, 249, 282, 220]
                })
            else:
                skipped_invalid += 1
        else:
            sequence = apply_mutation(P53_WT, mutation)
            if sequence:
                wt_aa, pos, mut_aa = parsed[0]
                processed_data.append({
                    'mutation': mutation,
                    'score': score,
                    'pos': pos,
                    'wt': wt_aa,
                    'alt': mut_aa,
                    'sequence': sequence,
                    'n_mutations': len(parsed),
                    'is_hotspot': any(p[1] in [175, 248, 273, 245, 249, 282, 220] for p in parsed)
                })

    result_df = pd.DataFrame(processed_data)
    logger.info(f"Loaded {len(result_df)} valid legacy DMS entries")
    logger.info(f"  - Single mutations: {len(result_df[result_df['n_mutations'] == 1])}")
    logger.info(f"  - Multiple mutations: {len(result_df[result_df['n_mutations'] > 1])}")
    logger.info(f"  - Skipped complex (fs/*/del): {skipped_complex}")
    logger.info(f"  - Skipped invalid: {skipped_invalid}")
    logger.info(f"  - Hotspot mutations: {len(result_df[result_df['is_hotspot']])}")

    return result_df


def hydrate_sequences(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds a 'sequence' column to a dataframe with a 'mutation' column.
    For pre-processed data that already has sequences, returns as-is.
    """
    if 'sequence' in df.columns:
        return df

    hydrated = []
    for _, row in df.iterrows():
        seq = apply_mutation(P53_WT, str(row["mutation"]))
        if seq:
            new_row = row.to_dict()
            new_row["sequence"] = seq
            hydrated.append(new_row)

    return pd.DataFrame(hydrated)


def get_mutation_statistics(df: pd.DataFrame) -> dict:
    """
    Compute statistics about the DMS dataset.
    """
    stats = {
        'total_entries': len(df),
        'unique_positions': df['pos'].nunique() if 'pos' in df.columns else 0,
        'score_mean': df['score'].mean(),
        'score_std': df['score'].std(),
        'score_min': df['score'].min(),
        'score_max': df['score'].max(),
        'pathogenic_count': len(df[df['score'] < -0.5]),
        'neutral_count': len(df[(df['score'] >= -0.5) & (df['score'] <= 0.5)]),
        'functional_count': len(df[df['score'] > 0.5]),
    }

    if 'is_hotspot' in df.columns:
        stats['hotspot_count'] = len(df[df['is_hotspot']])
        stats['hotspot_mean_score'] = df[df['is_hotspot']]['score'].mean()

    return stats


def load_double_mutant_data(path: Optional[Path] = None) -> pd.DataFrame:
    """Load double-mutant DMS data from external source.

    Expected CSV columns: pos1, aa1, pos2, aa2, score (or mutation column
    with comma-separated pairs like 'R175H,N268D').

    Falls back to ``data/raw/p53_DMS_double_mutants.csv`` when *path* is None.
    Returns an empty DataFrame if the file is not found (graceful degradation).
    """
    default_path = Path(__file__).parent.parent.parent / "data" / "raw" / "p53_DMS_double_mutants.csv"
    path = Path(path) if path else default_path
    if not path.exists():
        logger.info("Double-mutant DMS file not found at %s; using additive fallback", path)
        return pd.DataFrame()

    logger.info("Loading double-mutant DMS data from %s", path)
    df = pd.read_csv(path)

    # Normalise columns
    if "mutation" in df.columns and "pos1" not in df.columns:
        # Parse 'R175H,N268D' format
        rows = []
        score_col = next((c for c in df.columns if "score" in c.lower() or "z" in c.lower()), None)
        if score_col is None:
            logger.warning("No score column found in double-mutant CSV")
            return pd.DataFrame()
        for _, row in df.iterrows():
            parts = parse_mutation(str(row["mutation"]))
            if len(parts) == 2:
                (_, p1, a1), (_, p2, a2) = parts
                rows.append({"pos1": p1, "aa1": a1, "pos2": p2, "aa2": a2, "score": float(row[score_col])})
        df = pd.DataFrame(rows)

    required = {"pos1", "aa1", "pos2", "aa2", "score"}
    if not required.issubset(set(df.columns)):
        logger.warning("Double-mutant CSV missing columns: %s", required - set(df.columns))
        return pd.DataFrame()

    logger.info("Loaded %d double-mutant DMS entries", len(df))
    return df


def get_pairwise_epistasis_lookup(
    dms_df: Optional[pd.DataFrame] = None,
    double_df: Optional[pd.DataFrame] = None,
) -> Dict[tuple, float]:
    """Build (pos1, aa1, pos2, aa2) → epistasis score lookup.

    Uses real double-mutant data when available.  Falls back to an additive
    model: epistasis = observed_double - (single_A + single_B).  When no
    double-mutant data exists, returns an empty dict (callers should fall
    back to structural heuristics).

    Returns
    -------
    dict
        Keys are ``(pos1, aa1, pos2, aa2)`` tuples (positions 1-indexed).
        Values are Z-score differences (positive = synergistic).
    """
    if double_df is None:
        double_df = load_double_mutant_data()
    if double_df.empty:
        return {}

    # Load single-mutant data for additive baseline
    if dms_df is None:
        try:
            dms_df = get_dms_data()
        except FileNotFoundError:
            dms_df = pd.DataFrame()

    single_lookup: Dict[tuple, float] = {}
    if not dms_df.empty and "pos" in dms_df.columns and "alt" in dms_df.columns and "score" in dms_df.columns:
        for _, row in dms_df.iterrows():
            single_lookup[(int(row["pos"]), str(row["alt"]))] = float(row["score"])

    result: Dict[tuple, float] = {}
    for _, row in double_df.iterrows():
        p1, a1 = int(row["pos1"]), str(row["aa1"]).upper()
        p2, a2 = int(row["pos2"]), str(row["aa2"]).upper()
        observed = float(row["score"])

        # Additive expectation (sum of singles)
        s1 = single_lookup.get((p1, a1), 0.0)
        s2 = single_lookup.get((p2, a2), 0.0)
        epistasis = observed - (s1 + s2)

        # Store both orderings for easy lookup
        result[(p1, a1, p2, a2)] = epistasis
        result[(p2, a2, p1, a1)] = epistasis

    logger.info("Built pairwise epistasis lookup: %d pairs", len(result) // 2)
    return result


# Convenience function for quick loading
def get_dms_data() -> pd.DataFrame:
    """Load DMS data with default settings."""
    return load_dms_data()


# =============================================================================
# PHYSICS-BASED FALLBACK SCORING
# =============================================================================
# When DMS data is unavailable, use amino acid properties to estimate impact

# Kyte-Doolittle hydropathy scale
HYDROPATHY = {
    'I': 4.5, 'V': 4.2, 'L': 3.8, 'F': 2.8, 'C': 2.5, 'M': 1.9, 'A': 1.8,
    'G': -0.4, 'T': -0.7, 'S': -0.8, 'W': -0.9, 'Y': -1.3, 'P': -1.6,
    'H': -3.2, 'E': -3.5, 'Q': -3.5, 'D': -3.5, 'N': -3.5, 'K': -3.9, 'R': -4.5
}

# Amino acid molecular volumes (Å³)
AA_VOLUME = {
    'G': 60, 'A': 88, 'S': 89, 'C': 108, 'D': 111, 'P': 112, 'N': 114,
    'T': 116, 'E': 138, 'V': 140, 'Q': 143, 'H': 153, 'M': 162, 'I': 166,
    'L': 166, 'K': 168, 'R': 173, 'F': 189, 'Y': 193, 'W': 227
}

# Amino acid charges at physiological pH
AA_CHARGE = {
    'K': 1, 'R': 1, 'H': 0.1,  # Basic (positive)
    'D': -1, 'E': -1,          # Acidic (negative)
    'A': 0, 'C': 0, 'F': 0, 'G': 0, 'I': 0, 'L': 0, 'M': 0,
    'N': 0, 'P': 0, 'Q': 0, 'S': 0, 'T': 0, 'V': 0, 'W': 0, 'Y': 0
}

# BLOSUM62 substitution matrix (symmetric)
# Higher = more conservative substitution
BLOSUM62 = {
    ('A', 'A'): 4, ('A', 'R'): -1, ('A', 'N'): -2, ('A', 'D'): -2, ('A', 'C'): 0,
    ('A', 'Q'): -1, ('A', 'E'): -1, ('A', 'G'): 0, ('A', 'H'): -2, ('A', 'I'): -1,
    ('A', 'L'): -1, ('A', 'K'): -1, ('A', 'M'): -1, ('A', 'F'): -2, ('A', 'P'): -1,
    ('A', 'S'): 1, ('A', 'T'): 0, ('A', 'W'): -3, ('A', 'Y'): -2, ('A', 'V'): 0,
    ('R', 'R'): 5, ('R', 'N'): 0, ('R', 'D'): -2, ('R', 'C'): -3, ('R', 'Q'): 1,
    ('R', 'E'): 0, ('R', 'G'): -2, ('R', 'H'): 0, ('R', 'I'): -3, ('R', 'L'): -2,
    ('R', 'K'): 2, ('R', 'M'): -1, ('R', 'F'): -3, ('R', 'P'): -2, ('R', 'S'): -1,
    ('R', 'T'): -1, ('R', 'W'): -3, ('R', 'Y'): -2, ('R', 'V'): -3,
    ('N', 'N'): 6, ('N', 'D'): 1, ('N', 'C'): -3, ('N', 'Q'): 0, ('N', 'E'): 0,
    ('N', 'G'): 0, ('N', 'H'): 1, ('N', 'I'): -3, ('N', 'L'): -3, ('N', 'K'): 0,
    ('N', 'M'): -2, ('N', 'F'): -3, ('N', 'P'): -2, ('N', 'S'): 1, ('N', 'T'): 0,
    ('N', 'W'): -4, ('N', 'Y'): -2, ('N', 'V'): -3,
    ('D', 'D'): 6, ('D', 'C'): -3, ('D', 'Q'): 0, ('D', 'E'): 2, ('D', 'G'): -1,
    ('D', 'H'): -1, ('D', 'I'): -3, ('D', 'L'): -4, ('D', 'K'): -1, ('D', 'M'): -3,
    ('D', 'F'): -3, ('D', 'P'): -1, ('D', 'S'): 0, ('D', 'T'): -1, ('D', 'W'): -4,
    ('D', 'Y'): -3, ('D', 'V'): -3,
    ('C', 'C'): 9, ('C', 'Q'): -3, ('C', 'E'): -4, ('C', 'G'): -3, ('C', 'H'): -3,
    ('C', 'I'): -1, ('C', 'L'): -1, ('C', 'K'): -3, ('C', 'M'): -1, ('C', 'F'): -2,
    ('C', 'P'): -3, ('C', 'S'): -1, ('C', 'T'): -1, ('C', 'W'): -2, ('C', 'Y'): -2,
    ('C', 'V'): -1,
    ('Q', 'Q'): 5, ('Q', 'E'): 2, ('Q', 'G'): -2, ('Q', 'H'): 0, ('Q', 'I'): -3,
    ('Q', 'L'): -2, ('Q', 'K'): 1, ('Q', 'M'): 0, ('Q', 'F'): -3, ('Q', 'P'): -1,
    ('Q', 'S'): 0, ('Q', 'T'): -1, ('Q', 'W'): -2, ('Q', 'Y'): -1, ('Q', 'V'): -2,
    ('E', 'E'): 5, ('E', 'G'): -2, ('E', 'H'): 0, ('E', 'I'): -3, ('E', 'L'): -3,
    ('E', 'K'): 1, ('E', 'M'): -2, ('E', 'F'): -3, ('E', 'P'): -1, ('E', 'S'): 0,
    ('E', 'T'): -1, ('E', 'W'): -3, ('E', 'Y'): -2, ('E', 'V'): -2,
    ('G', 'G'): 6, ('G', 'H'): -2, ('G', 'I'): -4, ('G', 'L'): -4, ('G', 'K'): -2,
    ('G', 'M'): -3, ('G', 'F'): -3, ('G', 'P'): -2, ('G', 'S'): 0, ('G', 'T'): -2,
    ('G', 'W'): -2, ('G', 'Y'): -3, ('G', 'V'): -3,
    ('H', 'H'): 8, ('H', 'I'): -3, ('H', 'L'): -3, ('H', 'K'): -1, ('H', 'M'): -2,
    ('H', 'F'): -1, ('H', 'P'): -2, ('H', 'S'): -1, ('H', 'T'): -2, ('H', 'W'): -2,
    ('H', 'Y'): 2, ('H', 'V'): -3,
    ('I', 'I'): 4, ('I', 'L'): 2, ('I', 'K'): -3, ('I', 'M'): 1, ('I', 'F'): 0,
    ('I', 'P'): -3, ('I', 'S'): -2, ('I', 'T'): -1, ('I', 'W'): -3, ('I', 'Y'): -1,
    ('I', 'V'): 3,
    ('L', 'L'): 4, ('L', 'K'): -2, ('L', 'M'): 2, ('L', 'F'): 0, ('L', 'P'): -3,
    ('L', 'S'): -2, ('L', 'T'): -1, ('L', 'W'): -2, ('L', 'Y'): -1, ('L', 'V'): 1,
    ('K', 'K'): 5, ('K', 'M'): -1, ('K', 'F'): -3, ('K', 'P'): -1, ('K', 'S'): 0,
    ('K', 'T'): -1, ('K', 'W'): -3, ('K', 'Y'): -2, ('K', 'V'): -2,
    ('M', 'M'): 5, ('M', 'F'): 0, ('M', 'P'): -2, ('M', 'S'): -1, ('M', 'T'): -1,
    ('M', 'W'): -1, ('M', 'Y'): -1, ('M', 'V'): 1,
    ('F', 'F'): 6, ('F', 'P'): -4, ('F', 'S'): -2, ('F', 'T'): -2, ('F', 'W'): 1,
    ('F', 'Y'): 3, ('F', 'V'): -1,
    ('P', 'P'): 7, ('P', 'S'): -1, ('P', 'T'): -1, ('P', 'W'): -4, ('P', 'Y'): -3,
    ('P', 'V'): -2,
    ('S', 'S'): 4, ('S', 'T'): 1, ('S', 'W'): -3, ('S', 'Y'): -2, ('S', 'V'): -2,
    ('T', 'T'): 5, ('T', 'W'): -2, ('T', 'Y'): -2, ('T', 'V'): 0,
    ('W', 'W'): 11, ('W', 'Y'): 2, ('W', 'V'): -3,
    ('Y', 'Y'): 7, ('Y', 'V'): -1,
    ('V', 'V'): 4,
}

# p53 DNA-binding domain residues (critical for function)
P53_DBD_RANGE = (94, 292)  # Core DNA-binding domain

# Known critical functional positions in p53
P53_CRITICAL_POSITIONS = {
    # Zinc coordination sites (critical for structure)
    176, 179, 238, 242,
    # DNA-contacting residues
    248, 273, 280, 241, 282,
    # L2 loop (involved in DNA binding)
    163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175,
    # L3 loop (involved in DNA binding)
    237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250,
    # LSH motif
    119, 120, 121
}


def get_blosum62_score(aa1: str, aa2: str) -> int:
    """Get BLOSUM62 substitution score (symmetric lookup)."""
    if aa1 == aa2:
        return BLOSUM62.get((aa1, aa1), 0)
    # Try both orderings since matrix is symmetric
    return BLOSUM62.get((aa1, aa2), BLOSUM62.get((aa2, aa1), -4))


def physics_based_score(mutation: str) -> Dict[str, float]:
    """
    Calculate physics-based mutation impact score when DMS data unavailable.

    Uses amino acid properties to estimate functional impact:
    - Hydrophobicity change (burial disruption)
    - Size change (steric clashes)
    - Charge change (electrostatic effects)
    - BLOSUM62 evolutionary score (conservation)
    - Position in critical regions

    Returns dict with component scores and combined estimate.
    Score interpretation: negative = likely deleterious, positive = likely tolerated
    """
    parsed = parse_single_mutation(mutation)
    if not parsed:
        return {'score': 0.0, 'confidence': 0.0, 'method': 'invalid_mutation'}

    wt_aa, pos, mut_aa = parsed

    # Component calculations
    hydro_wt = HYDROPATHY.get(wt_aa, 0)
    hydro_mut = HYDROPATHY.get(mut_aa, 0)
    hydro_change = abs(hydro_mut - hydro_wt)  # Larger = more disruptive

    vol_wt = AA_VOLUME.get(wt_aa, 120)
    vol_mut = AA_VOLUME.get(mut_aa, 120)
    vol_change = abs(vol_mut - vol_wt)  # Larger = more steric clash risk

    charge_wt = AA_CHARGE.get(wt_aa, 0)
    charge_mut = AA_CHARGE.get(mut_aa, 0)
    charge_change = abs(charge_mut - charge_wt)  # Charge reversal is severe

    blosum = get_blosum62_score(wt_aa, mut_aa)  # Higher = more conservative

    # Position-based modifiers
    in_dbd = P53_DBD_RANGE[0] <= pos <= P53_DBD_RANGE[1]
    is_critical = pos in P53_CRITICAL_POSITIONS

    # Calculate combined score (calibrated to DMS scale: -2 to +2)
    # More negative = more likely deleterious

    # Start with BLOSUM as base (normalized from -4 to +11 range)
    base_score = (blosum + 4) / 15 * 2 - 1  # Maps to roughly -1 to +1

    # Penalize large property changes
    hydro_penalty = -0.15 * hydro_change  # Max ~1.35 penalty
    vol_penalty = -0.005 * vol_change      # Max ~0.8 penalty
    charge_penalty = -0.5 * charge_change  # Max 1.0 penalty for charge reversal

    # Position modifiers
    position_modifier = 0.0
    if is_critical:
        position_modifier = -0.5  # Critical positions more sensitive
    elif in_dbd:
        position_modifier = -0.2  # DBD generally more sensitive

    # Combined score
    combined = base_score + hydro_penalty + vol_penalty + charge_penalty + position_modifier
    combined = np.clip(combined, -2.0, 1.5)  # Clamp to reasonable DMS range

    # Confidence based on how many property changes occur
    n_changes = sum([
        hydro_change > 2.0,
        vol_change > 30,
        charge_change > 0,
        blosum < 0
    ])
    confidence = 0.3 + 0.15 * n_changes  # Higher confidence when clear signal

    return {
        'score': float(combined),
        'confidence': float(confidence),
        'method': 'physics_based',
        'components': {
            'blosum62': blosum,
            'hydropathy_change': float(hydro_change),
            'volume_change': float(vol_change),
            'charge_change': float(charge_change),
            'in_dbd': in_dbd,
            'is_critical': is_critical,
            'base_score': float(base_score),
            'hydro_penalty': float(hydro_penalty),
            'vol_penalty': float(vol_penalty),
            'charge_penalty': float(charge_penalty),
            'position_modifier': float(position_modifier)
        }
    }


def get_mutation_score(mutation: str, dms_df: Optional[pd.DataFrame] = None) -> Dict[str, float]:
    """
    Get mutation score, preferring DMS data but falling back to physics-based estimate.

    Args:
        mutation: Mutation string like 'R175H'
        dms_df: Optional pre-loaded DMS dataframe

    Returns:
        Dict with score, confidence, and method used
    """
    # Try DMS lookup first
    if dms_df is not None and not dms_df.empty:
        match = dms_df[dms_df['mutation'] == mutation]
        if not match.empty:
            return {
                'score': float(match['score'].iloc[0]),
                'confidence': 1.0,  # High confidence for experimental data
                'method': 'dms_experimental'
            }

    # Fall back to physics-based score
    return physics_based_score(mutation)


def get_coverage_report(mutations: List[str], dms_df: Optional[pd.DataFrame] = None) -> Dict:
    """
    Generate a detailed coverage report showing DMS vs physics-based scoring.

    Useful for understanding what data IS available vs estimated.
    """
    if dms_df is None:
        try:
            dms_df = get_dms_data()
        except FileNotFoundError:
            dms_df = pd.DataFrame()

    dms_mutations = set(dms_df['mutation'].tolist()) if not dms_df.empty else set()
    dms_positions = set(dms_df['pos'].tolist()) if 'pos' in dms_df.columns else set()

    results = []
    dms_count = 0
    physics_count = 0

    for mut in mutations:
        parsed = parse_single_mutation(mut)
        if not parsed:
            continue

        wt_aa, pos, mut_aa = parsed

        if mut in dms_mutations:
            score_info = {'score': float(dms_df[dms_df['mutation'] == mut]['score'].iloc[0]),
                         'confidence': 1.0, 'method': 'dms_experimental'}
            dms_count += 1
        else:
            score_info = physics_based_score(mut)
            physics_count += 1

        results.append({
            'mutation': mut,
            'position': pos,
            'wt_aa': wt_aa,
            'mut_aa': mut_aa,
            'score': score_info['score'],
            'confidence': score_info['confidence'],
            'method': score_info['method'],
            'position_has_dms': pos in dms_positions
        })

    return {
        'mutations': results,
        'summary': {
            'total': len(results),
            'dms_experimental': dms_count,
            'physics_estimated': physics_count,
            'dms_coverage': dms_count / max(len(results), 1),
            'positions_with_any_dms': sum(1 for r in results if r['position_has_dms']),
            'dms_dataset_size': len(dms_df),
            'dms_positions_covered': len(dms_positions)
        },
        'data_sources': {
            'current_file': 'Giacomelli 2018 cell line data (~367 entries)',
            'full_saturation': 'Download from MaveDB: https://www.mavedb.org/',
            'alternative': 'Nature Genetics 2024: https://www.nature.com/articles/s41588-024-02039-4'
        }
    }


def print_data_source_info() -> None:
    """Print helpful information about getting more DMS data."""
    info = """
╔══════════════════════════════════════════════════════════════════════╗
║                    p53 DMS DATA SOURCES                              ║
╠══════════════════════════════════════════════════════════════════════╣
║ CURRENT DATA: Giacomelli 2018 cell line (~367 entries, 88 positions) ║
║                                                                      ║
║ FOR FULL SATURATION MUTAGENESIS DATA (~8000+ variants):              ║
║                                                                      ║
║ 1. MaveDB (Recommended)                                              ║
║    → https://www.mavedb.org/                                         ║
║    → Search "TP53" or "p53 Giacomelli"                               ║
║    → Download CSV with all variant effects                           ║
║                                                                      ║
║ 2. Nature Genetics 2024 - Deep CRISPR Study (Most Comprehensive)     ║
║    → https://www.nature.com/articles/s41588-024-02039-4              ║
║    → Supplementary data contains all TP53 functional data            ║
║                                                                      ║
║ 3. Original Giacomelli 2018 Paper                                    ║
║    → https://www.nature.com/articles/s41588-018-0204-y               ║
║    → Supplementary Tables in paper                                   ║
║                                                                      ║
║ Place downloaded CSV in: data/raw/p53_DMS_Giacomelli_2018.csv        ║
╚══════════════════════════════════════════════════════════════════════╝
"""
    print(info)


def get_position_statistics(dms_df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """
    Get per-position statistics from available DMS data.

    Shows what positions HAVE data and their score distributions.
    """
    if dms_df is None:
        try:
            dms_df = get_dms_data()
        except FileNotFoundError:
            return pd.DataFrame()

    if dms_df.empty or 'pos' not in dms_df.columns:
        return pd.DataFrame()

    # Group by position
    pos_stats = dms_df.groupby('pos').agg({
        'score': ['mean', 'std', 'min', 'max', 'count'],
        'mutation': 'nunique'
    }).round(3)

    pos_stats.columns = ['mean_score', 'std_score', 'min_score', 'max_score',
                         'n_measurements', 'n_unique_mutations']
    pos_stats = pos_stats.reset_index()

    # Add domain annotation
    pos_stats['in_dbd'] = pos_stats['pos'].apply(
        lambda p: P53_DBD_RANGE[0] <= p <= P53_DBD_RANGE[1]
    )
    pos_stats['is_critical'] = pos_stats['pos'].apply(
        lambda p: p in P53_CRITICAL_POSITIONS
    )

    return pos_stats.sort_values('pos')


In [ ]:
%%writefile p53cad/engine/latent.py
from __future__ import annotations

import time
import torch
import torch.nn.functional as F
from transformers import EsmForMaskedLM, EsmTokenizer
from typing import Optional, List, Tuple, Callable, TypeVar
from pathlib import Path
import os
import logging
from p53cad.core.logging import get_logger

T = TypeVar("T")


def _load_with_retry(
    load_fn: Callable[[], T],
    description: str,
    logger: logging.Logger,
    max_retries: int = 3,
) -> T:
    """
    Attempt *load_fn* up to *max_retries* times with exponential back-off.

    Back-off schedule: 2**( 2*attempt - 1 ) seconds  ->  2 s, 8 s, 32 s
    for attempts 1, 2, 3 respectively.

    Parameters
    ----------
    load_fn : callable
        Zero-argument callable that returns the loaded artifact.
    description : str
        Human-readable label used in log messages (e.g. "ESM-2 tokenizer").
    logger : logging.Logger
        Logger instance for warnings / errors.
    max_retries : int
        Total number of attempts before giving up.

    Returns
    -------
    T
        Whatever *load_fn* returns on success.

    Raises
    ------
    RuntimeError
        After all retries are exhausted.
    """
    last_exc: Optional[Exception] = None
    for attempt in range(1, max_retries + 1):
        try:
            return load_fn()
        except Exception as exc:
            last_exc = exc
            if attempt < max_retries:
                backoff = 2 ** (2 * attempt - 1)  # 2, 8, 32
                logger.warning(
                    "Attempt %d/%d to load %s failed: %s  -- retrying in %ds",
                    attempt,
                    max_retries,
                    description,
                    exc,
                    backoff,
                )
                time.sleep(backoff)
            else:
                logger.error(
                    "All %d attempts to load %s failed.", max_retries, description
                )

    raise RuntimeError(
        f"Failed to load {description} after {max_retries} attempts "
        f"(last error: {last_exc}).  "
        "Please check your network connection, ensure the HuggingFace Hub is "
        "reachable, or authenticate with `huggingface-cli login`."
    )

class ManifoldEmbedder:
    """
    Handles interactions with the ESM-2 Protein Language Model.
    Focuses on encoding sequences into latent space and decoding them back.
    """
    def __init__(self, model_name: str = "facebook/esm2_t33_650M_UR50D", device: Optional[str] = None, lora_path: Optional[str] = None):
        self.logger = get_logger(__name__)
        self.model_name = model_name
        
        if device is None:
            if torch.backends.mps.is_available():
                self.device = torch.device("mps")
            elif torch.cuda.is_available():
                self.device = torch.device("cuda")
            else:
                self.device = torch.device("cpu")
        else:
            self.device = torch.device(device)
            
        self.logger.info(f"Loading ESM-2 model {model_name} on {self.device}...")

        # --- Offline-mode gate ---------------------------------------------------
        offline = os.environ.get("TRANSFORMERS_OFFLINE", "0") == "1"
        if offline:
            self.logger.info(
                "TRANSFORMERS_OFFLINE=1 detected -- loading from local cache only "
                "(no download retries)."
            )

        try:
            # --- Tokenizer --------------------------------------------------------
            if offline:
                self.tokenizer = EsmTokenizer.from_pretrained(model_name, local_files_only=True)
            else:
                self.tokenizer = _load_with_retry(
                    load_fn=lambda: EsmTokenizer.from_pretrained(model_name),
                    description=f"ESM-2 tokenizer ({model_name})",
                    logger=self.logger,
                )

            # --- Model ------------------------------------------------------------
            def _load_model() -> EsmForMaskedLM:
                try:
                    # Explainability requires attention tensors; SDPA often omits them.
                    return EsmForMaskedLM.from_pretrained(
                        model_name,
                        attn_implementation="eager",
                    )
                except TypeError:
                    self.logger.warning(
                        "Current transformers build does not support attn_implementation arg; "
                        "loading default attention backend."
                    )
                    return EsmForMaskedLM.from_pretrained(model_name)

            def _load_model_offline() -> EsmForMaskedLM:
                try:
                    return EsmForMaskedLM.from_pretrained(
                        model_name,
                        attn_implementation="eager",
                        local_files_only=True,
                    )
                except TypeError:
                    self.logger.warning(
                        "Current transformers build does not support attn_implementation arg; "
                        "loading default attention backend."
                    )
                    return EsmForMaskedLM.from_pretrained(model_name, local_files_only=True)

            if offline:
                self.model = _load_model_offline().to(self.device)
            else:
                self.model = _load_with_retry(
                    load_fn=_load_model,
                    description=f"ESM-2 model ({model_name})",
                    logger=self.logger,
                ).to(self.device)

            # Ensure hidden states are always returned to avoid NoneType errors during navigation
            self.model.config.output_hidden_states = True

            # Optional LoRA adapter loading (requires peft library)
            if lora_path is not None:
                try:
                    from peft import PeftModel
                    self.model = PeftModel.from_pretrained(self.model, lora_path)
                    self.logger.info("Loaded LoRA adapter from %s", lora_path)
                except ImportError:
                    self.logger.warning(
                        "peft library not installed; ignoring lora_path=%s. "
                        "Install with: pip install peft", lora_path
                    )
                except Exception as exc:
                    self.logger.warning("Failed to load LoRA adapter from %s: %s", lora_path, exc)

            self.model.eval()
            self.logger.info("Model loaded successfully with output_hidden_states=True.")
        except OSError as e:
            if offline:
                self.logger.error(
                    "TRANSFORMERS_OFFLINE=1 is set but the model '%s' was not found "
                    "in the local cache.  Either download the model first with "
                    "`huggingface-cli download %s` or unset the TRANSFORMERS_OFFLINE "
                    "environment variable to allow network access.",
                    model_name,
                    model_name,
                )
            else:
                self.logger.error(f"Failed to load model: {e}")
            raise
        except Exception as e:
            self.logger.error(f"Failed to load model: {e}")
            raise

    @property
    def hidden_size(self) -> int:
        """Return the hidden dimension of the loaded ESM-2 model."""
        return int(self.model.config.hidden_size)

    def get_embeddings(self, sequence: str) -> torch.Tensor:
        """
        Retrieves the initial token embeddings (input to the first layer).
        Returns: Tensor with requires_grad=True
        """
        inputs = self.tokenizer(sequence, return_tensors="pt", add_special_tokens=False).to(self.device)
        input_ids = inputs.input_ids
        
        # Get the embedding layer
        with torch.no_grad():
            # ESM-2 embedding layer is usually model.esm.embeddings.word_embeddings
            # but depends on HuggingFace version. For EsmForMaskedLM:
            emb_layer = self.model.esm.embeddings.word_embeddings
            embeddings = emb_layer(input_ids) # (1, L, D)
            
        return embeddings

    def latent_forward_ascent(
        self,
        embeddings: torch.Tensor,
        return_hidden: bool = False,
        return_attention: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, ...]:
        """
        Runs the transformer forward pass starting from soft embeddings.

        Parameters
        ----------
        embeddings : Tensor
            Soft input embeddings (1, L, D).
        return_hidden : bool
            If True, the 4th return value is the last hidden state tensor (1, L, D).
        return_attention : bool
            If True, an additional return value is a tuple of attention tensors,
            one per layer, each of shape (1, heads, L, L).

        Returns
        -------
        Tuple of (last_hidden_state, logits, probabilities[, hidden][, attentions])
        """
        esm_outputs = self.model.esm(
            inputs_embeds=embeddings,
            output_hidden_states=True,
            output_attentions=return_attention,
            return_dict=True,
        )
        h = esm_outputs.last_hidden_state

        logits = self.model.lm_head(h)
        probs = torch.softmax(logits[0], dim=-1)

        result = [h, logits, probs]
        if return_hidden:
            result.append(h)
        if return_attention:
            result.append(esm_outputs.attentions)
        return tuple(result)

    def encode(self, sequence: str) -> torch.Tensor:
        """
        Embeds a protein sequence into the latent space (last hidden state).
        Uses the base ESM model for cleaner embedding extraction.
        """
        inputs = self.tokenizer(sequence, return_tensors="pt", add_special_tokens=False).to(self.device)
        with torch.no_grad():
            # Use the base model (esm) directly for hidden states
            outputs = self.model.esm(**inputs, output_hidden_states=True, return_dict=True)
            embeddings = outputs.last_hidden_state
        return embeddings

    def decode(self, embeddings: torch.Tensor, sequence_len: Optional[int] = None) -> str:
        """
        Decodes latent embeddings back to a sequence using the LM head.
        Args:
            embeddings: Tensor of shape (1, L, D)
        """
        with torch.no_grad():
            logits = self.model.lm_head(embeddings)
            # Shape: (1, L, Vocab)
            
        params = {"dim": -1}
        probabilities = F.softmax(logits, **params)
        top_ids = torch.argmax(probabilities, dim=-1) # Shape (1, L)
        
        tokens = self.tokenizer.convert_ids_to_tokens(top_ids[0])
        # Filter out special tokens but keep the length? 
        # Actually, if we encoded without special tokens, the model SHOULD predict AA.
        # If it predicts special tokens, we might want to mask them or just keep going.
        # But convert_ids_to_tokens returns list of strings.
        
        # ESM tokenizer often handles tokens like this:
        # <cls>, <eos>, <pad>, <mask>, <unk>
        # And amino acids: 'L', 'A', etc.
        
        final_seq = []
        for t in tokens:
            if t in self.tokenizer.all_special_tokens:
                 # If the model hallucinates a special token, replacing with 'X' or closest AA is safer than dropping.
                 # For now, let's use 'X' to indicate uncertainty/error at that position
                 final_seq.append("X")
            else:
                 final_seq.append(t)
                 
        return "".join(final_seq)

    def get_logits(self, sequence: str) -> torch.Tensor:
        """Get raw logits for a sequence (useful for stability scoring)."""
        inputs = self.tokenizer(sequence, return_tensors="pt", add_special_tokens=False).to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
        return outputs.logits

    def get_dna_contact_prob(self, z: torch.Tensor, logits: Optional[torch.Tensor] = None, probs: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Refined Mechanistic Proxy: Estimates DNA binding recruitment force.
        Optimized: Accepts pre-calculated probabilities to skip redundant softmax.
        """
        z_sq = z.squeeze(0)
        hotspots = [119, 174, 240, 247, 272, 279]
        hotspots = [i for i in hotspots if i < z_sq.shape[0]]
        
        if not hotspots:
            return torch.tensor(0.0, device=z.device)
            
        latent_force = z_sq[hotspots, :].norm(dim=-1).mean()
        
        if probs is not None:
            # probs: (L, Vocab)
            pos_charge_ids = [10, 15, 21]
            charge_prob = probs[hotspots][:, pos_charge_ids].sum(dim=-1).mean()
            return 0.5 * latent_force + 5.0 * charge_prob
        elif logits is not None:
            probs_local = torch.softmax(logits[0], dim=-1)
            pos_charge_ids = [10, 15, 21]
            charge_prob = probs_local[hotspots][:, pos_charge_ids].sum(dim=-1).mean()
            return 0.5 * latent_force + 5.0 * charge_prob
            
        return latent_force

    def get_surface_charge_density(self, logits: Optional[torch.Tensor] = None, probs: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Calculates the density of positively charged residues at the DNA interface loops.
        """
        if probs is None and logits is not None:
             probs = torch.softmax(logits[0], dim=-1)
        
        if probs is None:
             return torch.tensor(0.0)

        pos_ids = [10, 15, 21] # R, K, H
        l1 = list(range(111, 124))
        l2 = list(range(162, 195))
        l3 = list(range(235, 251))
        all_interface = [i for i in (l1 + l2 + l3) if i < probs.shape[0]]
        
        if not all_interface:
            return torch.tensor(0.0, device=logits.device)
            
        charge_density = probs[all_interface][:, pos_ids].sum(dim=-1).mean()
        return charge_density

    def get_hydrophobic_packing(self, logits: Optional[torch.Tensor] = None, probs: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Estimates the core stability via hydrophobic packing density.
        """
        if probs is None and logits is not None:
             probs = torch.softmax(logits[0], dim=-1)
        
        if probs is None:
             return torch.tensor(0.0)

        # L(4), I(12), V(7), F(18), W(22), M(20)
        hydro_ids = [4, 12, 7, 18, 22, 20]
        
        # Scaffold residues (general core)
        core_res = [i for i in range(93, 312) if i < probs.shape[0]]
        # Exclude loops to focus on internal packing
        loops = set(range(111, 124)) | set(range(162, 195)) | set(range(235, 251))
        true_core = [i for i in core_res if i not in loops]
        
        if not true_core:
            return torch.tensor(0.0, device=logits.device)
            
        packing_score = probs[true_core][:, hydro_ids].sum(dim=-1).mean()
        return packing_score

class ManifoldWalker:
    """
    Performs vector arithmetic and interpolation in the latent space.
    """
    def __init__(self, embedder: ManifoldEmbedder):
        self.embedder = embedder
        self.logger = get_logger(__name__)

    def interpolate(self, seq_start: str, seq_end: str, steps: int = 10) -> List[str]:
        """
        Linearly interpolates between two sequences in latent space.
        Returns a list of decoded sequences along the path.
        """
        z_start = self.embedder.encode(seq_start)
        z_end = self.embedder.encode(seq_end)
        
        # Check shapes
        if z_start.shape != z_end.shape:
            # If lengths differ, we can't do simple element-wise interpolation easily 
            # without alignment or padding.
            self.logger.warning(f"Shape mismatch in interpolation: {z_start.shape} vs {z_end.shape}. Using smaller length.")
            min_len = min(z_start.shape[1], z_end.shape[1])
            z_start = z_start[:, :min_len, :]
            z_end = z_end[:, :min_len, :]

        results = []
        # t goes from 0 to 1
        alphas = torch.linspace(0, 1, steps)
        
        for t in alphas:
            # Linear Interpolation (SLERP is better for hyperspheres, but LERP is fine for now)
            z_interp = (1 - t) * z_start + t * z_end
            
            # Decode
            seq = self.embedder.decode(z_interp)
            results.append(seq)
            
        return results

    def steer(self, seq: str, direction_vector: torch.Tensor, magnitude: float = 1.0) -> str:
        """
        Moves the sequence embedding in a specific direction.
        """
        z = self.embedder.encode(seq)
        
        # Ensure direction matches shape
        if direction_vector.shape != z.shape:
             # Broadcasting or resizing logic might be needed
             pass
             
        z_new = z + (direction_vector * magnitude)
        return self.embedder.decode(z_new)

    def stability_score(self, sequence: str) -> float:
        """
        Uses Pseudo-Log-Likelihood (PLL) of the sequence under the model as a proxy for stability/fitness.
        Higher is better.
        """
        inputs = self.embedder.tokenizer(sequence, return_tensors="pt", add_special_tokens=False).to(self.embedder.device)
        labels = inputs.input_ids
        
        with torch.no_grad():
            outputs = self.embedder.model(**inputs, labels=labels)
            # Loss is CrossEntropyLoss (negative log likelihood)
            # We want likelihood, so we take negative loss
            neg_log_likelihood = -outputs.loss.item()
            
        return neg_log_likelihood

    def steer_with_gradient(self, z_start: torch.Tensor, oracle_model: torch.nn.Module, steps: int = 10, step_size: float = 0.1) -> List[str]:
        """
        Performs gradient ascent in latent space to maximize the predicted function score.
        z_{t+1} = z_t + step_size * grad(Function(z_t))
        """
        results = []
        # Clone and detach to start fresh computation graph
        # z_start (1, L, D)
        z = z_start.clone().detach().requires_grad_(True)
        
        # Optimizer can work better than manual update
        optimizer = torch.optim.Adam([z], lr=step_size)
        
        for i in range(steps):
            optimizer.zero_grad()
            
            # Predict function score
            # Oracle expects pooled embedding? 
            # Our oracle was trained on mean(dim=1).
            # z shape is (1, L, D)
            pooled = z.mean(dim=1) # (1, D)
            score = oracle_model(pooled)
            
            # Maximize score => Minimize negative score
            loss = -score
            loss.backward()
            optimizer.step()
            
            # Periodically decode to see what we have
            # Decoding is heavy, maybe just every few steps or return final trajectory
            with torch.no_grad():
                seq = self.embedder.decode(z)
                results.append(seq)
                
        return results


In [ ]:
%%writefile p53cad/engine/oracle.py
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

from p53cad.core.logging import get_logger
from p53cad.engine.latent import ManifoldEmbedder


class DMSDataset(Dataset):
    def __init__(
        self,
        sequences: List[str],
        scores: List[float],
        embedder: ManifoldEmbedder,
        per_position: bool = False,
    ):
        self.sequences = sequences
        self.scores = torch.tensor(scores, dtype=torch.float32)
        self.embedder = embedder
        self.per_position = per_position
        self.cache: Dict[str, torch.Tensor] = {}

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        score = self.scores[idx]

        if seq in self.cache:
            emb = self.cache[seq]
        else:
            with torch.no_grad():
                raw = self.embedder.encode(seq)  # (1, L, D)
                if self.per_position:
                    emb = raw.squeeze(0).cpu()  # (L, D)
                else:
                    emb = raw.mean(dim=1).squeeze(0).cpu()  # (D,)
            self.cache[seq] = emb

        return emb, score


class FunctionalNet(nn.Module):
    """
    Single-head MLP oracle.
    """

    def __init__(
        self,
        input_dim: int = 1280,
        hidden_dim: int = 128,
        num_layers: int = 2,
        dropout: float = 0.2,
    ):
        super().__init__()
        depth = max(int(num_layers), 1)
        width = max(int(hidden_dim), 16)
        drop = float(max(0.0, min(dropout, 0.9)))

        # Preserve legacy default geometry for checkpoint compatibility:
        # hidden_dim=128, num_layers=2 -> [128, 64]
        hidden_dims = [width]
        for layer_idx in range(1, depth):
            next_width = max(int(round(width / (2 ** layer_idx))), 16)
            hidden_dims.append(next_width)

        layers: list[nn.Module] = []
        in_dim = input_dim
        for idx, out_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(nn.ReLU())
            # Match legacy behavior: dropout only between hidden layers.
            if idx < len(hidden_dims) - 1 and drop > 0:
                layers.append(nn.Dropout(drop))
            in_dim = out_dim
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        target_device = self.net[0].weight.device
        if x.device != target_device:
            x = x.to(target_device)
        return self.net(x)


class AttentionPoolingNet(nn.Module):
    """Oracle with learnable attention pooling over sequence positions.

    Accepts (batch, seq_len, D) and learns which positions matter for
    functional scoring, eliminating the mean-pooling bottleneck.
    Also accepts (batch, D) for backward compatibility (passes through MLP head).

    Uses delta encoding: subtracts a cached WT baseline from per-position
    inputs so that mutated positions stand out as nonzero residuals.
    Without this, single-AA mutations are invisible among 393 identical positions.
    """

    def __init__(
        self,
        input_dim: int = 1280,
        hidden_dim: int = 256,
        n_heads: int = 4,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.input_dim = input_dim

        # WT baseline for delta encoding (set during training, saved in state_dict)
        self.register_buffer("wt_baseline", None)

        # Learnable query vector for attention pooling
        self.query = nn.Parameter(torch.randn(1, 1, input_dim) * 0.02)

        # Multi-head attention: query attends over sequence positions
        self.attn = nn.MultiheadAttention(
            embed_dim=input_dim, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.layer_norm = nn.LayerNorm(input_dim)

        # MLP head: input_dim → hidden → hidden//2 → 1
        self.head = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def set_wt_baseline(self, wt_emb: torch.Tensor) -> None:
        """Register WT per-position embedding for delta encoding.

        Parameters
        ----------
        wt_emb : Tensor
            Shape (L, D) or (1, L, D) — WT hidden states from ESM-2.
        """
        if wt_emb.dim() == 3:
            wt_emb = wt_emb.squeeze(0)
        self.wt_baseline = wt_emb.detach().clone()

    def forward(self, x):
        target_device = self.query.device
        if x.device != target_device:
            x = x.to(target_device)

        # Backward compat: if already pooled (batch, D), just run MLP head
        if x.dim() == 2:
            return self.head(x)

        # Delta encoding: subtract WT baseline so mutation positions have nonzero signal
        if self.wt_baseline is not None:
            wt = self.wt_baseline.to(x.device)
            x = x - wt.unsqueeze(0)  # (batch, L, D) - (1, L, D)

        # x: (batch, seq_len, D)
        batch_size = x.size(0)
        query = self.query.expand(batch_size, -1, -1)  # (batch, 1, D)

        # Attention pooling: query attends to all positions
        pooled, _attn_weights = self.attn(query, x, x)  # (batch, 1, D)
        pooled = self.layer_norm(pooled.squeeze(1))  # (batch, D)

        return self.head(pooled)


class FunctionalOracle:
    def __init__(
        self,
        model_path: Path | str = None,
        input_dim: int = 1280,
        hidden_dim: int = 128,
        num_layers: int = 2,
        dropout: float = 0.2,
        use_rtl: bool = False,
        arch: str = "legacy_mlp",
    ):
        self.logger = get_logger(__name__)
        self.device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = max(int(num_layers), 1)
        self.dropout = float(max(0.0, min(dropout, 0.9)))
        self.arch_name = arch

        if arch == "attention_pooling":
            self.model: nn.Module = AttentionPoolingNet(
                input_dim=self.input_dim,
                hidden_dim=self.hidden_dim,
                n_heads=4,
                dropout=self.dropout,
            )
        else:
            self.model: nn.Module = FunctionalNet(
                input_dim=self.input_dim,
                hidden_dim=self.hidden_dim,
                num_layers=self.num_layers,
                dropout=self.dropout,
            )

        if use_rtl:
            self.logger.warning("RTL oracle was removed. Falling back to %s.", self.arch_name)

        if model_path and Path(model_path).exists():
            self._load_model(model_path)

        self.model = self.model.to(self.device)
        self.model.eval()

    def _load_model(self, model_path: Path | str) -> None:
        self.logger.info(f"Loading oracle from {model_path}")
        payload = torch.load(model_path, map_location=self.device)

        # New checkpoint format with explicit metadata.
        if isinstance(payload, dict) and "model_state_dict" in payload:
            arch = payload.get("arch", "legacy_mlp")
            self.input_dim = int(payload.get("input_dim", self.input_dim))
            self.hidden_dim = int(payload.get("hidden_dim", self.hidden_dim))
            self.num_layers = int(payload.get("num_layers", self.num_layers))
            self.dropout = float(payload.get("dropout", self.dropout))

            if arch == "attention_pooling":
                model = AttentionPoolingNet(
                    input_dim=self.input_dim,
                    hidden_dim=self.hidden_dim,
                    n_heads=int(payload.get("n_heads", 4)),
                    dropout=self.dropout,
                )
            elif arch == "legacy_mlp":
                model = FunctionalNet(
                    input_dim=self.input_dim,
                    hidden_dim=self.hidden_dim,
                    num_layers=self.num_layers,
                    dropout=self.dropout,
                )
            else:
                self.logger.warning(
                    "Unsupported oracle architecture '%s' in %s. "
                    "Falling back to legacy_mlp.",
                    arch,
                    model_path,
                )
                model = FunctionalNet(
                    input_dim=self.input_dim,
                    hidden_dim=self.hidden_dim,
                    num_layers=self.num_layers,
                    dropout=self.dropout,
                )
                self.model = model
                self.arch_name = "legacy_mlp"
                return

            try:
                sd = payload["model_state_dict"]
                # wt_baseline registered as None isn't in the expected state_dict;
                # pre-register with matching shape so load_state_dict can fill it.
                if "wt_baseline" in sd and hasattr(model, "wt_baseline") and model.wt_baseline is None:
                    model.register_buffer("wt_baseline", torch.zeros_like(sd["wt_baseline"]))
                model.load_state_dict(sd)
            except Exception as exc:
                self.logger.warning(
                    "Failed to load oracle checkpoint %s: %s. Using fresh %s weights.",
                    model_path,
                    exc,
                    arch,
                )
            else:
                self.model = model
                self.arch_name = arch
                self.logger.info(
                    "Loaded oracle architecture: %s (input=%d, hidden=%d, layers=%d, dropout=%.2f)",
                    arch,
                    self.input_dim,
                    self.hidden_dim,
                    self.num_layers,
                    self.dropout,
                )
            return

        # Legacy checkpoint format: raw state_dict only.
        state_dict = payload
        model = FunctionalNet(
            input_dim=self.input_dim,
            hidden_dim=self.hidden_dim,
            num_layers=self.num_layers,
            dropout=self.dropout,
        )
        try:
            model.load_state_dict(state_dict)
        except Exception as exc:
            self.logger.warning(
                "Failed to load legacy oracle checkpoint %s: %s. Using fresh legacy_mlp weights.",
                model_path,
                exc,
            )
            return

        self.model = model
        self.arch_name = "legacy_mlp"
        self.logger.info("Loaded legacy oracle checkpoint.")

    def train(
        self,
        dms_data,
        embedder: ManifoldEmbedder,
        epochs: int = 10,
        save_path: Path = None,
        val_split: float = 0.1,
        early_stopping_patience: int = 8,
        min_delta: float = 1e-4,
        batch_size: int = 32,
        seed: int = 42,
    ):
        self.logger.info(f"Training FunctionalOracle on {len(dms_data)} sequences...")
        self.model.train()

        use_per_position = isinstance(self.model, AttentionPoolingNet)

        # For attention oracle: compute WT baseline for delta encoding
        if use_per_position:
            from p53cad.data.dms import P53_WT

            with torch.no_grad():
                wt_emb = embedder.encode(P53_WT).squeeze(0).cpu()  # (L, D)
            self.model.set_wt_baseline(wt_emb)
            self.logger.info(
                "Set WT baseline for delta encoding (%s)", tuple(wt_emb.shape)
            )

        dataset = DMSDataset(
            dms_data["sequence"].tolist(),
            dms_data["score"].tolist(),
            embedder,
            per_position=use_per_position,
        )
        total_samples = len(dataset)
        val_fraction = float(max(0.0, min(val_split, 0.49)))
        patience = max(int(early_stopping_patience), 0)
        delta = float(max(min_delta, 0.0))
        train_loader: DataLoader
        val_loader: Optional[DataLoader] = None

        if total_samples >= 5 and val_fraction > 0.0:
            val_size = int(round(total_samples * val_fraction))
            val_size = max(1, min(val_size, total_samples - 1))
            train_size = total_samples - val_size
            split_gen = torch.Generator().manual_seed(seed)
            train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=split_gen)
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
            self.logger.info(
                "Using validation split: train=%d, val=%d (%.1f%%)",
                train_size,
                val_size,
                100.0 * val_size / total_samples,
            )
        else:
            train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
            self.logger.info(
                "Validation disabled (samples=%d, val_split=%.3f). Training on full set.",
                total_samples,
                val_fraction,
            )

        optimizer = optim.AdamW(self.model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()
        best_metric = float("inf")
        best_epoch = 0
        epochs_completed = 0
        patience_counter = 0
        best_state_dict: Optional[Dict[str, torch.Tensor]] = None

        for epoch in range(epochs):
            total_loss = 0.0

            for X, y in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
                X = X.to(self.device)
                y = y.to(self.device).unsqueeze(1)

                optimizer.zero_grad()
                pred = self.model(X)
                loss = criterion(pred, y)
                loss.backward()
                optimizer.step()
                total_loss += float(loss.item())

            epochs_completed = epoch + 1
            n_train_batches = max(len(train_loader), 1)
            train_loss = total_loss / n_train_batches
            metric_for_selection = train_loss
            val_loss: Optional[float] = None

            if val_loader is not None:
                self.model.eval()
                val_total = 0.0
                with torch.no_grad():
                    for X_val, y_val in val_loader:
                        X_val = X_val.to(self.device)
                        y_val = y_val.to(self.device).unsqueeze(1)
                        pred_val = self.model(X_val)
                        val_total += float(criterion(pred_val, y_val).item())
                self.model.train()
                n_val_batches = max(len(val_loader), 1)
                val_loss = val_total / n_val_batches
                metric_for_selection = val_loss
                self.logger.info(
                    "Epoch %d Train Loss: %.4f | Val Loss: %.4f",
                    epoch + 1,
                    train_loss,
                    val_loss,
                )
            else:
                self.logger.info("Epoch %d Train Loss: %.4f", epoch + 1, train_loss)

            if metric_for_selection + delta < best_metric:
                best_metric = metric_for_selection
                best_epoch = epoch + 1
                best_state_dict = {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in self.model.state_dict().items()
                }
                patience_counter = 0
            elif val_loader is not None and patience > 0:
                patience_counter += 1
                if patience_counter >= patience:
                    self.logger.info(
                        "Early stopping at epoch %d (best epoch %d, best val loss %.4f).",
                        epoch + 1,
                        best_epoch,
                        best_metric,
                    )
                    break

        if best_state_dict is not None:
            self.model.load_state_dict(best_state_dict)
            self.logger.info(
                "Restored best checkpoint from epoch %d (selection metric %.4f).",
                best_epoch,
                best_metric,
            )
        self.model.eval()
        if save_path:
            checkpoint = {
                "arch": self.arch_name,
                "input_dim": self.input_dim,
                "hidden_dim": int(self.hidden_dim),
                "num_layers": int(self.num_layers),
                "dropout": float(self.dropout),
                "n_heads": int(self.model.attn.num_heads) if hasattr(self.model, "attn") else 0,
                "model_state_dict": self.model.state_dict(),
                "training": {
                    "epochs_requested": int(epochs),
                    "epochs_completed": int(epochs_completed),
                    "best_epoch": int(best_epoch) if best_epoch else int(epochs_completed),
                    "best_selection_metric": float(best_metric),
                    "val_split": float(val_fraction),
                    "early_stopping_patience": int(patience),
                    "min_delta": float(delta),
                    "train_samples": int(total_samples - (len(val_loader.dataset) if val_loader is not None else 0)),
                    "val_samples": int(len(val_loader.dataset) if val_loader is not None else 0),
                },
            }
            torch.save(checkpoint, save_path)
            self.logger.info(f"Saved oracle checkpoint to {save_path} (arch={self.arch_name})")

    def _prepare_input(self, embedding: torch.Tensor) -> torch.Tensor:
        if not torch.is_tensor(embedding):
            embedding = torch.tensor(embedding, dtype=torch.float32)
        vec = embedding.to(self.device).float()
        if vec.dim() == 3:
            # Attention oracle consumes (batch, L, D) directly
            if isinstance(self.model, AttentionPoolingNet):
                return vec
            return vec.mean(dim=1)
        if vec.dim() == 2:
            return vec
        if vec.dim() == 1:
            return vec.unsqueeze(0)
        raise ValueError(f"Unsupported embedding shape for oracle prediction: {tuple(vec.shape)}")

    def predict(self, embedding: torch.Tensor) -> float:
        """Predict function score from an embedding tensor."""
        with torch.no_grad():
            vec = self._prepare_input(embedding)
            return float(self.model(vec).squeeze(-1).mean().item())

    def predict_batch(self, embeddings: torch.Tensor) -> torch.Tensor:
        """Predict function scores for a batch of embeddings."""
        vec = self._prepare_input(embeddings)
        return self.model(vec)

    def predict_with_routing(self, embedding: torch.Tensor) -> Dict[str, float]:
        """
        Backward-compatible shim: routing diagnostics are removed.
        """
        return {"score": self.predict(embedding), "arch": self.arch_name}


def compute_masked_marginal_pll(
    embedder: ManifoldEmbedder,
    sequence: str,
    mutated_positions: List[int],
) -> float:
    """Compute masked marginal pseudo-log-likelihood at mutated positions.

    For each position in *mutated_positions* (1-indexed), mask the token,
    run ESM-2 forward, and accumulate log P(true_aa | context).  This is
    the gold-standard zero-shot fitness predictor (Meier et al. 2021).

    Too expensive for the inner optimisation loop (one forward pass per
    position) but ideal for ranking ~30 shortlist candidates.

    Parameters
    ----------
    embedder : ManifoldEmbedder
        Loaded ESM-2 wrapper with tokenizer and model.
    sequence : str
        Full-length protein sequence (e.g. 393 AA for p53).
    mutated_positions : list of int
        1-indexed residue positions to evaluate.

    Returns
    -------
    float
        Sum of log P(aa_i | context) over mutated positions.
    """
    if not mutated_positions:
        return 0.0

    # Tokenize without special tokens — positions map 1:1 to token indices
    inputs = embedder.tokenizer(
        sequence, return_tensors="pt", add_special_tokens=False
    ).to(embedder.device)
    input_ids = inputs.input_ids  # (1, L)

    mask_token_id = embedder.tokenizer.mask_token_id
    total_log_prob = 0.0

    with torch.no_grad():
        for pos in mutated_positions:
            idx = pos - 1  # 0-indexed
            if idx < 0 or idx >= input_ids.shape[1]:
                continue

            masked_ids = input_ids.clone()
            masked_ids[0, idx] = mask_token_id

            outputs = embedder.model(input_ids=masked_ids, return_dict=True)
            logits = outputs.logits  # (1, L, vocab)
            log_probs = F.log_softmax(logits[0, idx], dim=-1)
            true_token = input_ids[0, idx].item()
            total_log_prob += float(log_probs[true_token].item())

    return total_log_prob


def compute_conditional_rescue_scores(
    embedder: ManifoldEmbedder,
    cancer_seq: str,
    positions: List[int],
) -> Dict[int, Dict[str, float]]:
    """Compute ESM-2 P(aa | cancer_context) at each position.

    Masks each position in the cancer sequence, runs ESM-2 forward, and
    returns the full amino acid log-probability distribution.  This tells
    us which substitutions ESM-2 considers favorable *in the cancer context*,
    enabling conditional DMS scoring.

    Parameters
    ----------
    embedder : ManifoldEmbedder
        Loaded ESM-2 wrapper.
    cancer_seq : str
        Full-length cancer-mutant protein sequence.
    positions : list of int
        1-indexed residue positions to evaluate.

    Returns
    -------
    dict
        {pos: {aa_letter: log_prob, ...}, ...}
    """
    if not positions:
        return {}

    inputs = embedder.tokenizer(
        cancer_seq, return_tensors="pt", add_special_tokens=False
    ).to(embedder.device)
    input_ids = inputs.input_ids
    mask_token_id = embedder.tokenizer.mask_token_id

    # Build AA token id → letter mapping
    aa_letters = "ACDEFGHIKLMNPQRSTVWY"
    aa_token_ids = []
    for aa in aa_letters:
        tid = embedder.tokenizer.convert_tokens_to_ids(aa)
        aa_token_ids.append(tid)

    result: Dict[int, Dict[str, float]] = {}

    with torch.no_grad():
        for pos in positions:
            idx = pos - 1
            if idx < 0 or idx >= input_ids.shape[1]:
                continue

            masked_ids = input_ids.clone()
            masked_ids[0, idx] = mask_token_id

            outputs = embedder.model(input_ids=masked_ids, return_dict=True)
            logits = outputs.logits
            log_probs = F.log_softmax(logits[0, idx], dim=-1)

            pos_scores: Dict[str, float] = {}
            for aa_char, tid in zip(aa_letters, aa_token_ids):
                pos_scores[aa_char] = float(log_probs[tid].item())
            result[pos] = pos_scores

    return result


In [ ]:
%%writefile p53cad/engine/pareto.py
"""Multi-objective Pareto optimization for p53 rescue candidate ranking.

Implements NSGA-II non-dominated sorting and crowding distance for ranking
candidates across multiple objectives (oracle score, PLL, DMS quality,
identity, stability).  All objectives are expressed as values to MAXIMIZE.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence

import numpy as np


@dataclass
class ParetoSolution:
    """A single candidate in multi-objective space."""

    candidate_uid: str
    objectives: Dict[str, float]  # all values to MAXIMIZE
    pareto_rank: int = 0
    crowding_distance: float = 0.0
    metadata: Dict[str, Any] = field(default_factory=dict)


class ParetoFront:
    """NSGA-II non-dominated sorting and crowding distance computation.

    Usage::

        front = ParetoFront()
        for cand in candidates:
            front.add(ParetoSolution(
                candidate_uid=cand["candidate_uid"],
                objectives={
                    "oracle_score": cand["score"],
                    "pll": cand.get("pll", 0.0),
                    "dms_quality": -cand.get("rescue_dms_mean", 0.0),  # negate: lower Z = better
                    "identity": cand.get("identity", 100.0),
                    "stability": cand.get("stability", 0.0),
                },
            ))
        rank1 = front.get_front(rank=1)
        best30 = front.best_by_crowding(top_n=30)
    """

    OBJECTIVES = ["oracle_score", "pll", "dms_quality", "identity", "stability"]

    def __init__(self, objectives: Optional[Sequence[str]] = None):
        self._objectives = list(objectives or self.OBJECTIVES)
        self._solutions: List[ParetoSolution] = []
        self._dirty = True

    def add(self, solution: ParetoSolution) -> None:
        """Add a solution and mark ranks as stale."""
        self._solutions.append(solution)
        self._dirty = True

    def add_batch(self, solutions: Sequence[ParetoSolution]) -> None:
        """Add multiple solutions at once."""
        self._solutions.extend(solutions)
        self._dirty = True

    @property
    def solutions(self) -> List[ParetoSolution]:
        if self._dirty:
            self._recompute_ranks()
        return list(self._solutions)

    def _dominates(self, a: ParetoSolution, b: ParetoSolution) -> bool:
        """Return True if solution *a* Pareto-dominates *b*.

        *a* dominates *b* iff a is >= b on all objectives and strictly > on at least one.
        """
        at_least_one_better = False
        for obj in self._objectives:
            va = a.objectives.get(obj, 0.0)
            vb = b.objectives.get(obj, 0.0)
            if va < vb:
                return False
            if va > vb:
                at_least_one_better = True
        return at_least_one_better

    def _recompute_ranks(self) -> None:
        """Assign NSGA-II Pareto ranks via non-dominated sorting."""
        n = len(self._solutions)
        if n == 0:
            self._dirty = False
            return

        # domination_count[i] = number of solutions that dominate i
        domination_count = [0] * n
        # dominated_set[i] = indices of solutions that i dominates
        dominated_set: List[List[int]] = [[] for _ in range(n)]

        for i in range(n):
            for j in range(i + 1, n):
                if self._dominates(self._solutions[i], self._solutions[j]):
                    dominated_set[i].append(j)
                    domination_count[j] += 1
                elif self._dominates(self._solutions[j], self._solutions[i]):
                    dominated_set[j].append(i)
                    domination_count[i] += 1

        # Build fronts
        current_front: List[int] = []
        for i in range(n):
            if domination_count[i] == 0:
                self._solutions[i].pareto_rank = 1
                current_front.append(i)

        rank = 1
        while current_front:
            next_front: List[int] = []
            for i in current_front:
                for j in dominated_set[i]:
                    domination_count[j] -= 1
                    if domination_count[j] == 0:
                        self._solutions[j].pareto_rank = rank + 1
                        next_front.append(j)
            rank += 1
            current_front = next_front

        self._compute_crowding_distances()
        self._dirty = False

    def _compute_crowding_distances(self) -> None:
        """Compute crowding distance within each Pareto rank."""
        if not self._solutions:
            return

        max_rank = max(s.pareto_rank for s in self._solutions)

        for rank in range(1, max_rank + 1):
            indices = [i for i, s in enumerate(self._solutions) if s.pareto_rank == rank]
            if len(indices) <= 2:
                for i in indices:
                    self._solutions[i].crowding_distance = float("inf")
                continue

            # Initialize distances
            distances = {i: 0.0 for i in indices}

            for obj in self._objectives:
                # Sort indices by this objective
                sorted_idx = sorted(indices, key=lambda i: self._solutions[i].objectives.get(obj, 0.0))

                # Boundary solutions get infinite distance
                distances[sorted_idx[0]] = float("inf")
                distances[sorted_idx[-1]] = float("inf")

                # Objective range for normalization
                obj_min = self._solutions[sorted_idx[0]].objectives.get(obj, 0.0)
                obj_max = self._solutions[sorted_idx[-1]].objectives.get(obj, 0.0)
                obj_range = obj_max - obj_min
                if obj_range < 1e-12:
                    continue

                for k in range(1, len(sorted_idx) - 1):
                    prev_val = self._solutions[sorted_idx[k - 1]].objectives.get(obj, 0.0)
                    next_val = self._solutions[sorted_idx[k + 1]].objectives.get(obj, 0.0)
                    distances[sorted_idx[k]] += (next_val - prev_val) / obj_range

            for i in indices:
                self._solutions[i].crowding_distance = distances[i]

    def get_front(self, rank: int = 1) -> List[ParetoSolution]:
        """Return all solutions with the given Pareto rank."""
        if self._dirty:
            self._recompute_ranks()
        return [s for s in self._solutions if s.pareto_rank == rank]

    def best_by_crowding(self, top_n: int = 30) -> List[ParetoSolution]:
        """Return top-N solutions sorted by (rank asc, crowding_distance desc)."""
        if self._dirty:
            self._recompute_ranks()

        sorted_solutions = sorted(
            self._solutions,
            key=lambda s: (s.pareto_rank, -s.crowding_distance),
        )
        return sorted_solutions[:top_n]

    def __len__(self) -> int:
        return len(self._solutions)

    def summary(self) -> Dict[str, Any]:
        """Return summary statistics of the Pareto front."""
        if self._dirty:
            self._recompute_ranks()
        if not self._solutions:
            return {"n_solutions": 0, "n_fronts": 0}

        ranks = [s.pareto_rank for s in self._solutions]
        return {
            "n_solutions": len(self._solutions),
            "n_fronts": max(ranks),
            "rank_1_count": sum(1 for r in ranks if r == 1),
            "rank_distribution": {r: sum(1 for rr in ranks if rr == r) for r in sorted(set(ranks))},
        }


In [ ]:
%%writefile p53cad/engine/campaign.py
from __future__ import annotations

from contextlib import contextmanager
from dataclasses import dataclass
from datetime import datetime, timezone
import json
from pathlib import Path
import platform
import shutil
import subprocess
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import math

from p53cad.analysis.clinical_impact import ClinicalImpactEngine
from p53cad.core.logging import get_logger
from p53cad.data.dms import P53_WT, apply_mutation, get_dms_data, get_pairwise_epistasis_lookup, parse_single_mutation
from p53cad.engine.latent import ManifoldEmbedder
from p53cad.engine.oracle import AttentionPoolingNet, FunctionalOracle, compute_conditional_rescue_scores
from p53cad.results.schema import (
    BIG8_HOTSPOTS,
    DEFAULT_DELIVERY_METHODS,
    build_run_id,
    build_scenario_matrix,
    scenarios_to_frame,
    select_presentation_shortlist,
)
from p53cad.results.store import CampaignStore


logger = get_logger(__name__)


@contextmanager
def _prevent_sleep():
    """Prevent macOS idle/system sleep during long campaigns via caffeinate."""
    proc = None
    if platform.system() == "Darwin" and shutil.which("caffeinate"):
        try:
            proc = subprocess.Popen(
                ["caffeinate", "-is"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            logger.info("Sleep prevention enabled (caffeinate PID %d)", proc.pid)
        except OSError:
            logger.debug("Failed to start caffeinate, continuing without sleep prevention")
    try:
        yield
    finally:
        if proc is not None:
            proc.terminate()
            proc.wait(timeout=5)
            logger.info("Sleep prevention disabled")


AA_IDS = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]

WEIGHT_PROFILES = [
    {"function": 4.0, "stability": 8.0, "binding": 2.5, "name": "Balanced", "color": "#2563EB"},
    {"function": 2.0, "stability": 15.0, "binding": 2.0, "name": "Stability-First", "color": "#0EA5E9"},
    {"function": 3.0, "stability": 5.0, "binding": 8.0, "name": "Binding-Optimized", "color": "#10B981"},
    {"function": 8.0, "stability": 4.0, "binding": 3.0, "name": "Function-Maximized", "color": "#14B8A6"},
    {"function": 5.0, "stability": 10.0, "binding": 5.0, "name": "Conservative", "color": "#1D4ED8"},
    {"function": 6.0, "stability": 6.0, "binding": 6.0, "name": "Experimental", "color": "#0891B2"},
]


@dataclass
class ScenarioRuntime:
    scenario_id: str
    target_label: str
    targets: List[str]
    delivery_method: str


class CampaignRunner:
    def __init__(
        self,
        *,
        store: Optional[CampaignStore] = None,
        oracle_path: str | Path = "data/models/functional_oracle.pt",
        device: Optional[str] = None,
    ):
        self.store = store or CampaignStore()
        self.oracle_path = Path(oracle_path)
        self.device = device

        self.embedder: Optional[ManifoldEmbedder] = None
        self.oracle: Optional[FunctionalOracle] = None
        self.clinical: Optional[ClinicalImpactEngine] = None

        self.calibration_profile = self._score_calibration_profile()
        self.uncertainty_weight = 0.8
        self.ood_rank_weight = 1.25
        self.ood_radius = 1.75
        self.ood_loss_weight = 12.0

        # ── DMS quality lookup for evidence-aware optimization ──
        self._dms_lookup: Dict[Tuple[int, str], float] = {}
        self._dms_quality_tensor: Optional[torch.Tensor] = None
        self._load_dms_lookup()

        # ── Double-mutant pairwise epistasis lookup ──
        self._pairwise_dms: Dict[tuple, float] = get_pairwise_epistasis_lookup()

        # ── Structural contact map from WT PDB (for contact & epistasis penalties) ──
        self._wt_contacts: Dict[int, List[Tuple[int, float]]] = {}
        self._ca_coords: Dict[int, np.ndarray] = {}
        self._load_contact_map()

        # ── WT hidden-state cache for contact preservation penalty ──
        self._wt_hidden: Optional[torch.Tensor] = None

        # ── Performance caches (populated lazily, cleared between runs) ──
        self._emb_cache: Dict[str, torch.Tensor] = {}
        self._baseline_cache: Dict[str, Dict[str, float]] = {}

    def _load_dms_lookup(self) -> None:
        """Load raw DMS Z-scores into a (position, amino_acid) → Z-score dict."""
        dms_path = Path(__file__).parent.parent.parent / "data" / "raw" / "p53_DMS_Giacomelli_2018.csv"
        if not dms_path.exists():
            logger.warning("DMS CSV not found at %s; DMS-aware optimization disabled", dms_path)
            return
        try:
            raw = pd.read_csv(dms_path)
            score_col = "A549_p53WT_Nutlin-3_Z-score"
            if score_col not in raw.columns:
                logger.warning("DMS score column '%s' not found; DMS-aware optimization disabled", score_col)
                return
            for _, row in raw.iterrows():
                pos = int(row.get("Position", 0))
                aa = str(row.get("AA_variant", ""))
                z = row.get(score_col)
                if pd.isna(z) or pos < 1 or pos > len(P53_WT) or len(aa) != 1:
                    continue
                self._dms_lookup[(pos, aa.upper())] = float(z)
            logger.info("Loaded %d DMS entries for evidence-aware optimization", len(self._dms_lookup))
        except Exception as exc:
            logger.warning("Failed to load DMS lookup: %s", exc)

    def _load_contact_map(self) -> None:
        """Parse WT PDB and build contact map for structural penalties."""
        pdb_path = Path(__file__).parent.parent.parent / "data" / "raw" / "p53_wt.pdb"
        if not pdb_path.exists():
            logger.warning("WT PDB not found at %s; contact/epistasis penalties disabled", pdb_path)
            return
        try:
            from p53cad.engine.explainability import EnergyDecomposer
            self._ca_coords = EnergyDecomposer._parse_ca_coordinates(str(pdb_path))
            all_positions = sorted(self._ca_coords.keys())
            self._wt_contacts = EnergyDecomposer._find_contacts(
                self._ca_coords, all_positions, cutoff=8.0
            )
            logger.info("Contact map: %d residues, %d contacts loaded",
                        len(self._ca_coords),
                        sum(len(v) for v in self._wt_contacts.values()))
        except Exception as exc:
            logger.warning("Failed to load contact map: %s", exc)

    def _get_wt_hidden(self, device: torch.device) -> Optional[torch.Tensor]:
        """Cache the WT hidden states for contact preservation penalty."""
        if self._wt_hidden is not None and self._wt_hidden.device == device:
            return self._wt_hidden
        if self.embedder is None:
            return None
        emb_wt = self._get_cached_embedding(P53_WT)
        with torch.no_grad():
            h, _, _ = self.embedder.latent_forward_ascent(emb_wt)
            self._wt_hidden = h.detach()
        return self._wt_hidden

    def _get_dms_quality_tensor(self, device: torch.device) -> Optional[torch.Tensor]:
        """Build (seq_len, 20) tensor of raw DMS Z-scores for differentiable DMS penalty.

        Values: raw Nutlin-3 Z-scores with a compensatory dead zone applied:
        - Z < -0.5: kept as-is (strongly functional — safe rescue candidates)
        - -0.5 ≤ Z ≤ 2.0: zeroed out (neutral/compensatory zone — may rescue in context)
        - Z > 2.0: kept as-is (catastrophically LoF — penalize)

        Missing entries default to 0.0 (neutral — no guidance).

        The dead zone prevents penalizing mutations in the Z=+0.14 to +1.59 range
        where known intragenic suppressors (compensatory rescues) are found.
        """
        if not self._dms_lookup:
            return None
        if self._dms_quality_tensor is not None and self._dms_quality_tensor.device == device:
            return self._dms_quality_tensor

        assert self.embedder is not None
        # Map AA letters → index in AA_IDS
        aa_to_idx: Dict[str, int] = {}
        for i, aa_id in enumerate(AA_IDS):
            tokens = self.embedder.tokenizer.convert_ids_to_tokens([aa_id])
            if tokens:
                aa_to_idx[str(tokens[0]).upper()] = i

        tensor = torch.zeros(len(P53_WT), len(AA_IDS), device=device)
        filled = 0
        for (pos, aa), z_score in self._dms_lookup.items():
            if aa in aa_to_idx and 1 <= pos <= len(P53_WT):
                tensor[pos - 1, aa_to_idx[aa]] = z_score
                filled += 1

        # Zero out the compensatory zone where rescue mutations live
        # Z < -0.5: keep (strongly functional — safe rescue candidates)
        # -0.5 ≤ Z ≤ 2.0: zero (neutral — could be compensatory)
        # Z > 2.0: keep (catastrophically LoF — penalize)
        compensatory = (tensor >= -0.5) & (tensor <= 2.0)
        n_zeroed = int(compensatory.sum().item())
        tensor[compensatory] = 0.0

        logger.info(
            "DMS quality tensor: %d/%d entries filled, %d zeroed (compensatory dead zone) on %s",
            filled, len(self._dms_lookup), n_zeroed, device,
        )
        self._dms_quality_tensor = tensor
        return tensor

    def _load_models(self) -> None:
        if self.embedder is not None and self.oracle is not None and self.clinical is not None:
            return
        self.embedder = ManifoldEmbedder(device=self.device)
        self.oracle = FunctionalOracle(model_path=self.oracle_path)
        self.clinical = ClinicalImpactEngine()

    def _get_cached_embedding(self, seq: str) -> torch.Tensor:
        """Return detached embedding for *seq*, using cache to avoid redundant ESM-2 passes."""
        if seq not in self._emb_cache:
            assert self.embedder is not None
            self._emb_cache[seq] = self.embedder.get_embeddings(seq).detach()
        return self._emb_cache[seq]

    def _get_cached_baseline(
        self, target_seq: str, pooled_target_ref: torch.Tensor, mc_samples: int
    ) -> Dict[str, float]:
        """Return baseline metrics for *target_seq*, using cache for shared targets."""
        if target_seq not in self._baseline_cache:
            self._baseline_cache[target_seq] = self._compute_baseline_metrics(
                target_seq, pooled_target_ref, mc_samples=mc_samples
            )
        return self._baseline_cache[target_seq]

    def run(
        self,
        *,
        run_id: Optional[str] = None,
        seed: int = 42,
        resume: bool = True,
        include_pairs: bool = True,
        hotspots: Sequence[str] | None = None,
        delivery_methods: Sequence[str] | None = None,
        max_scenarios: Optional[int] = None,
        shortlist_n: int = 30,
        budget: str = "high",
        with_clinical: bool = True,
    ) -> Dict[str, Any]:
        with _prevent_sleep():
            return self._run_inner(
                run_id=run_id, seed=seed, resume=resume,
                include_pairs=include_pairs, hotspots=hotspots,
                delivery_methods=delivery_methods,
                max_scenarios=max_scenarios, shortlist_n=shortlist_n,
                budget=budget, with_clinical=with_clinical,
            )

    def _run_inner(
        self,
        *,
        run_id: Optional[str] = None,
        seed: int = 42,
        resume: bool = True,
        include_pairs: bool = True,
        hotspots: Sequence[str] | None = None,
        delivery_methods: Sequence[str] | None = None,
        max_scenarios: Optional[int] = None,
        shortlist_n: int = 30,
        budget: str = "high",
        with_clinical: bool = True,
    ) -> Dict[str, Any]:
        self._load_models()
        assert self.embedder is not None
        assert self.oracle is not None
        assert self.clinical is not None

        # Clear per-run caches
        self._emb_cache.clear()
        self._baseline_cache.clear()

        run_id = run_id or build_run_id("campaign")
        scenarios = build_scenario_matrix(
            hotspots=hotspots or BIG8_HOTSPOTS,
            delivery_methods=delivery_methods or DEFAULT_DELIVERY_METHODS,
            include_pairs=include_pairs,
        )
        if max_scenarios is not None:
            scenarios = scenarios[: int(max_scenarios)]

        config = {
            "seed": int(seed),
            "budget": budget,
            "include_pairs": bool(include_pairs),
            "hotspots": list(hotspots or BIG8_HOTSPOTS),
            "delivery_methods": list(delivery_methods or DEFAULT_DELIVERY_METHODS),
            "max_scenarios": int(max_scenarios) if max_scenarios is not None else None,
            "shortlist_n": int(shortlist_n),
            "with_clinical": bool(with_clinical),
            "pass_a": self._pass_a_config(budget),
            "pass_b": self._pass_b_config(budget),
        }

        runtime_caps = {
            "device": str(self.embedder.device),
            "oracle_path": str(self.oracle_path),
        }
        paths = self.store.init_run(run_id, config=config, runtime_caps=runtime_caps, resume=resume)
        self.store.update_manifest(run_id, {"status": "running"})

        ckpt_dir = paths.checkpoint_state_path.parent
        candidates_jsonl = ckpt_dir / "candidates.jsonl"
        trajectories_jsonl = ckpt_dir / "trajectories.jsonl"
        clinical_jsonl = ckpt_dir / "clinical.jsonl"
        scenarios_jsonl = ckpt_dir / "scenario_metrics.jsonl"

        done_df = self.store.load_scenario_checkpoints(run_id)
        done_pairs = set()
        if not done_df.empty:
            for _, row in done_df.iterrows():
                if str(row.get("status", "")) == "done":
                    done_pairs.add((str(row.get("scenario_id")), str(row.get("pass_name"))))

        pass_a_cfg = self._pass_a_config(budget)
        pass_b_cfg = self._pass_b_config(budget)

        scenario_objs = [
            ScenarioRuntime(
                scenario_id=s.scenario_id,
                target_label=s.target_label,
                targets=list(s.targets),
                delivery_method=s.delivery_method,
            )
            for s in scenarios
        ]

        # Pass A screening
        pass_a_best: Dict[str, float] = {}
        for idx, sc in enumerate(scenario_objs, start=1):
            key = (sc.scenario_id, "screen")
            if resume and key in done_pairs:
                continue
            logger.info("Pass A [%d/%d] %s", idx, len(scenario_objs), sc.scenario_id)
            out = self._run_scenario(sc, pass_name="screen", config=pass_a_cfg, seed=seed)
            self._append_rows(candidates_jsonl, out["candidates"])
            self._append_rows(trajectories_jsonl, out["trajectories"])
            self._append_rows(scenarios_jsonl, [out["scenario_metrics"]])
            pass_a_best[sc.scenario_id] = float(out["scenario_metrics"].get("best_score", -np.inf))
            self.store.append_scenario_checkpoint(
                run_id,
                {
                    "scenario_id": sc.scenario_id,
                    "pass_name": "screen",
                    "status": "done",
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                },
            )

        # Load pass A from checkpoint to select pass B scenarios.
        screen_df = self._read_jsonl(scenarios_jsonl)
        if screen_df.empty:
            raise RuntimeError("No screen metrics found; campaign cannot continue.")
        screen_df = screen_df[screen_df["pass_name"] == "screen"].copy()
        screen_df["best_score"] = pd.to_numeric(screen_df["best_score"], errors="coerce").fillna(-np.inf)
        screen_df = screen_df.sort_values("best_score", ascending=False)
        top_count = max(1, int(np.ceil(len(screen_df) * 0.4)))

        # Delivery diversity floor: ensure at least top-2 per delivery method
        # reach Pass B, then fill remaining with global top scores.
        delivery_floor = 2
        selected_pass_b: set[str] = set()
        if "scenario_id" in screen_df.columns:
            dm_col = screen_df["scenario_id"].astype(str).str.rsplit("__", n=1)
            screen_df["_delivery"] = dm_col.str[-1]
            for dm in DEFAULT_DELIVERY_METHODS:
                dm_rows = screen_df[screen_df["_delivery"] == dm]
                for sid in dm_rows.head(delivery_floor)["scenario_id"].astype(str):
                    selected_pass_b.add(sid)
            # Fill remaining slots from global ranking
            for sid in screen_df["scenario_id"].astype(str):
                if len(selected_pass_b) >= top_count:
                    break
                selected_pass_b.add(sid)
            screen_df.drop(columns=["_delivery"], inplace=True)
        else:
            selected_pass_b = set(screen_df.head(top_count)["scenario_id"].astype(str).tolist())

        # Profile pruning: pick top-3 profiles per scenario from Pass A candidates.
        screen_cands = self._read_jsonl(candidates_jsonl)
        profile_map: Dict[str, Optional[List[str]]] = {}
        if not screen_cands.empty and "profile" in screen_cands.columns and "score" in screen_cands.columns:
            sc_screen = screen_cands[screen_cands["pass_name"] == "screen"].copy() if "pass_name" in screen_cands.columns else screen_cands.copy()
            sc_screen["score"] = pd.to_numeric(sc_screen["score"], errors="coerce").fillna(-np.inf)
            for sid in selected_pass_b:
                sc_rows = sc_screen[sc_screen["scenario_id"] == sid]
                if sc_rows.empty:
                    profile_map[sid] = None  # fallback: all profiles
                else:
                    top_profiles = (
                        sc_rows.groupby("profile")["score"]
                        .max()
                        .sort_values(ascending=False)
                        .head(3)
                        .index.tolist()
                    )
                    profile_map[sid] = top_profiles if len(top_profiles) >= 1 else None
        logger.info(
            "Pass B: %d scenarios selected (%.0f%%), profiles pruned to top-3 per scenario",
            len(selected_pass_b),
            100.0 * len(selected_pass_b) / max(len(screen_df), 1),
        )

        # Pass B deep refinement on top scenarios only.
        selected_objs = [sc for sc in scenario_objs if sc.scenario_id in selected_pass_b]
        for idx, sc in enumerate(selected_objs, start=1):
            key = (sc.scenario_id, "deep")
            if resume and key in done_pairs:
                continue
            prune = profile_map.get(sc.scenario_id)
            logger.info("Pass B [%d/%d] %s  profiles=%s", idx, len(selected_objs), sc.scenario_id, prune or "all")
            out = self._run_scenario(sc, pass_name="deep", config=pass_b_cfg, seed=seed, allowed_profiles=prune)
            self._append_rows(candidates_jsonl, out["candidates"])
            self._append_rows(trajectories_jsonl, out["trajectories"])
            self._append_rows(scenarios_jsonl, [out["scenario_metrics"]])

            if with_clinical:
                clinical_rows = self._run_clinical_for_scenario(out["candidates"]) 
                self._append_rows(clinical_jsonl, clinical_rows)

            self.store.append_scenario_checkpoint(
                run_id,
                {
                    "scenario_id": sc.scenario_id,
                    "pass_name": "deep",
                    "status": "done",
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                },
            )

        # Build final parquet artifacts from checkpoints.
        scenario_df = self._read_jsonl(scenarios_jsonl)
        candidate_df = self._read_jsonl(candidates_jsonl)
        trajectory_df = self._read_jsonl(trajectories_jsonl)
        clinical_df = self._read_jsonl(clinical_jsonl)

        if not candidate_df.empty and not clinical_df.empty and "candidate_uid" in candidate_df.columns:
            clinical_agg = clinical_df.sort_values("clinical_score", ascending=False).drop_duplicates("candidate_uid")
            clinical_agg = clinical_agg[["candidate_uid", "clinical_score", "clinical_viability"]]
            candidate_df = candidate_df.merge(clinical_agg, on="candidate_uid", how="left")

        self.store.write_tables(
            run_id,
            scenarios_df=scenario_df,
            candidates_df=candidate_df,
            trajectories_df=trajectory_df,
            clinical_df=clinical_df,
        )

        top30 = select_presentation_shortlist(
            candidate_df[candidate_df["pass_name"] == "deep"].copy() if "pass_name" in candidate_df.columns else candidate_df,
            top_n=shortlist_n,
            max_per_target=4,
            delivery_methods=delivery_methods or DEFAULT_DELIVERY_METHODS,
        )
        self.store.write_top30(run_id, top30)

        summary = self._build_summary(run_id, scenario_df, candidate_df, top30)
        self.store.write_summary(run_id, summary)
        self.store.update_manifest(
            run_id,
            {
                "status": "completed",
                "n_scenarios": int(len(scenario_objs)),
                "n_candidates": int(len(candidate_df)),
                "n_trajectories": int(len(trajectory_df)),
                "n_shortlist": int(len(top30)),
            },
        )

        # Post-campaign analysis: ESMFold validation, MD scripts, PyMOL scripts
        post_results = self._post_campaign_analysis(run_id, top30, paths)

        return {
            "run_id": run_id,
            "run_dir": str(paths.run_dir),
            "n_scenarios": int(len(scenario_objs)),
            "n_candidates": int(len(candidate_df)),
            "n_shortlist": int(len(top30)),
            "selected_pass_b": int(len(selected_objs)),
            "post_analysis": post_results,
        }

    def _post_campaign_analysis(
        self,
        run_id: str,
        top_df: pd.DataFrame,
        paths: Any,
    ) -> Dict[str, Any]:
        """Run post-campaign analysis: ESMFold validation, MD scripts, PyMOL scripts."""
        results: Dict[str, Any] = {}
        run_dir = paths.run_dir

        # 1. ESMFold structural validation for all shortlisted candidates
        try:
            from p53cad.engine.md_validation import StructurePredictor, FreeEnergyCalculator

            predictor = StructurePredictor()
            fe_calc = FreeEnergyCalculator()
            validations = []

            for _, row in top_df.iterrows():
                seq = str(row.get("sequence", ""))
                target_label = str(row.get("target_label", "unknown"))
                muts = json.loads(str(row.get("mutations_json", "[]")))
                rescue_muts = [m for m in muts if m not in json.loads(str(row.get("targets_json", "[]")))]

                pred = predictor.predict_esmfold(seq, name=target_label)
                fe_results = [fe_calc.calculate_ddg(P53_WT, seq, m) for m in rescue_muts]

                validations.append({
                    "target_label": target_label,
                    "rescue_mutations": rescue_muts,
                    "mean_plddt": float(pred.mean_plddt),
                    "model": pred.model,
                    "n_residues": len(seq),
                    "mean_ddg": float(np.mean([fe.ddg_average for fe in fe_results])) if fe_results else 0.0,
                    "max_ddg": float(max((fe.ddg_average for fe in fe_results), default=0.0)),
                    "is_estimated": pred.pdb_string.startswith("HEADER    ESTIMATED"),
                })

            val_path = run_dir / "esmfold_validation.json"
            val_path.write_text(json.dumps(validations, indent=2), encoding="utf-8")
            results["esmfold_validations"] = len(validations)
            logger.info("ESMFold validation: %d candidates validated -> %s", len(validations), val_path)
        except Exception as exc:
            logger.warning("ESMFold validation failed: %s", exc)
            results["esmfold_error"] = str(exc)

        # 2. Generate MD validation scripts for top 5 candidates
        try:
            from p53cad.engine.md_validation import MDSimulationGenerator

            md_gen = MDSimulationGenerator()
            md_dir = run_dir / "md_scripts"
            md_dir.mkdir(exist_ok=True)
            n_md = 0

            for _, row in top_df.head(5).iterrows():
                seq = str(row.get("sequence", ""))
                target_label = str(row.get("target_label", "unknown"))
                muts = json.loads(str(row.get("mutations_json", "[]")))
                safe_name = target_label.replace("+", "_").replace(" ", "_")
                config = md_gen.generate_config(safe_name, seq, muts)

                for plat in ["openmm", "gromacs"]:
                    script = md_gen.generate_script(config, plat)
                    ext = ".py" if plat == "openmm" else ".sh"
                    script_path = md_dir / f"{safe_name}_{plat}{ext}"
                    script_path.write_text(script, encoding="utf-8")
                    n_md += 1

            results["md_scripts"] = n_md
            logger.info("MD scripts: %d files generated -> %s", n_md, md_dir)
        except Exception as exc:
            logger.warning("MD script generation failed: %s", exc)
            results["md_error"] = str(exc)

        # 3. Generate PyMOL visualization scripts for top 5 candidates
        try:
            from p53cad.viz.pymol import PyMolGenerator

            pymol_gen = PyMolGenerator()
            pymol_paths = pymol_gen.generate_campaign_pml(top_df, run_dir / "pymol", top_n=5)
            results["pymol_scripts"] = len(pymol_paths)
            logger.info("PyMOL scripts: %d files generated", len(pymol_paths))
        except Exception as exc:
            logger.warning("PyMOL script generation failed: %s", exc)
            results["pymol_error"] = str(exc)

        return results

    def report_run(self, run_id: str, shortlist_n: int = 30) -> Dict[str, Any]:
        bundle = self.store.load_run_bundle(run_id)
        candidate_df = bundle["candidates"]
        if candidate_df.empty:
            raise RuntimeError(f"No candidates found for run {run_id}")

        top30 = select_presentation_shortlist(
            candidate_df[candidate_df["pass_name"] == "deep"].copy() if "pass_name" in candidate_df.columns else candidate_df,
            top_n=shortlist_n,
            max_per_target=4,
            delivery_methods=DEFAULT_DELIVERY_METHODS,
        )
        self.store.write_top30(run_id, top30)
        summary = self._build_summary(run_id, bundle["scenarios"], candidate_df, top30)
        self.store.write_summary(run_id, summary)
        self.store.update_manifest(run_id, {"status": "reported", "n_shortlist": int(len(top30))})
        return {
            "run_id": run_id,
            "n_candidates": int(len(candidate_df)),
            "n_shortlist": int(len(top30)),
            "run_dir": str(bundle["run_dir"]),
        }

    def _pass_a_config(self, budget: str) -> Dict[str, Any]:
        if budget == "fast":
            return {"steps": 40, "restarts": 1, "repeats": 1, "mc_samples": 2}
        if budget == "medium":
            return {"steps": 60, "restarts": 1, "repeats": 1, "mc_samples": 3}
        return {"steps": 80, "restarts": 1, "repeats": 1, "mc_samples": 4}

    def _pass_b_config(self, budget: str) -> Dict[str, Any]:
        if budget == "fast":
            return {"steps": 120, "restarts": 1, "repeats": 1, "mc_samples": 4}
        if budget == "medium":
            return {"steps": 200, "restarts": 2, "repeats": 2, "mc_samples": 8}
        # High: reduced from 3×3=9 to 2×2=4 trials per profile (55% fewer trials)
        return {"steps": 280, "restarts": 2, "repeats": 2, "mc_samples": 12}

    def _run_scenario(
        self,
        scenario: ScenarioRuntime,
        *,
        pass_name: str,
        config: Dict[str, int],
        seed: int,
        allowed_profiles: Optional[List[str]] = None,
    ) -> Dict[str, Any]:
        assert self.embedder is not None
        assert self.oracle is not None

        target_seq = self._build_target_sequence(scenario.targets)
        emb_target_ref = self._get_cached_embedding(target_seq)
        emb_wt = self._get_cached_embedding(P53_WT)
        with torch.no_grad():
            z_target_ref, _, _ = self.embedder.latent_forward_ascent(emb_target_ref)
            pooled_target_ref = z_target_ref.mean(dim=1)
            if pooled_target_ref.shape[-1] != self.oracle.input_dim:
                pooled_target_ref = pooled_target_ref[:, :self.oracle.input_dim]

        wt_aa_tensor = self._wt_aa_tensor(device=emb_target_ref.device)

        min_identity = self._delivery_identity_floor(scenario.delivery_method)
        min_stability = -0.2
        min_binding = 5.0
        lock_positions: set[int] = set()
        for target in scenario.targets:
            pos = self._mutation_pos(target)
            if pos is not None:
                lock_positions.add(int(pos))
        lock_positions.update([175, 248, 273])
        lock_positions_sorted = sorted(lock_positions)
        max_mutations = int(len(P53_WT) * (100 - min_identity) / 100)
        locked_indices = [p - 1 for p in lock_positions_sorted]

        baseline = self._get_cached_baseline(target_seq, pooled_target_ref, mc_samples=config["mc_samples"])
        dms_quality = self._get_dms_quality_tensor(emb_target_ref.device)

        # Pre-compute conditional rescue scores: ESM-2 P(aa | cancer_context)
        # at candidate positions in the DNA-binding domain (DBD).
        candidate_positions = [p for p in range(94, 293) if (p - 1) not in locked_indices]
        conditional_scores: Optional[Dict[int, Dict[str, float]]] = None
        try:
            conditional_scores = compute_conditional_rescue_scores(
                self.embedder, target_seq, candidate_positions[:50]
            )
        except Exception as exc:
            logger.warning("Conditional rescue score computation failed: %s", exc)

        candidates: List[Dict[str, Any]] = []
        trajectories: List[Dict[str, Any]] = []

        profiles = WEIGHT_PROFILES
        if allowed_profiles is not None:
            allowed_set = set(allowed_profiles)
            profiles = [p for p in WEIGHT_PROFILES if p["name"] in allowed_set]

        trial_counter = 0
        for repeat_idx in range(int(config["repeats"])):
            for profile in profiles:
                for restart_idx in range(int(config["restarts"])):
                    trial_counter += 1
                    trial_seed = int(seed + (repeat_idx * 1000) + (restart_idx * 100) + trial_counter)
                    cand, traj_rows = self._run_single_trial(
                        scenario=scenario,
                        target_seq=target_seq,
                        emb_target_ref=emb_target_ref,
                        emb_wt=emb_wt,
                        pooled_target_ref=pooled_target_ref,
                        wt_aa_tensor=wt_aa_tensor,
                        profile=profile,
                        pass_name=pass_name,
                        repeat_idx=repeat_idx + 1,
                        restart_idx=restart_idx + 1,
                        trial_idx=trial_counter,
                        trial_seed=trial_seed,
                        n_steps=int(config["steps"]),
                        mc_samples=int(config["mc_samples"]),
                        min_identity=min_identity,
                        min_stability=min_stability,
                        min_binding=min_binding,
                        max_mutations=max_mutations,
                        locked_indices=locked_indices,
                        baseline_score=float(baseline["score"]),
                        dms_quality=dms_quality,
                        conditional_scores=conditional_scores,
                    )
                    candidates.append(cand)
                    trajectories.extend(traj_rows)

        # Run one autoregressive trial per scenario for diversity
        try:
            ar_cand, ar_traj = self._run_autoregressive_trial(
                scenario=scenario,
                target_seq=target_seq,
                locked_indices=locked_indices,
                max_mutations=max_mutations,
                pass_name=pass_name,
                trial_idx=trial_counter + 1,
                trial_seed=seed + 9999,
                baseline_score=float(baseline["score"]),
            )
            candidates.append(ar_cand)
            trajectories.extend(ar_traj)
        except Exception as exc:
            logger.warning("Autoregressive trial failed: %s", exc)

        # Multi-objective Pareto ranking across all trial candidates
        try:
            from p53cad.engine.pareto import ParetoFront, ParetoSolution
            pareto = ParetoFront()
            for c in candidates:
                pareto.add(ParetoSolution(
                    candidate_uid=str(c.get("candidate_uid", "")),
                    objectives={
                        "oracle_score": float(c.get("score", 0.0)),
                        "pll": float(c.get("stability", 0.0)),
                        "dms_quality": -float(c.get("rescue_dms_mean", 0.0)),
                        "identity": float(c.get("identity", 100.0)),
                        "stability": float(c.get("stability", 0.0)),
                    },
                ))
            ranked = pareto.solutions
            uid_to_rank = {s.candidate_uid: s.pareto_rank for s in ranked}
            for c in candidates:
                c["pareto_rank"] = uid_to_rank.get(str(c.get("candidate_uid", "")), 999)
        except Exception as exc:
            logger.warning("Pareto ranking failed: %s", exc)
            for c in candidates:
                c["pareto_rank"] = 0

        best_score = max((float(c.get("score", -np.inf)) for c in candidates), default=-np.inf)
        metrics = {
            "scenario_id": scenario.scenario_id,
            "target_label": scenario.target_label,
            "targets": json.dumps(scenario.targets),
            "delivery_method": scenario.delivery_method,
            "pass_name": pass_name,
            "n_candidates": int(len(candidates)),
            "best_score": float(best_score),
            "baseline_score": float(baseline["score"]),
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        return {"candidates": candidates, "trajectories": trajectories, "scenario_metrics": metrics}

    def _run_single_trial(
        self,
        *,
        scenario: ScenarioRuntime,
        target_seq: str,
        emb_target_ref: torch.Tensor,
        emb_wt: torch.Tensor,
        pooled_target_ref: torch.Tensor,
        wt_aa_tensor: torch.Tensor,
        profile: Dict[str, Any],
        pass_name: str,
        repeat_idx: int,
        restart_idx: int,
        trial_idx: int,
        trial_seed: int,
        n_steps: int,
        mc_samples: int,
        min_identity: float,
        min_stability: float,
        min_binding: float,
        max_mutations: int,
        locked_indices: List[int],
        baseline_score: float,
        dms_quality: Optional[torch.Tensor] = None,
        conditional_scores: Optional[Dict[int, Dict[str, float]]] = None,
    ) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
        assert self.embedder is not None
        assert self.oracle is not None

        torch.manual_seed(trial_seed)
        np.random.seed(trial_seed)

        emb = emb_target_ref.clone().detach().requires_grad_(True)
        with torch.no_grad():
            emb.data += torch.randn_like(emb) * 0.05

        optimizer = torch.optim.Adam([emb], lr=0.03)

        # Pre-compute WT hidden states for contact preservation penalty
        wt_hidden = self._get_wt_hidden(emb.device)

        # Pre-compute contact neighbor indices for locked (mutating) positions
        # These are the 0-indexed positions whose structural neighbours we monitor
        contact_neighbor_indices: List[List[int]] = []
        has_contacts = bool(self._wt_contacts and wt_hidden is not None)
        if has_contacts:
            for li in locked_indices:
                res_id = li + 1  # 1-indexed
                neighbors = self._wt_contacts.get(res_id, [])
                # Convert neighbor residue IDs to 0-indexed, filter to valid range
                ni = [r - 1 for r, _ in neighbors if 0 <= r - 1 < len(P53_WT)]
                contact_neighbor_indices.append(ni)

        # Pre-compute pairwise Cα distances for epistasis proximity penalty
        # We track pairs of all mutatable (non-locked) positions that are close
        _epistasis_pairs: List[Tuple[int, int, float]] = []  # (idx_i, idx_j, distance)
        if self._ca_coords:
            all_pos = list(range(len(P53_WT)))
            locked_set = set(locked_indices)
            for i_idx in range(len(all_pos)):
                if i_idx in locked_set:
                    continue
                ri = i_idx + 1  # 1-indexed
                if ri not in self._ca_coords:
                    continue
                for j_idx in range(i_idx + 1, len(all_pos)):
                    if j_idx in locked_set:
                        continue
                    rj = j_idx + 1
                    if rj not in self._ca_coords:
                        continue
                    diff = self._ca_coords[ri] - self._ca_coords[rj]
                    dist = float(np.sqrt(np.sum(diff ** 2)))
                    if dist < 10.0:
                        _epistasis_pairs.append((i_idx, j_idx, dist))
        cached_epistasis_loss = torch.zeros(1, device=emb.device)

        # Pre-compute position-weighted pooling kernel for mutation-neighborhood oracle.
        # Gaussian centered on each locked_index with σ=10 residues — upweights positions
        # near cancer sites so the oracle sees a stronger signal from rescue mutations.
        seq_len = emb.size(1)
        pool_weights = torch.ones(seq_len, device=emb.device)
        for li in locked_indices:
            dists = (torch.arange(seq_len, device=emb.device).float() - li)
            pool_weights += torch.exp(-dists**2 / 200.0)  # σ=10 → 2σ²=200
        pool_weights = pool_weights / pool_weights.sum()  # (L,)

        trajectory: List[Dict[str, Any]] = []
        best_valid_state: Optional[Dict[str, Any]] = None
        best_valid_score = -float("inf")
        checks_without_improve = 0
        cached_uncertainty = 0.0

        for step_idx in range(1, int(n_steps) + 1):
            optimizer.zero_grad()
            z, logits, _ = self.embedder.latent_forward_ascent(emb)
            pooled = z.mean(dim=1)
            if pooled.shape[-1] != self.oracle.input_dim:
                pooled = pooled[:, :self.oracle.input_dim]

            # Attention oracle receives full per-position embeddings;
            # legacy MLP oracle receives mean-pooled embeddings.
            _uses_attention = isinstance(self.oracle.model, AttentionPoolingNet)
            if _uses_attention:
                z_oracle = z
                if z_oracle.shape[-1] != self.oracle.input_dim:
                    z_oracle = z_oracle[:, :, :self.oracle.input_dim]
                raw_score_t = self.oracle.model(z_oracle).squeeze(-1)
            else:
                raw_score_t = self.oracle.model(pooled).squeeze(-1)

            # Mutation-neighborhood oracle: position-weighted pooling that amplifies
            # the signal near cancer sites instead of averaging uniformly over 393 positions.
            pooled_local = (z.squeeze(0) * pool_weights.unsqueeze(1)).sum(dim=0, keepdim=True)  # (1, D)
            if pooled_local.shape[-1] != self.oracle.input_dim:
                pooled_local = pooled_local[:, :self.oracle.input_dim]
            if _uses_attention:
                # Attention oracle already does position weighting natively
                local_score_t = raw_score_t
            else:
                local_score_t = self.oracle.model(pooled_local).squeeze(-1)
            local_score_term = -2.0 * local_score_t

            probs_full = torch.softmax(logits, dim=-1)
            logits_aa = logits[:, :, AA_IDS]
            log_probs = F.log_softmax(logits_aa, dim=-1)
            stability_t = log_probs.max(dim=-1).values.mean(dim=-1)
            dna_force_t = self._batch_dna_force(z, probs_full)
            hydro_packing_t = self._batch_hydrophobic_packing(probs_full)
            ood_distance_t = torch.norm(pooled - pooled_target_ref, p=2, dim=-1)

            probs_aa = torch.softmax(logits_aa, dim=-1)

            # PLL: expected pseudo-log-likelihood — how natural this sequence looks to ESM-2
            expected_pll = (probs_aa * log_probs).sum(dim=-1).mean(dim=-1)  # [batch]

            wt_probs = probs_aa[:, torch.arange(len(P53_WT), device=emb.device), wt_aa_tensor]
            expected_mutations = (1.0 - wt_probs).sum(dim=-1)
            expected_identity = 100.0 * (1.0 - expected_mutations / float(len(P53_WT)))

            score_term = -raw_score_t * float(profile["function"])
            stability_term = -float(profile["stability"]) * stability_t
            binding_term = -float(profile["binding"]) * dna_force_t
            hydro_term = -3.0 * hydro_packing_t
            ood_penalty = self.ood_loss_weight * F.relu(ood_distance_t - self.ood_radius)
            mutation_penalty = 50.0 * F.relu(expected_mutations - max_mutations)
            identity_penalty = 500.0 * F.relu((min_identity - 5.0) - expected_identity)
            stability_penalty = 100.0 * F.relu(min_stability - stability_t)
            binding_penalty = 80.0 * F.relu(min_binding - dna_force_t)
            if locked_indices:
                lock_penalty = 800.0 * (emb[:, locked_indices, :] - emb_target_ref[:, locked_indices, :]).pow(2).mean(dim=(1, 2))
            else:
                lock_penalty = torch.zeros_like(raw_score_t)
            if locked_indices:
                l1_mask = torch.ones(emb.size(1), device=emb.device, dtype=torch.bool)
                for li in locked_indices:
                    l1_mask[li] = False
                l1_penalty = 40.0 * (emb[:, l1_mask, :] - emb_wt[:, l1_mask, :]).abs().mean(dim=(1, 2))
            else:
                l1_penalty = 40.0 * (emb - emb_wt).abs().mean(dim=(1, 2))

            # DMS quality penalty: steer optimizer toward individually functional rescue mutations.
            # dms_quality has raw Nutlin-3 Z-scores: negative = functional (good), positive = LoF (bad).
            # Expected DMS = sum_over_AAs(P(aa|pos) * Z(pos, aa)), weighted by mutation probability.
            if dms_quality is not None:
                expected_dms = (probs_aa * dms_quality.unsqueeze(0)).sum(dim=-1)  # (batch, seq_len)
                mut_prob = 1.0 - wt_probs  # probability of mutation at each position
                weighted_dms = mut_prob * expected_dms
                # Only apply to non-locked positions (rescue sites, not cancer targets)
                dms_mask = l1_mask if locked_indices else torch.ones(emb.size(1), device=emb.device, dtype=torch.bool)
                dms_penalty = 10.0 * weighted_dms[:, dms_mask].mean(dim=-1)
            else:
                dms_penalty = torch.zeros_like(raw_score_t)

            # Epistasis penalty: computed every 10 steps to limit overhead.
            # Uses detached probs to avoid stale-graph errors on subsequent backward().
            if _epistasis_pairs and step_idx % 10 == 1:
                probs_aa_d = probs_aa.detach()
                wt_probs_d = wt_probs.detach()

                # A. Structural proximity: penalize co-mutations at close positions
                proximity_val = 0.0
                for i_idx, j_idx, dist in _epistasis_pairs:
                    mp_i = float((1.0 - probs_aa_d[:, i_idx, wt_aa_tensor[i_idx]]).item())
                    mp_j = float((1.0 - probs_aa_d[:, j_idx, wt_aa_tensor[j_idx]]).item())
                    proximity_val += mp_i * mp_j * math.exp(-dist / 5.0)
                n_pairs = max(len(_epistasis_pairs), 1)
                proximity_val /= n_pairs

                # B. Attention coupling: high mutual attention = functionally coupled
                attn_val = 0.0
                try:
                    with torch.no_grad():
                        _, _, _, attns = self.embedder.latent_forward_ascent(
                            emb.detach(), return_attention=True
                        )
                    # attns is a tuple of (1, heads, L, L) per layer — average all
                    attn_stack = torch.stack([a.mean(dim=1) for a in attns]).mean(dim=0)  # (1, L, L)
                    for i_idx, j_idx, _ in _epistasis_pairs:
                        mp_i = float((1.0 - probs_aa_d[:, i_idx, wt_aa_tensor[i_idx]]).item())
                        mp_j = float((1.0 - probs_aa_d[:, j_idx, wt_aa_tensor[j_idx]]).item())
                        mutual_attn = float(((attn_stack[:, i_idx, j_idx] + attn_stack[:, j_idx, i_idx]) / 2.0).item())
                        attn_val += mp_i * mp_j * mutual_attn
                    attn_val /= n_pairs
                except Exception:
                    pass

                cached_epistasis_loss = torch.tensor(
                    2.0 * (proximity_val + attn_val), device=emb.device
                )

            epistasis_penalty = cached_epistasis_loss.expand_as(raw_score_t)

            # PLL term: maximize sequence naturalness (negative because we minimize loss)
            pll_term = -3.0 * expected_pll

            # Cancer-site PLL: maximize log-probability specifically at locked (cancer) positions.
            # This directly measures "does the current sequence context make ESM-2 more confident
            # about the cancer residue?" — provides focused gradient that global PLL dilutes.
            if locked_indices:
                cancer_site_lp = log_probs[:, locked_indices, :].max(dim=-1).values.mean(dim=-1)
                cancer_pll_term = -5.0 * cancer_site_lp
            else:
                cancer_pll_term = torch.zeros_like(raw_score_t)

            # Contact preservation: penalize when mutated positions disrupt structural neighbors
            if has_contacts and wt_hidden is not None:
                cos_sims = []
                for ni_list in contact_neighbor_indices:
                    if not ni_list:
                        continue
                    ni_t = torch.tensor(ni_list, device=emb.device)
                    cur_h = z[:, ni_t, :]     # (batch, n_neighbors, D)
                    wt_h = wt_hidden[:, ni_t, :]  # (1, n_neighbors, D)
                    # Cosine similarity per neighbor, averaged
                    sim = F.cosine_similarity(cur_h, wt_h, dim=-1).mean(dim=-1)  # (batch,)
                    cos_sims.append(sim)
                if cos_sims:
                    mean_cos = torch.stack(cos_sims).mean(dim=0)  # (batch,)
                    contact_penalty = 5.0 * (1.0 - mean_cos)
                else:
                    contact_penalty = torch.zeros_like(raw_score_t)
            else:
                contact_penalty = torch.zeros_like(raw_score_t)

            # Conditional DMS term: reward mutations that ESM-2 predicts are favorable
            # in the cancer context (not just individually functional on WT).
            if conditional_scores:
                cond_log_prob = torch.zeros_like(raw_score_t)
                for pos, aa_scores in conditional_scores.items():
                    idx = pos - 1
                    if 0 <= idx < probs_aa.shape[1]:
                        for aa_char, lp in aa_scores.items():
                            tok_id = self.embedder.tokenizer.convert_tokens_to_ids(aa_char)
                            if tok_id in AA_IDS:
                                aa_idx = AA_IDS.index(tok_id)
                                cond_log_prob += probs_aa[:, idx, aa_idx] * lp
                cond_rescue_term = -2.0 * cond_log_prob / max(len(conditional_scores), 1)
            else:
                cond_rescue_term = torch.zeros_like(raw_score_t)

            # DBD structural confidence: use ESM-2's per-position confidence as a
            # fast proxy for structural integrity (no extra forward pass needed).
            dbd_range = list(range(93, 292))
            dbd_confidence = log_probs[:, dbd_range, :].max(dim=-1).values.mean(dim=-1)
            structure_term = -3.0 * dbd_confidence

            loss_vec = (
                score_term
                + local_score_term
                + stability_term
                + binding_term
                + hydro_term
                + pll_term
                + cancer_pll_term
                + contact_penalty
                + epistasis_penalty
                + ood_penalty
                + mutation_penalty
                + identity_penalty
                + stability_penalty
                + binding_penalty
                + lock_penalty
                + l1_penalty
                + dms_penalty
                + cond_rescue_term
                + structure_term
            )

            loss = loss_vec.mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_([emb], max_norm=1.0)
            optimizer.step()

            if step_idx % 5 != 0 and step_idx != n_steps:
                continue

            with torch.no_grad():
                raw_score = float(raw_score_t[0].item())
                ood_distance = float(ood_distance_t[0].item())
                if step_idx == 5 or step_idx == n_steps or step_idx % 40 == 0:
                    # Use fewer MC samples for intermediate checks; full count only at final step
                    intermediate_mc = min(mc_samples, 4)
                    unc_samples = mc_samples if step_idx == n_steps else intermediate_mc
                    cached_uncertainty = self._estimate_uncertainty(pooled, mc_samples=unc_samples, z_full=z)

                score_bundle = self._build_trust_adjusted_score(
                    raw_score=raw_score,
                    pooled=pooled,
                    pooled_ref=pooled_target_ref,
                    cached_uncertainty=cached_uncertainty,
                )

                current_seq = self._decode_sequence(logits_aa)
                muts = [f"{P53_WT[j]}{j+1}{current_seq[j]}" for j in range(len(P53_WT)) if P53_WT[j] != current_seq[j]]
                mut_positions = [self._mutation_pos(m) for m in muts]
                mut_positions = [int(p) for p in mut_positions if p is not None]
                seq_identity = 100.0 * (1.0 - len(muts) / float(len(P53_WT)))

                state = {
                    "step": int(step_idx),
                    "score": float(score_bundle["score_adjusted"]),
                    "score_raw": float(score_bundle["score_raw"]),
                    "score_calibrated": float(score_bundle["score_calibrated"]),
                    "stability": float(stability_t[0].item()),
                    "binding": float(dna_force_t[0].item()),
                    "identity": float(seq_identity),
                    "n_mutations": int(len(muts)),
                    "mutations_json": json.dumps(muts),
                    "mut_positions_json": json.dumps(mut_positions),
                    "sequence": current_seq,
                    "uncertainty": float(score_bundle["uncertainty"]),
                    "ood_distance": float(score_bundle["ood_distance"]),
                    "lx": float(pooled[0, 0].item()),
                    "ly": float(pooled[0, 1].item()),
                    "lz": float(pooled[0, 2].item()) if pooled.shape[-1] > 2 else float(score_bundle["score_adjusted"]),
                    "loss_total": float(loss_vec[0].item()),
                    "loss_score_term": float(score_term[0].item()),
                    "loss_stability_term": float(stability_term[0].item()),
                    "loss_binding_term": float(binding_term[0].item()),
                    "loss_hydrophobic_term": float(hydro_term[0].item()),
                    "loss_ood_penalty": float(ood_penalty[0].item()),
                    "loss_mutation_penalty": float(mutation_penalty[0].item()),
                    "loss_identity_penalty": float(identity_penalty[0].item()),
                    "loss_stability_penalty": float(stability_penalty[0].item()),
                    "loss_binding_penalty": float(binding_penalty[0].item()),
                    "loss_lock_penalty": float(lock_penalty[0].item()),
                    "loss_l1_penalty": float(l1_penalty[0].item()),
                    "loss_pll_term": float(pll_term[0].item()),
                    "loss_contact_penalty": float(contact_penalty[0].item()),
                    "loss_epistasis_penalty": float(epistasis_penalty[0].item()),
                    "loss_dms_penalty": float(dms_penalty[0].item()),
                    "loss_cancer_pll_term": float(cancer_pll_term[0].item()),
                    "loss_local_score_term": float(local_score_term[0].item()),
                    "loss_cond_rescue_term": float(cond_rescue_term[0].item()),
                    "loss_structure_term": float(structure_term[0].item()),
                }
                trajectory.append(state)

                is_valid = (
                    seq_identity >= min_identity
                    and float(stability_t[0].item()) >= min_stability
                    and float(dna_force_t[0].item()) >= min_binding
                )

                improved = False
                if is_valid and state["score"] > (best_valid_score + 1e-3):
                    best_valid_state = dict(state)
                    best_valid_score = float(state["score"])
                    improved = True

                if improved:
                    checks_without_improve = 0
                else:
                    checks_without_improve += 1
                if step_idx >= 40 and checks_without_improve >= 6:
                    break

        final_state = best_valid_state if best_valid_state is not None else trajectory[-1]
        final_muts = json.loads(final_state.get("mutations_json", "[]"))
        final_positions = json.loads(final_state.get("mut_positions_json", "[]"))

        # Compute rescue DMS quality: mean raw Z-score of non-target mutations
        target_set = set(str(t).strip().upper() for t in scenario.targets)
        rescue_muts = [m for m in final_muts if m not in target_set]
        rescue_dms_scores = []
        n_functional_rescues = 0
        for rm in rescue_muts:
            parsed = parse_single_mutation(rm)
            if parsed is not None:
                _, pos, var_aa = parsed
                z = self._dms_lookup.get((pos, var_aa))
                if z is not None:
                    rescue_dms_scores.append(z)
                    if z < 0:
                        n_functional_rescues += 1
        rescue_dms_mean = float(np.mean(rescue_dms_scores)) if rescue_dms_scores else 0.0

        candidate_uid = (
            f"{scenario.scenario_id}|{pass_name}|rep{repeat_idx}|rst{restart_idx}|trial{trial_idx}"
        )
        candidate = {
            "candidate_uid": candidate_uid,
            "scenario_id": scenario.scenario_id,
            "target_label": scenario.target_label,
            "targets_json": json.dumps(scenario.targets),
            "delivery_method": scenario.delivery_method,
            "pass_name": pass_name,
            "repeat_idx": int(repeat_idx),
            "restart_idx": int(restart_idx),
            "trial_idx": int(trial_idx),
            "profile": profile["name"],
            "sequence": final_state["sequence"],
            "score": float(final_state["score"]),
            "score_raw": float(final_state.get("score_raw", final_state["score"])),
            "score_calibrated": float(final_state.get("score_calibrated", final_state["score"])),
            "score_gain_vs_target": float(final_state["score"] - baseline_score),
            "stability": float(final_state["stability"]),
            "binding": float(final_state["binding"]),
            "identity": float(final_state["identity"]),
            "n_mutations": int(final_state["n_mutations"]),
            "mutations_json": json.dumps(final_muts),
            "mut_positions_json": json.dumps(final_positions),
            "uncertainty": float(final_state.get("uncertainty", 0.0)),
            "ood_distance": float(final_state.get("ood_distance", 0.0)),
            "rescue_dms_mean": float(rescue_dms_mean),
            "n_functional_rescues": int(n_functional_rescues),
            "n_rescue_mutations": int(len(rescue_muts)),
            "meets_constraints": bool(
                float(final_state["identity"]) >= min_identity
                and float(final_state["stability"]) >= min_stability
                and float(final_state["binding"]) >= min_binding
            ),
            "selection_reason": "trust_adjusted_rank",
            "receptor_source": "n/a",
            "docking_backend": "n/a",
            "md_status": "n/a",
        }

        traj_rows = []
        for row in trajectory:
            payload = dict(row)
            payload["candidate_uid"] = candidate_uid
            payload["scenario_id"] = scenario.scenario_id
            payload["target_label"] = scenario.target_label
            payload["delivery_method"] = scenario.delivery_method
            payload["pass_name"] = pass_name
            payload["profile"] = profile["name"]
            traj_rows.append(payload)

        return candidate, traj_rows

    def _run_autoregressive_trial(
        self,
        *,
        scenario: ScenarioRuntime,
        target_seq: str,
        locked_indices: List[int],
        max_mutations: int,
        pass_name: str,
        trial_idx: int,
        trial_seed: int,
        baseline_score: float,
        n_candidates: int = 50,
        top_k: int = 3,
        temperature: float = 0.8,
    ) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
        """Gibbs-like autoregressive sampling: propose mutations one position at a time.

        1. Start from the cancer sequence.
        2. Rank positions by ESM-2 attention (which positions the model focuses on).
        3. For top-N candidate positions (not locked):
           a. Mask the position in the current sequence.
           b. Get ESM-2's distribution over 20 AAs.
           c. Try top-K candidates; accept if oracle score improves.
        4. Iterate passes until convergence or max_mutations reached.
        """
        assert self.embedder is not None
        assert self.oracle is not None

        torch.manual_seed(trial_seed)
        np.random.seed(trial_seed)

        locked_set = set(locked_indices)
        current_seq = list(target_seq)
        best_seq = list(target_seq)
        best_score = baseline_score

        trajectory: List[Dict[str, Any]] = []
        mutations_applied: List[str] = []

        # Get attention weights to rank positions by importance
        emb = self.embedder.get_embeddings("".join(current_seq)).detach()
        with torch.no_grad():
            _, _, _, attns = self.embedder.latent_forward_ascent(emb, return_attention=True)
            # Average attention across layers and heads → (L, L)
            attn_avg = torch.stack([a.mean(dim=1) for a in attns]).mean(dim=0).squeeze(0)  # (L, L)
            # Per-position attention magnitude = sum of attention received from all other positions
            pos_importance = attn_avg.sum(dim=0)  # (L,)

        # Rank positions: highest attention, not locked, within DBD
        dbd_range = set(range(93, 292))
        candidate_positions = [
            i for i in range(len(P53_WT))
            if i not in locked_set and i in dbd_range
        ]
        candidate_positions.sort(key=lambda i: float(pos_importance[i].item()), reverse=True)
        candidate_positions = candidate_positions[:n_candidates]

        n_mutations_applied = 0
        for pass_num in range(3):  # Up to 3 passes over candidate positions
            improved_this_pass = False
            for pos_idx in candidate_positions:
                if n_mutations_applied >= max_mutations:
                    break

                # Mask this position and get ESM-2's distribution
                seq_str = "".join(current_seq)
                inputs = self.embedder.tokenizer(seq_str, return_tensors="pt", add_special_tokens=False).to(self.embedder.device)
                masked_ids = inputs.input_ids.clone()
                masked_ids[0, pos_idx] = self.embedder.tokenizer.mask_token_id

                with torch.no_grad():
                    outputs = self.embedder.model(input_ids=masked_ids, return_dict=True)
                    logits = outputs.logits[0, pos_idx]
                    probs = torch.softmax(logits / temperature, dim=-1)

                # Get top-K amino acid candidates at this position
                top_probs, top_ids = probs.topk(top_k)
                original_aa = current_seq[pos_idx]

                for k_idx in range(top_k):
                    candidate_aa_id = top_ids[k_idx].item()
                    candidate_tokens = self.embedder.tokenizer.convert_ids_to_tokens([candidate_aa_id])
                    if not candidate_tokens:
                        continue
                    candidate_aa = candidate_tokens[0]
                    if candidate_aa == original_aa or len(candidate_aa) != 1 or candidate_aa not in "ACDEFGHIKLMNPQRSTVWY":
                        continue

                    # Try this substitution
                    test_seq = list(current_seq)
                    test_seq[pos_idx] = candidate_aa
                    test_str = "".join(test_seq)

                    # Score with oracle
                    test_emb = self.embedder.get_embeddings(test_str).detach()
                    with torch.no_grad():
                        test_z, _, _ = self.embedder.latent_forward_ascent(test_emb)
                        if isinstance(self.oracle.model, AttentionPoolingNet):
                            test_oracle_input = test_z
                            if test_oracle_input.shape[-1] != self.oracle.input_dim:
                                test_oracle_input = test_oracle_input[:, :, :self.oracle.input_dim]
                            test_score = float(self.oracle.model(test_oracle_input).squeeze(-1).item())
                        else:
                            test_pooled = test_z.mean(dim=1)
                            if test_pooled.shape[-1] != self.oracle.input_dim:
                                test_pooled = test_pooled[:, :self.oracle.input_dim]
                            test_score = float(self.oracle.model(test_pooled).squeeze(-1).item())

                    if test_score > best_score:
                        current_seq = test_seq
                        best_score = test_score
                        best_seq = list(test_seq)
                        mut_label = f"{P53_WT[pos_idx]}{pos_idx+1}{candidate_aa}"
                        mutations_applied.append(mut_label)
                        n_mutations_applied += 1
                        improved_this_pass = True

                        trajectory.append({
                            "step": n_mutations_applied,
                            "score": best_score,
                            "mutation_applied": mut_label,
                            "n_mutations": n_mutations_applied,
                            "sequence": "".join(best_seq),
                        })
                        break  # Accept first improvement, move to next position

            if not improved_this_pass:
                break  # Converged

        # Build candidate dict
        final_seq = "".join(best_seq)
        all_muts = [f"{P53_WT[j]}{j+1}{best_seq[j]}" for j in range(len(P53_WT)) if P53_WT[j] != best_seq[j]]
        mut_positions = [j + 1 for j in range(len(P53_WT)) if P53_WT[j] != best_seq[j]]
        seq_identity = 100.0 * (1.0 - len(all_muts) / float(len(P53_WT)))

        # DMS quality for rescue mutations
        target_set = set(str(t).strip().upper() for t in scenario.targets)
        rescue_muts = [m for m in all_muts if m not in target_set]
        rescue_dms_scores = []
        n_functional_rescues = 0
        for rm in rescue_muts:
            parsed = parse_single_mutation(rm)
            if parsed is not None:
                _, pos, var_aa = parsed
                z_val = self._dms_lookup.get((pos, var_aa))
                if z_val is not None:
                    rescue_dms_scores.append(z_val)
                    if z_val < 0:
                        n_functional_rescues += 1
        rescue_dms_mean = float(np.mean(rescue_dms_scores)) if rescue_dms_scores else 0.0

        candidate_uid = f"{scenario.scenario_id}|{pass_name}|autoregressive|trial{trial_idx}"
        candidate = {
            "candidate_uid": candidate_uid,
            "scenario_id": scenario.scenario_id,
            "target_label": scenario.target_label,
            "targets_json": json.dumps(scenario.targets),
            "delivery_method": scenario.delivery_method,
            "pass_name": pass_name,
            "repeat_idx": 1,
            "restart_idx": 1,
            "trial_idx": int(trial_idx),
            "profile": "Autoregressive",
            "sequence": final_seq,
            "score": float(best_score),
            "score_raw": float(best_score),
            "score_calibrated": float(self._calibrate_score(best_score)),
            "score_gain_vs_target": float(best_score - baseline_score),
            "stability": 0.0,
            "binding": 0.0,
            "identity": float(seq_identity),
            "n_mutations": int(len(all_muts)),
            "mutations_json": json.dumps(all_muts),
            "mut_positions_json": json.dumps(mut_positions),
            "uncertainty": 0.0,
            "ood_distance": 0.0,
            "rescue_dms_mean": float(rescue_dms_mean),
            "n_functional_rescues": int(n_functional_rescues),
            "n_rescue_mutations": int(len(rescue_muts)),
            "meets_constraints": bool(seq_identity >= 90.0),
            "selection_reason": "autoregressive_sampling",
            "receptor_source": "n/a",
            "docking_backend": "n/a",
            "md_status": "n/a",
        }

        traj_rows = []
        for row in trajectory:
            payload = dict(row)
            payload["candidate_uid"] = candidate_uid
            payload["scenario_id"] = scenario.scenario_id
            payload["target_label"] = scenario.target_label
            payload["delivery_method"] = scenario.delivery_method
            payload["pass_name"] = pass_name
            payload["profile"] = "Autoregressive"
            traj_rows.append(payload)

        return candidate, traj_rows

    def _run_clinical_for_scenario(self, candidates: List[Dict[str, Any]], top_k: int = 3) -> List[Dict[str, Any]]:
        assert self.clinical is not None

        if not candidates:
            return []
        df = pd.DataFrame(candidates)
        df = df.sort_values("score", ascending=False).head(int(max(top_k, 1)))

        rows: List[Dict[str, Any]] = []
        for _, row in df.iterrows():
            candidate_uid = str(row["candidate_uid"])
            targets = json.loads(str(row.get("targets_json", "[]")))
            cancer_mut = str(targets[0]) if targets else "R175H"
            rescue_muts = json.loads(str(row.get("mutations_json", "[]")))
            rescue_seq = str(row.get("sequence", P53_WT))
            report = self.clinical.generate_report(
                name=f"{candidate_uid.replace('|', '_')}",
                wt_sequence=P53_WT,
                rescue_sequence=rescue_seq,
                cancer_mutation=cancer_mut,
                rescue_mutations=rescue_muts,
            )
            rows.append(
                {
                    "candidate_uid": candidate_uid,
                    "scenario_id": str(row.get("scenario_id", "")),
                    "clinical_score": float(report.overall_clinical_score),
                    "clinical_viability": str(report.clinical_viability),
                    "us_annual_patients": int(report.patient_population.total_patients_per_year),
                    "global_estimate": int(report.patient_population.global_estimate),
                    "therapeutic_window": str(report.therapeutic_index.therapeutic_window),
                    "delivery_recommendation": str(report.delivery_options[0].method if report.delivery_options else "n/a"),
                }
            )
        return rows

    def _build_target_sequence(self, targets: Sequence[str]) -> str:
        seq = P53_WT
        for mut in targets:
            token = str(mut).strip().upper()
            if parse_single_mutation(token) is None:
                continue
            updated = apply_mutation(seq, token)
            if updated is not None:
                seq = updated
        return seq

    def _wt_aa_tensor(self, device: torch.device) -> torch.Tensor:
        assert self.embedder is not None
        wt_aa_indices = []
        for aa in P53_WT:
            aa_id = self.embedder.tokenizer.convert_tokens_to_ids(aa)
            if aa_id in AA_IDS:
                wt_aa_indices.append(AA_IDS.index(aa_id))
            else:
                wt_aa_indices.append(0)
        return torch.tensor(wt_aa_indices, device=device)

    def _delivery_identity_floor(self, delivery_method: str) -> float:
        mode = str(delivery_method).strip().lower()
        if mode == "protein_therapy":
            return 92.0
        if mode == "mrna_therapy":
            return 92.0
        return 90.0

    def _batch_dna_force(self, z_batch: torch.Tensor, probs_full: torch.Tensor) -> torch.Tensor:
        hotspots = [119, 174, 240, 247, 272, 279]
        hotspots = [i for i in hotspots if i < z_batch.shape[1]]
        if not hotspots:
            return torch.zeros(z_batch.shape[0], device=z_batch.device)
        pos_charge_ids = [10, 15, 21]
        latent_force = z_batch[:, hotspots, :].norm(dim=-1).mean(dim=-1)
        charge_prob = probs_full[:, hotspots][:, :, pos_charge_ids].sum(dim=-1).mean(dim=-1)
        return 0.5 * latent_force + 5.0 * charge_prob

    def _batch_hydrophobic_packing(self, probs_full: torch.Tensor) -> torch.Tensor:
        hydro_ids = [4, 12, 7, 18, 22, 20]
        core_res = [i for i in range(93, 312) if i < probs_full.shape[1]]
        loops = set(range(111, 124)) | set(range(162, 195)) | set(range(235, 251))
        true_core = [i for i in core_res if i not in loops]
        if not true_core:
            return torch.zeros(probs_full.shape[0], device=probs_full.device)
        return probs_full[:, true_core][:, :, hydro_ids].sum(dim=-1).mean(dim=-1)

    def _decode_sequence(self, logits_aa: torch.Tensor) -> str:
        assert self.embedder is not None
        top_ids_aa = torch.argmax(logits_aa, dim=-1)[0]
        aa_local = [AA_IDS[idx] for idx in top_ids_aa.tolist()]
        tokens = self.embedder.tokenizer.convert_ids_to_tokens(aa_local)
        return "".join(tokens)[: len(P53_WT)]

    def _mutation_pos(self, mut: str) -> Optional[int]:
        digits = "".join(ch for ch in str(mut) if ch.isdigit())
        if not digits:
            return None
        try:
            return int(digits)
        except ValueError:
            return None

    def _compute_baseline_metrics(self, target_seq: str, pooled_target_ref: torch.Tensor, mc_samples: int) -> Dict[str, float]:
        assert self.embedder is not None
        assert self.oracle is not None

        emb = self.embedder.get_embeddings(target_seq).detach()
        with torch.no_grad():
            z, logits, _ = self.embedder.latent_forward_ascent(emb)
            pooled = z.mean(dim=1)
            if pooled.shape[-1] != self.oracle.input_dim:
                pooled = pooled[:, :self.oracle.input_dim]
            _uses_attention = isinstance(self.oracle.model, AttentionPoolingNet)
            if _uses_attention:
                z_oracle = z
                if z_oracle.shape[-1] != self.oracle.input_dim:
                    z_oracle = z_oracle[:, :, :self.oracle.input_dim]
                raw_score = float(self.oracle.model(z_oracle).item())
            else:
                raw_score = float(self.oracle.model(pooled).item())
            logits_aa = logits[:, :, AA_IDS]
            stability = float(F.log_softmax(logits_aa, dim=-1).max(dim=-1).values.mean().item())
            probs_full = torch.softmax(logits, dim=-1)
            binding = float(self._batch_dna_force(z, probs_full)[0].item())

        bundle = self._build_trust_adjusted_score(
            raw_score=raw_score,
            pooled=pooled,
            pooled_ref=pooled_target_ref,
            cached_uncertainty=self._estimate_uncertainty(pooled, mc_samples=mc_samples, z_full=z),
        )
        return {
            "score": float(bundle["score_adjusted"]),
            "score_raw": float(bundle["score_raw"]),
            "score_calibrated": float(bundle["score_calibrated"]),
            "stability": stability,
            "binding": binding,
            "uncertainty": float(bundle["uncertainty"]),
            "ood_distance": float(bundle["ood_distance"]),
        }

    def _score_calibration_profile(self) -> Dict[str, float]:
        fallback = {"clip_low": -3.0, "clip_high": 4.0, "center": 0.0, "scale": 1.0}
        try:
            dms_df = get_dms_data()
        except Exception:
            return fallback
        if dms_df is None or dms_df.empty or "score" not in dms_df.columns:
            return fallback

        dms_work = dms_df.copy()
        if "n_mutations" in dms_work.columns:
            dms_work = dms_work[dms_work["n_mutations"] == 1].copy()
        scores = pd.to_numeric(dms_work["score"], errors="coerce").dropna().to_numpy(dtype=float)
        if scores.size < 20:
            return fallback

        q1, q5, q95, q99 = np.percentile(scores, [1, 5, 95, 99])
        center = float(np.median(scores))
        scale = float(max((q95 - q5) / 2.0, 1e-3))
        clip_low = float(min(q1, q5))
        clip_high = float(max(q99, q95))
        if clip_high <= clip_low:
            clip_low, clip_high = -3.0, 4.0

        return {"clip_low": clip_low, "clip_high": clip_high, "center": center, "scale": scale}

    def _calibrate_score(self, raw_score: float) -> float:
        center = float(self.calibration_profile.get("center", 0.0))
        scale = float(max(self.calibration_profile.get("scale", 1.0), 1e-6))
        clip_low = float(self.calibration_profile.get("clip_low", -3.0))
        clip_high = float(self.calibration_profile.get("clip_high", 4.0))
        squashed = center + scale * np.tanh((float(raw_score) - center) / scale)
        return float(np.clip(squashed, clip_low, clip_high))

    def _estimate_uncertainty(self, pooled: torch.Tensor, mc_samples: int = 8, z_full: Optional[torch.Tensor] = None) -> float:
        assert self.oracle is not None
        n_samples = max(int(mc_samples), 2)
        was_training = self.oracle.model.training
        _uses_attention = isinstance(self.oracle.model, AttentionPoolingNet)
        oracle_input = z_full if (_uses_attention and z_full is not None) else pooled
        preds = []
        try:
            self.oracle.model.train()
            with torch.no_grad():
                for _ in range(n_samples):
                    preds.append(float(self.oracle.model(oracle_input).squeeze(-1).mean().item()))
        except Exception:
            return 0.0
        finally:
            if not was_training:
                self.oracle.model.eval()
        return float(np.std(np.asarray(preds, dtype=float)))

    def _build_trust_adjusted_score(
        self,
        *,
        raw_score: float,
        pooled: Optional[torch.Tensor],
        pooled_ref: Optional[torch.Tensor],
        cached_uncertainty: float,
    ) -> Dict[str, float]:
        calibrated = self._calibrate_score(raw_score)
        ood_distance = 0.0
        if pooled is not None and pooled_ref is not None:
            with torch.no_grad():
                ood_distance = float(torch.norm(pooled - pooled_ref, p=2, dim=-1).mean().item())

        adjusted = calibrated - self.uncertainty_weight * float(cached_uncertainty) - self.ood_rank_weight * max(
            0.0, ood_distance - self.ood_radius
        )
        adjusted = float(
            np.clip(
                adjusted,
                float(self.calibration_profile.get("clip_low", -3.0)),
                float(self.calibration_profile.get("clip_high", 4.0)),
            )
        )
        return {
            "score_raw": float(raw_score),
            "score_calibrated": float(calibrated),
            "score_adjusted": float(adjusted),
            "uncertainty": float(cached_uncertainty),
            "ood_distance": float(ood_distance),
        }

    def _append_rows(self, path: Path, rows: List[Dict[str, Any]]) -> None:
        if not rows:
            return
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("a", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row) + "\n")

    def _read_jsonl(self, path: Path) -> pd.DataFrame:
        if not path.exists():
            return pd.DataFrame()
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    logger.warning("Skipping malformed jsonl row from %s", path)
        return pd.DataFrame(rows)

    def _build_summary(
        self,
        run_id: str,
        scenario_df: pd.DataFrame,
        candidate_df: pd.DataFrame,
        top30_df: pd.DataFrame,
    ) -> str:
        lines: List[str] = []
        lines.append(f"# Campaign Summary: {run_id}")
        lines.append("")
        lines.append(f"Generated at: {datetime.now(timezone.utc).isoformat()}")
        lines.append("")
        lines.append("## Run Totals")
        unique_scenarios = 0
        pass_breakdown: Dict[str, int] = {}
        if scenario_df is not None and not scenario_df.empty:
            if "scenario_id" in scenario_df.columns:
                unique_scenarios = int(scenario_df["scenario_id"].astype(str).nunique())
            else:
                unique_scenarios = int(len(scenario_df))
            if "pass_name" in scenario_df.columns:
                pass_breakdown = (
                    scenario_df["pass_name"]
                    .astype(str)
                    .value_counts()
                    .sort_index()
                    .to_dict()
                )
        lines.append(f"- Scenarios (unique): {unique_scenarios}")
        if pass_breakdown:
            lines.append(f"- Scenario pass rows: {pass_breakdown}")
        lines.append(f"- Candidates: {len(candidate_df)}")
        lines.append(f"- Presentation shortlist: {len(top30_df)}")
        lines.append("")

        if not candidate_df.empty:
            best = candidate_df.sort_values("score", ascending=False).head(1).iloc[0]
            lines.append("## Best Candidate")
            lines.append(f"- Candidate UID: {best.get('candidate_uid', 'n/a')}")
            lines.append(f"- Target: {best.get('target_label', 'n/a')}")
            lines.append(f"- Delivery: {best.get('delivery_method', 'n/a')}")
            lines.append(f"- Score: {float(best.get('score', 0.0)):.3f}")
            lines.append(f"- Identity: {float(best.get('identity', 0.0)):.1f}%")
            lines.append(f"- Mutations: {int(best.get('n_mutations', 0))}")
            lines.append("")

        if not top30_df.empty:
            lines.append("## Top 10 from Shortlist")
            top10 = top30_df.head(10)
            for _, row in top10.iterrows():
                lines.append(
                    f"- #{int(row.get('presentation_rank', 0))}: "
                    f"{row.get('target_label', 'n/a')} | {row.get('delivery_method', 'n/a')} | "
                    f"score={float(row.get('score', 0.0)):.3f} | "
                    f"clinical={float(row.get('clinical_score', np.nan)) if pd.notna(row.get('clinical_score', np.nan)) else 'n/a'}"
                )

        return "\n".join(lines) + "\n"



def scenario_matrix_frame(
    *,
    include_pairs: bool = True,
    hotspots: Sequence[str] | None = None,
    delivery_methods: Sequence[str] | None = None,
) -> pd.DataFrame:
    scenarios = build_scenario_matrix(
        hotspots=hotspots or BIG8_HOTSPOTS,
        delivery_methods=delivery_methods or DEFAULT_DELIVERY_METHODS,
        include_pairs=include_pairs,
    )
    return scenarios_to_frame(scenarios)


In [ ]:
%%writefile p53cad/analysis/clinical_impact.py
"""
Clinical Impact Quantification Module for p53-proteoMgCAD

Quantifies the clinical relevance of rescue mutations:
1. Patient Stratification - How many patients would benefit (TCGA data)
2. Therapeutic Index - Safety margins based on sequence identity
3. Delivery Feasibility - Scoring for different delivery methods
4. Immunogenicity Risk - Assessment of immune response potential
"""

import numpy as np
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import defaultdict
import json


# ============================================================================
# TCGA MUTATION FREQUENCY DATA
# ============================================================================

# From TCGA pan-cancer analysis (approximate frequencies)
# Based on cBioPortal and COSMIC database
TCGA_P53_MUTATIONS = {
    # Hotspot mutations (% of all p53 mutations)
    "R175H": {"frequency": 5.6, "cancer_types": ["breast", "colon", "ovarian", "lung", "pancreatic"]},
    "R248Q": {"frequency": 4.4, "cancer_types": ["colon", "breast", "ovarian", "lung", "gastric"]},
    "R273H": {"frequency": 4.2, "cancer_types": ["colon", "breast", "lung", "head_neck", "bladder"]},
    "R248W": {"frequency": 3.8, "cancer_types": ["colon", "breast", "ovarian", "lung"]},
    "R273C": {"frequency": 2.1, "cancer_types": ["colon", "breast", "lung"]},
    "Y220C": {"frequency": 1.8, "cancer_types": ["breast", "lung", "colon", "ovarian"]},
    "G245S": {"frequency": 2.4, "cancer_types": ["breast", "colon", "lung", "ovarian"]},
    "R249S": {"frequency": 2.9, "cancer_types": ["liver", "lung", "esophageal"]},  # Aflatoxin-associated
    "R282W": {"frequency": 2.0, "cancer_types": ["breast", "colon", "ovarian", "lung"]},
    "G245D": {"frequency": 1.5, "cancer_types": ["colon", "breast", "lung"]},
    "R175G": {"frequency": 0.8, "cancer_types": ["breast", "colon"]},
    "C176F": {"frequency": 0.7, "cancer_types": ["breast", "ovarian"]},
    "H179R": {"frequency": 0.9, "cancer_types": ["breast", "colon", "lung"]},
    "H179Y": {"frequency": 0.6, "cancer_types": ["breast", "colon"]},
    "C242F": {"frequency": 0.8, "cancer_types": ["lung", "colon"]},
    "C242S": {"frequency": 0.5, "cancer_types": ["lung", "colon"]},
    "G245C": {"frequency": 0.7, "cancer_types": ["breast", "colon"]},
    "R280K": {"frequency": 0.6, "cancer_types": ["breast", "colon", "lung"]},
    "D281G": {"frequency": 0.4, "cancer_types": ["breast", "colon"]},
    "E285K": {"frequency": 0.5, "cancer_types": ["colon", "breast"]},

    # Contact mutations
    "R273L": {"frequency": 0.9, "cancer_types": ["colon", "breast", "lung"]},
    "S241F": {"frequency": 0.6, "cancer_types": ["breast", "colon"]},
    "R248L": {"frequency": 0.5, "cancer_types": ["colon", "breast"]},

    # Structural mutations
    "V143A": {"frequency": 0.4, "cancer_types": ["breast", "colon"]},
    "V157F": {"frequency": 0.5, "cancer_types": ["lung", "colon"]},
    "Y163C": {"frequency": 0.4, "cancer_types": ["breast", "lung"]},
    "V173L": {"frequency": 0.3, "cancer_types": ["colon", "breast"]},
    "I195T": {"frequency": 0.3, "cancer_types": ["breast", "colon"]},
    "I255F": {"frequency": 0.3, "cancer_types": ["lung", "colon"]},
}

# Cancer incidence data (new cases per year, US)
CANCER_INCIDENCE = {
    "breast": 290000,
    "lung": 235000,
    "colon": 150000,
    "ovarian": 20000,
    "pancreatic": 62000,
    "gastric": 27000,
    "liver": 42000,
    "bladder": 83000,
    "head_neck": 66000,
    "esophageal": 21000,
}

# p53 mutation rate by cancer type (% of cases with p53 mutation)
P53_MUTATION_RATE = {
    "ovarian": 96,  # High-grade serous
    "esophageal": 83,
    "lung": 75,  # NSCLC
    "colon": 60,
    "pancreatic": 70,
    "head_neck": 85,
    "gastric": 50,
    "bladder": 50,
    "breast": 30,  # Higher in triple-negative
    "liver": 30,
}


# ============================================================================
# DATA CLASSES
# ============================================================================

@dataclass
class PatientPopulation:
    """Estimated patient population for a specific mutation."""
    mutation: str
    mutation_frequency: float  # % of p53 mutations
    total_patients_per_year: int  # US estimate
    by_cancer_type: Dict[str, int]  # patients per cancer type
    global_estimate: int  # worldwide estimate (3x US)
    five_year_cumulative: int


@dataclass
class TherapeuticIndex:
    """Therapeutic index assessment."""
    sequence_identity: float  # % identity to WT
    mutation_count: int
    immunogenicity_score: float  # 0-1 (lower is better)
    off_target_risk: float  # 0-1
    therapeutic_window: str  # "narrow", "moderate", "wide"
    fda_pathway: str  # "gene_therapy", "cell_therapy", "protein"
    regulatory_complexity: str  # "low", "medium", "high"


@dataclass
class DeliveryFeasibility:
    """Delivery method feasibility assessment."""
    method: str
    feasibility_score: float  # 0-1
    advantages: List[str]
    challenges: List[str]
    estimated_cost_per_dose: str
    regulatory_status: str
    clinical_examples: List[str]
    identity_requirement: float  # minimum % identity for this method


@dataclass
class ImmunogenicityAssessment:
    """Immunogenicity risk assessment."""
    overall_risk: str  # "low", "moderate", "high"
    risk_score: float  # 0-1
    neoantigens_predicted: int
    hla_binding_peptides: int
    risk_factors: List[str]
    mitigation_strategies: List[str]


@dataclass
class ClinicalImpactReport:
    """Complete clinical impact report."""
    rescue_name: str
    cancer_mutation: str
    rescue_mutations: List[str]

    patient_population: PatientPopulation
    therapeutic_index: TherapeuticIndex
    delivery_options: List[DeliveryFeasibility]
    immunogenicity: ImmunogenicityAssessment

    overall_clinical_score: float  # 0-100
    clinical_viability: str  # "high", "moderate", "low", "not_viable"
    key_advantages: List[str]
    key_challenges: List[str]
    recommended_development_path: str


# ============================================================================
# PATIENT STRATIFICATION
# ============================================================================

class PatientStratifier:
    """Estimate patient populations for p53 mutations."""

    def __init__(self):
        self.mutation_data = TCGA_P53_MUTATIONS
        self.incidence = CANCER_INCIDENCE
        self.p53_rate = P53_MUTATION_RATE

    def estimate_population(self, mutation: str) -> PatientPopulation:
        """Estimate patient population for a specific mutation."""

        if mutation not in self.mutation_data:
            # Unknown mutation - estimate conservatively
            return self._estimate_unknown_mutation(mutation)

        mut_info = self.mutation_data[mutation]
        frequency = mut_info["frequency"]
        cancer_types = mut_info["cancer_types"]

        by_cancer = {}
        total = 0

        for cancer in cancer_types:
            if cancer in self.incidence and cancer in self.p53_rate:
                # Patients with this cancer AND p53 mutation AND this specific variant
                cases = self.incidence[cancer]
                p53_cases = cases * (self.p53_rate[cancer] / 100)
                mutation_cases = p53_cases * (frequency / 100)
                by_cancer[cancer] = int(mutation_cases)
                total += int(mutation_cases)

        return PatientPopulation(
            mutation=mutation,
            mutation_frequency=frequency,
            total_patients_per_year=total,
            by_cancer_type=by_cancer,
            global_estimate=total * 3,  # Rough global multiplier
            five_year_cumulative=total * 5
        )

    def _estimate_unknown_mutation(self, mutation: str) -> PatientPopulation:
        """Estimate for unknown mutations."""
        # Assume rare mutation (~0.1% of p53 mutations)
        frequency = 0.1

        # Estimate across common p53-mutated cancers
        total = 0
        by_cancer = {}

        for cancer in ["breast", "colon", "lung", "ovarian"]:
            cases = self.incidence.get(cancer, 0)
            p53_cases = cases * (self.p53_rate.get(cancer, 50) / 100)
            mutation_cases = p53_cases * (frequency / 100)
            by_cancer[cancer] = int(mutation_cases)
            total += int(mutation_cases)

        return PatientPopulation(
            mutation=mutation,
            mutation_frequency=frequency,
            total_patients_per_year=total,
            by_cancer_type=by_cancer,
            global_estimate=total * 3,
            five_year_cumulative=total * 5
        )

    def rank_mutations_by_impact(self, top_n: int = 20) -> List[Tuple[str, int]]:
        """Rank mutations by patient impact."""
        populations = []
        for mutation in self.mutation_data:
            pop = self.estimate_population(mutation)
            populations.append((mutation, pop.total_patients_per_year))

        populations.sort(key=lambda x: x[1], reverse=True)
        return populations[:top_n]

    def get_addressable_market(self, mutations: List[str]) -> Dict:
        """Calculate total addressable market for a set of mutations."""
        total_patients = 0
        by_cancer = defaultdict(int)

        for mutation in mutations:
            pop = self.estimate_population(mutation)
            total_patients += pop.total_patients_per_year
            for cancer, count in pop.by_cancer_type.items():
                by_cancer[cancer] += count

        return {
            "mutations_targeted": mutations,
            "total_patients_per_year": total_patients,
            "global_estimate": total_patients * 3,
            "by_cancer_type": dict(by_cancer),
            "five_year_cumulative": total_patients * 5,
            "market_potential": self._estimate_market_value(total_patients)
        }

    def _estimate_market_value(self, patients: int) -> str:
        """Rough market value estimate."""
        # Assume $100-500k per patient for gene therapy
        low = patients * 100000
        high = patients * 500000

        if high > 1e9:
            return f"${low/1e9:.1f}B - ${high/1e9:.1f}B"
        else:
            return f"${low/1e6:.0f}M - ${high/1e6:.0f}M"


# ============================================================================
# THERAPEUTIC INDEX CALCULATOR
# ============================================================================

class TherapeuticIndexCalculator:
    """Calculate therapeutic index metrics."""

    def __init__(self):
        self.fda_identity_thresholds = {
            "gene_therapy": 90,  # Lower threshold acceptable
            "mrna_therapy": 92,
            "protein_therapy": 95,  # Highest requirement
            "cell_therapy": 90,
        }

    def calculate(self, wt_sequence: str, rescue_sequence: str,
                 mutations: List[str]) -> TherapeuticIndex:
        """Calculate therapeutic index for a rescue design."""

        # Sequence identity
        identity = self._calculate_identity(wt_sequence, rescue_sequence)

        # Mutation count
        mutation_count = len(mutations)

        # Immunogenicity score
        immuno_score = self._estimate_immunogenicity(mutations, identity)

        # Off-target risk
        off_target = self._estimate_off_target_risk(mutations)

        # Determine therapeutic window
        if identity >= 98 and immuno_score < 0.2:
            window = "wide"
        elif identity >= 95 and immuno_score < 0.4:
            window = "moderate"
        else:
            window = "narrow"

        # Determine best FDA pathway
        fda_pathway = self._recommend_fda_pathway(identity, mutation_count)

        # Regulatory complexity
        if mutation_count <= 3 and identity >= 97:
            complexity = "low"
        elif mutation_count <= 6 and identity >= 94:
            complexity = "medium"
        else:
            complexity = "high"

        return TherapeuticIndex(
            sequence_identity=identity,
            mutation_count=mutation_count,
            immunogenicity_score=immuno_score,
            off_target_risk=off_target,
            therapeutic_window=window,
            fda_pathway=fda_pathway,
            regulatory_complexity=complexity
        )

    def _calculate_identity(self, seq1: str, seq2: str) -> float:
        """Calculate sequence identity percentage."""
        if len(seq1) != len(seq2):
            min_len = min(len(seq1), len(seq2))
            seq1 = seq1[:min_len]
            seq2 = seq2[:min_len]

        matches = sum(1 for a, b in zip(seq1, seq2) if a == b)
        return (matches / len(seq1)) * 100

    def _estimate_immunogenicity(self, mutations: List[str],
                                identity: float) -> float:
        """Estimate immunogenicity risk (0-1, lower is better)."""
        base_risk = 0.1

        # Add risk per mutation
        mutation_risk = len(mutations) * 0.05

        # Identity factor
        identity_risk = max(0, (100 - identity) * 0.02)

        # Check for known immunogenic positions
        immunogenic_positions = [17, 18, 19, 20, 21, 22, 23, 24, 25]  # MDM2 binding / exposed
        for mut in mutations:
            pos = int(mut[1:-1])
            if pos in immunogenic_positions:
                mutation_risk += 0.1

        total_risk = base_risk + mutation_risk + identity_risk
        return min(1.0, total_risk)

    def _estimate_off_target_risk(self, mutations: List[str]) -> float:
        """Estimate off-target effects risk."""
        # Based on how close mutations are to critical functional sites
        critical_positions = {
            # Zinc coordination
            176: 0.3, 179: 0.3, 238: 0.3, 242: 0.3,
            # DNA binding
            248: 0.2, 273: 0.2, 280: 0.2,
            # Tetramerization
            341: 0.2, 344: 0.2, 348: 0.2,
        }

        risk = 0.05  # Base risk
        for mut in mutations:
            pos = int(mut[1:-1])
            if pos in critical_positions:
                risk += critical_positions[pos]

        return min(1.0, risk)

    def _recommend_fda_pathway(self, identity: float,
                              mutation_count: int) -> str:
        """Recommend FDA regulatory pathway."""
        if identity >= 97 and mutation_count <= 3:
            return "gene_therapy"  # AAV delivery feasible
        elif identity >= 94:
            return "mrna_therapy"  # mRNA with lipid nanoparticles
        elif identity >= 90:
            return "cell_therapy"  # Ex vivo modification
        else:
            return "protein_therapy"  # Purified protein (hardest)


# ============================================================================
# DELIVERY FEASIBILITY ASSESSOR
# ============================================================================

class DeliveryFeasibilityAssessor:
    """Assess feasibility of different delivery methods."""

    def __init__(self):
        self.delivery_methods = self._define_delivery_methods()

    def _define_delivery_methods(self) -> List[Dict]:
        """Define available delivery methods."""
        return [
            {
                "method": "AAV Gene Therapy",
                "description": "Adeno-associated virus delivery of p53 gene",
                "identity_requirement": 90,
                "advantages": [
                    "Long-lasting expression",
                    "Established clinical precedent (Luxturna, Zolgensma)",
                    "Tissue-specific targeting possible",
                    "Single dose potential"
                ],
                "challenges": [
                    "Pre-existing immunity (~30% population)",
                    "Limited cargo capacity (4.7kb)",
                    "Manufacturing complexity",
                    "High cost per dose"
                ],
                "cost": "$100k-$500k per dose",
                "regulatory_status": "Multiple approved products",
                "examples": ["Luxturna", "Zolgensma", "Hemgenix"]
            },
            {
                "method": "Lentiviral Gene Therapy",
                "description": "Lentiviral vector for stable integration",
                "identity_requirement": 90,
                "advantages": [
                    "Larger cargo capacity",
                    "Stable integration",
                    "Works in dividing cells",
                    "Ex vivo modification possible"
                ],
                "challenges": [
                    "Insertional mutagenesis risk",
                    "Ex vivo manufacturing required",
                    "Cannot target all tissues"
                ],
                "cost": "$300k-$1M per treatment",
                "regulatory_status": "Approved for blood disorders",
                "examples": ["Kymriah", "Yescarta", "Breyanzi"]
            },
            {
                "method": "mRNA-LNP Therapy",
                "description": "mRNA encapsulated in lipid nanoparticles",
                "identity_requirement": 92,
                "advantages": [
                    "Transient expression (safety)",
                    "Rapid manufacturing",
                    "Lower cost than viral vectors",
                    "Repeat dosing possible"
                ],
                "challenges": [
                    "Short duration of expression",
                    "Requires repeated dosing",
                    "Liver tropism of LNPs",
                    "Cold chain requirements"
                ],
                "cost": "$5k-$50k per dose course",
                "regulatory_status": "COVID vaccines approved",
                "examples": ["Comirnaty", "Spikevax"]
            },
            {
                "method": "Purified Protein Therapy",
                "description": "Direct delivery of purified p53 protein",
                "identity_requirement": 95,
                "advantages": [
                    "Precise dosing",
                    "Immediate effect",
                    "Well-understood PK/PD"
                ],
                "challenges": [
                    "Cell penetration barrier",
                    "Short half-life",
                    "Immunogenicity concerns",
                    "Manufacturing complexity"
                ],
                "cost": "$10k-$100k per course",
                "regulatory_status": "Challenging for transcription factors",
                "examples": ["Insulin (proof of concept)"]
            },
            {
                "method": "CRISPR Prime Editing",
                "description": "Direct genome editing to introduce rescue mutations",
                "identity_requirement": 100,  # Creates WT-like sequence
                "advantages": [
                    "Permanent correction",
                    "Creates near-WT sequence",
                    "No foreign protein expression",
                    "Precise edits possible"
                ],
                "challenges": [
                    "Delivery to solid tumors difficult",
                    "Off-target editing risk",
                    "Low efficiency in vivo",
                    "Regulatory uncertainty"
                ],
                "cost": "Uncertain ($100k+)",
                "regulatory_status": "Early clinical trials",
                "examples": ["EDIT-101 (Editas)", "CTX001 (CRISPR Therapeutics)"]
            },
            {
                "method": "Oncolytic Virus + p53",
                "description": "Armed oncolytic virus expressing p53",
                "identity_requirement": 90,
                "advantages": [
                    "Tumor selectivity",
                    "Immune activation",
                    "Bystander effect",
                    "Clinical precedent (IMLYGIC)"
                ],
                "challenges": [
                    "Systemic delivery limited",
                    "Anti-vector immunity",
                    "Variable tumor penetration"
                ],
                "cost": "$50k-$200k per course",
                "regulatory_status": "IMLYGIC approved (melanoma)",
                "examples": ["IMLYGIC", "ONYX-015 (historical)"]
            }
        ]

    def assess_all_methods(self, sequence_identity: float,
                          mutation_count: int) -> List[DeliveryFeasibility]:
        """Assess feasibility of all delivery methods."""
        assessments = []

        for method_info in self.delivery_methods:
            score = self._calculate_feasibility(
                method_info, sequence_identity, mutation_count
            )

            assessment = DeliveryFeasibility(
                method=method_info["method"],
                feasibility_score=score,
                advantages=method_info["advantages"],
                challenges=method_info["challenges"],
                estimated_cost_per_dose=method_info["cost"],
                regulatory_status=method_info["regulatory_status"],
                clinical_examples=method_info["examples"],
                identity_requirement=method_info["identity_requirement"]
            )
            assessments.append(assessment)

        # Sort by feasibility score
        assessments.sort(key=lambda x: x.feasibility_score, reverse=True)
        return assessments

    def _calculate_feasibility(self, method_info: Dict,
                              identity: float,
                              mutation_count: int) -> float:
        """Calculate feasibility score for a delivery method."""
        score = 0.5  # Base score

        # Identity requirement check
        req_identity = method_info["identity_requirement"]
        if identity >= req_identity:
            score += 0.3
        elif identity >= req_identity - 3:
            score += 0.1
        else:
            score -= 0.3

        # Mutation count factor (fewer = better)
        if mutation_count <= 3:
            score += 0.1
        elif mutation_count <= 5:
            score += 0.05
        elif mutation_count > 10:
            score -= 0.1

        # Regulatory status bonus
        if "approved" in method_info["regulatory_status"].lower():
            score += 0.1

        return max(0, min(1, score))


# ============================================================================
# IMMUNOGENICITY ASSESSOR
# ============================================================================

class ImmunogenicityAssessor:
    """Assess immunogenicity risk of rescue designs."""

    def __init__(self):
        # Common HLA alleles for neoantigen prediction (simplified)
        self.common_hlas = ["A*02:01", "A*01:01", "B*07:02", "B*08:01"]

    def assess(self, wt_sequence: str, rescue_sequence: str,
              mutations: List[str]) -> ImmunogenicityAssessment:
        """Assess immunogenicity risk."""

        # Count predicted neoantigens
        neoantigens = self._predict_neoantigens(mutations, rescue_sequence)

        # Estimate HLA binding peptides
        hla_peptides = self._estimate_hla_binders(mutations, rescue_sequence)

        # Identify risk factors
        risk_factors = self._identify_risk_factors(mutations, rescue_sequence)

        # Calculate risk score
        risk_score = self._calculate_risk_score(
            len(mutations), neoantigens, hla_peptides, risk_factors
        )

        # Determine overall risk
        if risk_score < 0.3:
            overall_risk = "low"
        elif risk_score < 0.6:
            overall_risk = "moderate"
        else:
            overall_risk = "high"

        # Mitigation strategies
        mitigations = self._suggest_mitigations(overall_risk, risk_factors)

        return ImmunogenicityAssessment(
            overall_risk=overall_risk,
            risk_score=risk_score,
            neoantigens_predicted=neoantigens,
            hla_binding_peptides=hla_peptides,
            risk_factors=risk_factors,
            mitigation_strategies=mitigations
        )

    def _predict_neoantigens(self, mutations: List[str],
                            sequence: str) -> int:
        """Predict number of potential neoantigens (simplified)."""
        # Each mutation creates potential neoantigen
        # Surface-exposed mutations have higher risk
        exposed_positions = set(range(1, 50)) | set(range(290, 393))  # N/C termini

        count = 0
        for mut in mutations:
            pos = int(mut[1:-1])
            if pos in exposed_positions:
                count += 2  # Higher risk
            else:
                count += 1

        return count

    def _estimate_hla_binders(self, mutations: List[str],
                             sequence: str) -> int:
        """Estimate HLA-binding peptides from mutations."""
        # Simplified: each mutation can generate ~4 potential 9-mer peptides
        # Real implementation would use NetMHC or similar
        return len(mutations) * 4

    def _identify_risk_factors(self, mutations: List[str],
                              sequence: str) -> List[str]:
        """Identify specific immunogenicity risk factors."""
        risks = []

        # High mutation count
        if len(mutations) > 5:
            risks.append(f"High mutation count ({len(mutations)} mutations)")

        # Check for immunogenic amino acid changes
        immunogenic_changes = {'A': 'K', 'A': 'R', 'A': 'E', 'A': 'D'}  # Non-self-like
        for mut in mutations:
            new_aa = mut[-1]
            if new_aa in 'KRDE':  # Charged residues often immunogenic
                risks.append(f"Charged residue introduction at {mut}")

        # Surface-exposed mutations
        exposed = set(range(1, 50)) | set(range(290, 393))
        for mut in mutations:
            pos = int(mut[1:-1])
            if pos in exposed:
                risks.append(f"Surface-exposed mutation {mut}")

        return risks[:5]  # Top 5 risks

    def _calculate_risk_score(self, mutation_count: int,
                             neoantigens: int, hla_peptides: int,
                             risk_factors: List[str]) -> float:
        """Calculate overall immunogenicity risk score."""
        score = 0.1  # Base risk

        # Mutation count contribution
        score += mutation_count * 0.05

        # Neoantigen contribution
        score += neoantigens * 0.03

        # Risk factor contribution
        score += len(risk_factors) * 0.05

        return min(1.0, score)

    def _suggest_mitigations(self, risk_level: str,
                            risk_factors: List[str]) -> List[str]:
        """Suggest immunogenicity mitigation strategies."""
        mitigations = []

        mitigations.append("Use immunosuppression during initial dosing")
        mitigations.append("Consider tolerization protocol before treatment")

        if risk_level in ["moderate", "high"]:
            mitigations.append("Prioritize mRNA delivery for transient expression")
            mitigations.append("Monitor for anti-drug antibodies")
            mitigations.append("Consider dose titration strategy")

        if "Surface-exposed" in str(risk_factors):
            mitigations.append("Evaluate alternative mutations at buried positions")

        return mitigations


# ============================================================================
# CLINICAL IMPACT ENGINE
# ============================================================================

class ClinicalImpactEngine:
    """
    Main engine for clinical impact assessment.
    """

    def __init__(self):
        self.patient_stratifier = PatientStratifier()
        self.ti_calculator = TherapeuticIndexCalculator()
        self.delivery_assessor = DeliveryFeasibilityAssessor()
        self.immuno_assessor = ImmunogenicityAssessor()

    def generate_report(self, name: str, wt_sequence: str,
                       rescue_sequence: str, cancer_mutation: str,
                       rescue_mutations: List[str]) -> ClinicalImpactReport:
        """Generate comprehensive clinical impact report."""

        # Patient population
        patient_pop = self.patient_stratifier.estimate_population(cancer_mutation)

        # Therapeutic index
        ti = self.ti_calculator.calculate(wt_sequence, rescue_sequence, rescue_mutations)

        # Delivery options
        delivery_options = self.delivery_assessor.assess_all_methods(
            ti.sequence_identity, ti.mutation_count
        )

        # Immunogenicity
        immuno = self.immuno_assessor.assess(wt_sequence, rescue_sequence, rescue_mutations)

        # Calculate overall clinical score
        clinical_score = self._calculate_clinical_score(
            patient_pop, ti, delivery_options, immuno
        )

        # Determine viability
        if clinical_score >= 70:
            viability = "high"
        elif clinical_score >= 50:
            viability = "moderate"
        elif clinical_score >= 30:
            viability = "low"
        else:
            viability = "not_viable"

        # Key advantages and challenges
        advantages = self._identify_advantages(patient_pop, ti, delivery_options)
        challenges = self._identify_challenges(ti, immuno, delivery_options)

        # Recommended path
        recommended_path = self._recommend_development_path(
            ti, delivery_options, patient_pop
        )

        return ClinicalImpactReport(
            rescue_name=name,
            cancer_mutation=cancer_mutation,
            rescue_mutations=rescue_mutations,
            patient_population=patient_pop,
            therapeutic_index=ti,
            delivery_options=delivery_options,
            immunogenicity=immuno,
            overall_clinical_score=clinical_score,
            clinical_viability=viability,
            key_advantages=advantages,
            key_challenges=challenges,
            recommended_development_path=recommended_path
        )

    def _calculate_clinical_score(self, patient_pop: PatientPopulation,
                                 ti: TherapeuticIndex,
                                 delivery_options: List[DeliveryFeasibility],
                                 immuno: ImmunogenicityAssessment) -> float:
        """Calculate overall clinical score (0-100)."""
        score = 0

        # Patient population (25 points max)
        if patient_pop.total_patients_per_year >= 10000:
            score += 25
        elif patient_pop.total_patients_per_year >= 5000:
            score += 20
        elif patient_pop.total_patients_per_year >= 1000:
            score += 15
        elif patient_pop.total_patients_per_year >= 100:
            score += 10
        else:
            score += 5

        # Therapeutic index (25 points max)
        if ti.sequence_identity >= 98:
            score += 15
        elif ti.sequence_identity >= 95:
            score += 10
        elif ti.sequence_identity >= 92:
            score += 5

        if ti.therapeutic_window == "wide":
            score += 10
        elif ti.therapeutic_window == "moderate":
            score += 5

        # Delivery feasibility (25 points max)
        best_delivery = delivery_options[0] if delivery_options else None
        if best_delivery:
            score += best_delivery.feasibility_score * 25

        # Immunogenicity (25 points max, inverted)
        immuno_penalty = immuno.risk_score * 25
        score += (25 - immuno_penalty)

        return min(100, max(0, score))

    def _identify_advantages(self, patient_pop: PatientPopulation,
                            ti: TherapeuticIndex,
                            delivery_options: List[DeliveryFeasibility]) -> List[str]:
        """Identify key advantages."""
        advantages = []

        if patient_pop.total_patients_per_year >= 5000:
            advantages.append(f"Large addressable patient population ({patient_pop.total_patients_per_year:,} patients/year in US)")

        if ti.sequence_identity >= 97:
            advantages.append(f"High sequence identity ({ti.sequence_identity:.1f}%) minimizes immunogenicity")

        if ti.mutation_count <= 3:
            advantages.append(f"Minimal mutations ({ti.mutation_count}) for regulatory simplicity")

        if delivery_options and delivery_options[0].feasibility_score >= 0.7:
            advantages.append(f"Favorable delivery via {delivery_options[0].method}")

        if ti.therapeutic_window == "wide":
            advantages.append("Wide therapeutic window provides safety margin")

        return advantages[:5]

    def _identify_challenges(self, ti: TherapeuticIndex,
                            immuno: ImmunogenicityAssessment,
                            delivery_options: List[DeliveryFeasibility]) -> List[str]:
        """Identify key challenges."""
        challenges = []

        if ti.sequence_identity < 95:
            challenges.append(f"Lower sequence identity ({ti.sequence_identity:.1f}%) may increase immunogenicity")

        if ti.mutation_count > 5:
            challenges.append(f"Multiple mutations ({ti.mutation_count}) increase regulatory complexity")

        if immuno.overall_risk in ["moderate", "high"]:
            challenges.append(f"{immuno.overall_risk.capitalize()} immunogenicity risk requires monitoring")

        if ti.regulatory_complexity == "high":
            challenges.append("Complex regulatory pathway expected")

        if delivery_options:
            challenges.extend(delivery_options[0].challenges[:2])

        return challenges[:5]

    def _recommend_development_path(self, ti: TherapeuticIndex,
                                   delivery_options: List[DeliveryFeasibility],
                                   patient_pop: PatientPopulation) -> str:
        """Recommend development path."""
        if ti.sequence_identity >= 97 and ti.mutation_count <= 3:
            best_delivery = delivery_options[0].method if delivery_options else "AAV"
            return f"Proceed with {best_delivery} delivery. Fast-track potential for orphan indication. Target initial trial in {patient_pop.by_cancer_type.get(list(patient_pop.by_cancer_type.keys())[0] if patient_pop.by_cancer_type else 'solid tumors', 'solid tumors')} patients."

        elif ti.sequence_identity >= 94:
            return "Consider mRNA-LNP delivery for transient expression. Requires repeat dosing but lower immunogenicity risk. Phase 1 dose-escalation recommended."

        elif ti.sequence_identity >= 90:
            return "Ex vivo cell therapy approach recommended. Modify patient cells, validate correction, then reinfuse. Higher complexity but better safety profile."

        else:
            return "Consider alternative rescue designs with fewer mutations. Current design may face significant immunogenicity and regulatory challenges."


# ============================================================================
# EXPORT FUNCTIONS
# ============================================================================

def export_clinical_report(report: ClinicalImpactReport, output_path: str) -> str:
    """Export clinical impact report to markdown."""

    lines = []
    lines.append(f"# Clinical Impact Report: {report.rescue_name}")
    lines.append(f"\nGenerated for ISEF 2026")

    lines.append(f"\n## Summary")
    lines.append(f"- **Cancer Mutation:** {report.cancer_mutation}")
    lines.append(f"- **Rescue Mutations:** {', '.join(report.rescue_mutations)}")
    lines.append(f"- **Clinical Viability:** {report.clinical_viability.upper()}")
    lines.append(f"- **Overall Score:** {report.overall_clinical_score:.0f}/100")

    lines.append(f"\n## Patient Population")
    lines.append(f"- **Annual US Patients:** {report.patient_population.total_patients_per_year:,}")
    lines.append(f"- **Global Estimate:** {report.patient_population.global_estimate:,}")
    lines.append(f"- **5-Year Cumulative:** {report.patient_population.five_year_cumulative:,}")
    lines.append(f"\n### By Cancer Type")
    for cancer, count in report.patient_population.by_cancer_type.items():
        lines.append(f"  - {cancer.title()}: {count:,}")

    lines.append(f"\n## Therapeutic Index")
    lines.append(f"- **Sequence Identity:** {report.therapeutic_index.sequence_identity:.1f}%")
    lines.append(f"- **Mutation Count:** {report.therapeutic_index.mutation_count}")
    lines.append(f"- **Therapeutic Window:** {report.therapeutic_index.therapeutic_window}")
    lines.append(f"- **Recommended FDA Pathway:** {report.therapeutic_index.fda_pathway}")
    lines.append(f"- **Regulatory Complexity:** {report.therapeutic_index.regulatory_complexity}")

    lines.append(f"\n## Delivery Options")
    for i, delivery in enumerate(report.delivery_options[:3], 1):
        lines.append(f"\n### {i}. {delivery.method}")
        lines.append(f"- **Feasibility Score:** {delivery.feasibility_score:.2f}")
        lines.append(f"- **Cost:** {delivery.estimated_cost_per_dose}")
        lines.append(f"- **Regulatory Status:** {delivery.regulatory_status}")
        lines.append(f"- **Advantages:** {', '.join(delivery.advantages[:2])}")
        lines.append(f"- **Challenges:** {', '.join(delivery.challenges[:2])}")

    lines.append(f"\n## Immunogenicity Assessment")
    lines.append(f"- **Overall Risk:** {report.immunogenicity.overall_risk.upper()}")
    lines.append(f"- **Risk Score:** {report.immunogenicity.risk_score:.2f}")
    lines.append(f"- **Predicted Neoantigens:** {report.immunogenicity.neoantigens_predicted}")
    if report.immunogenicity.risk_factors:
        lines.append(f"- **Risk Factors:**")
        for rf in report.immunogenicity.risk_factors[:3]:
            lines.append(f"  - {rf}")
    if report.immunogenicity.mitigation_strategies:
        lines.append(f"- **Mitigation Strategies:**")
        for ms in report.immunogenicity.mitigation_strategies[:3]:
            lines.append(f"  - {ms}")

    lines.append(f"\n## Key Advantages")
    for adv in report.key_advantages:
        lines.append(f"- {adv}")

    lines.append(f"\n## Key Challenges")
    for chal in report.key_challenges:
        lines.append(f"- {chal}")

    lines.append(f"\n## Recommended Development Path")
    lines.append(f"\n{report.recommended_development_path}")

    content = "\n".join(lines)

    with open(output_path, 'w') as f:
        f.write(content)

    return output_path


# ============================================================================
# CLI INTERFACE
# ============================================================================

def main():
    """Command-line interface for clinical impact assessment."""
    import argparse

    parser = argparse.ArgumentParser(description="p53-proteoMgCAD Clinical Impact")
    parser.add_argument("--cancer", type=str, required=True,
                       help="Cancer mutation (e.g., R175H)")
    parser.add_argument("--rescue", type=str, nargs='+', required=True,
                       help="Rescue mutations (e.g., N268D T125R)")
    parser.add_argument("--output", type=str, default="clinical_impact_report.md",
                       help="Output file path")

    args = parser.parse_args()

    # Load WT sequence
    from ..data.dms import P53_WT
    wt_sequence = P53_WT

    # Apply rescue mutations
    rescue_sequence = wt_sequence
    for mut in args.rescue:
        pos = int(mut[1:-1]) - 1
        new_aa = mut[-1]
        rescue_sequence = rescue_sequence[:pos] + new_aa + rescue_sequence[pos+1:]

    # Generate report
    engine = ClinicalImpactEngine()
    report = engine.generate_report(
        name=f"p53_{args.cancer}_rescue",
        wt_sequence=wt_sequence,
        rescue_sequence=rescue_sequence,
        cancer_mutation=args.cancer,
        rescue_mutations=args.rescue
    )

    # Export
    export_clinical_report(report, args.output)

    # Print summary
    print("\n" + "="*60)
    print("CLINICAL IMPACT ASSESSMENT")
    print("="*60)
    print(f"\nCancer Mutation: {args.cancer}")
    print(f"Rescue Mutations: {', '.join(args.rescue)}")
    print(f"\nClinical Viability: {report.clinical_viability.upper()}")
    print(f"Overall Score: {report.overall_clinical_score:.0f}/100")
    print(f"Patient Population: {report.patient_population.total_patients_per_year:,} patients/year (US)")
    print(f"\nFull report saved to: {args.output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile p53cad/results/schema.py
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from itertools import combinations
import json
from typing import Any, Dict, Iterable, List, Sequence

import numpy as np
import pandas as pd

SCHEMA_VERSION = "1.0.0"

BIG8_HOTSPOTS = [
    "R175H",
    "R248Q",
    "R248W",
    "R273H",
    "R273C",
    "G245S",
    "R249S",
    "R282W",
]

DEFAULT_DELIVERY_METHODS = ["gene_therapy", "mrna_therapy", "protein_therapy"]


@dataclass(frozen=True)
class ScenarioSpec:
    scenario_id: str
    target_label: str
    targets: tuple[str, ...]
    delivery_method: str


def build_run_id(prefix: str = "campaign") -> str:
    ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    return f"{prefix}_{ts}"


def _normalize_mutation_label(muts: Sequence[str]) -> str:
    return "+".join(sorted(str(m).strip().upper() for m in muts if str(m).strip()))


def _normalize_mutation_token(value: Any) -> str:
    token = str(value).strip().upper()
    return "".join(ch for ch in token if not ch.isspace())


def _parse_mutation_list(value: Any, *, plus_separator: bool = False) -> List[str]:
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []

    raw_items: List[Any]
    if isinstance(value, (list, tuple, set)):
        raw_items = list(value)
    else:
        text = str(value).strip()
        if not text:
            return []
        try:
            parsed = json.loads(text)
            if isinstance(parsed, (list, tuple, set)):
                raw_items = list(parsed)
            else:
                raw_items = [text]
        except Exception:
            split_char = "+" if plus_separator and ("+" in text) else ","
            text_norm = text.replace(";", ",") if split_char == "," else text
            raw_items = [part for part in text_norm.split(split_char)]

    out = []
    for item in raw_items:
        token = _normalize_mutation_token(item)
        if token:
            out.append(token)
    return out


def _targets_for_row(row: pd.Series) -> List[str]:
    for key in ("targets_json", "targets"):
        if key in row and pd.notna(row.get(key)):
            vals = _parse_mutation_list(row.get(key))
            if vals:
                return vals

    label = str(row.get("target_label", "")).strip().upper()
    if label:
        vals = _parse_mutation_list(label, plus_separator=True)
        if vals:
            return vals

    scenario_id = str(row.get("scenario_id", ""))
    if "__" in scenario_id:
        target_part = scenario_id.split("__", 1)[0]
        vals = _parse_mutation_list(target_part, plus_separator=True)
        if vals:
            return vals

    return []


def _mutations_for_row(row: pd.Series) -> List[str]:
    for key in ("mutations_json", "mutations"):
        if key in row and pd.notna(row.get(key)):
            vals = _parse_mutation_list(row.get(key))
            if vals or str(row.get(key)).strip() == "[]":
                return vals
    return []


def _jaccard_overlap(set_a: set[str], set_b: set[str]) -> float:
    if not set_a and not set_b:
        return 1.0
    union = set_a | set_b
    if not union:
        return 0.0
    return float(len(set_a & set_b) / len(union))


def build_scenario_matrix(
    hotspots: Sequence[str] | None = None,
    delivery_methods: Sequence[str] | None = None,
    include_pairs: bool = True,
) -> List[ScenarioSpec]:
    base = [str(m).strip().upper() for m in (hotspots or BIG8_HOTSPOTS) if str(m).strip()]
    modes = [str(m).strip().lower() for m in (delivery_methods or DEFAULT_DELIVERY_METHODS) if str(m).strip()]

    singles = [(m,) for m in base]
    pairs = list(combinations(base, 2)) if include_pairs else []
    targets = singles + pairs

    scenarios: List[ScenarioSpec] = []
    for mut_group in targets:
        target_label = _normalize_mutation_label(mut_group)
        for delivery in modes:
            scenario_id = f"{target_label}__{delivery}"
            scenarios.append(
                ScenarioSpec(
                    scenario_id=scenario_id,
                    target_label=target_label,
                    targets=tuple(mut_group),
                    delivery_method=delivery,
                )
            )
    return scenarios


def scenarios_to_frame(scenarios: Iterable[ScenarioSpec]) -> pd.DataFrame:
    rows = []
    for s in scenarios:
        rows.append(
            {
                "scenario_id": s.scenario_id,
                "target_label": s.target_label,
                "targets": list(s.targets),
                "delivery_method": s.delivery_method,
                "n_targets": len(s.targets),
            }
        )
    return pd.DataFrame(rows)


def select_presentation_shortlist(
    candidates_df: pd.DataFrame,
    top_n: int = 30,
    max_per_target: int = 4,
    delivery_methods: Sequence[str] | None = None,
    min_per_delivery: int = 2,
    max_per_base_mutation: int = 12,
) -> pd.DataFrame:
    """
    Rank and select presentation candidates with strict rescue semantics.

    Strict policy:
      - retain all target mutation(s)
      - include at least one additional second-site mutation
      - dedupe by sequence and canonical mutation set

    Ranking base:
      - trust-adjusted score (desc)
      - clinical_score (desc)
      - uncertainty (asc)
      - ood_distance (asc)
      - n_mutations (asc)

    Diversity stage:
      - quota seeds across delivery, profile, and target hotspots
      - delivery floor ensures min_per_delivery per method
      - base-mutation cap prevents any single hotspot from dominating via combos
      - greedy diversity fill using normalized score + novelty - overlap penalty
    """
    if candidates_df is None or candidates_df.empty:
        cols = list(candidates_df.columns) if candidates_df is not None else []
        for needed in ["selection_score", "diversity_novelty", "max_mutation_overlap", "selection_reason", "presentation_rank"]:
            if needed not in cols:
                cols.append(needed)
        return pd.DataFrame(columns=cols)

    work = candidates_df.copy().reset_index(drop=True)

    for col, default in [
        ("score", -np.inf),
        ("clinical_score", -np.inf),
        ("pareto_rank", np.inf),
        ("rescue_dms_mean", np.inf),
        ("uncertainty", np.inf),
        ("ood_distance", np.inf),
        ("n_mutations", np.inf),
        ("target_label", "unknown"),
        ("delivery_method", "unknown"),
        ("profile", "Unknown"),
        ("candidate_uid", ""),
        ("sequence", ""),
    ]:
        if col not in work.columns:
            work[col] = default

    work["score"] = pd.to_numeric(work["score"], errors="coerce").fillna(-np.inf)
    work["clinical_score"] = pd.to_numeric(work["clinical_score"], errors="coerce").fillna(-np.inf)
    work["pareto_rank"] = pd.to_numeric(work["pareto_rank"], errors="coerce").fillna(np.inf)
    work["rescue_dms_mean"] = pd.to_numeric(work["rescue_dms_mean"], errors="coerce").fillna(np.inf)
    work["uncertainty"] = pd.to_numeric(work["uncertainty"], errors="coerce").fillna(np.inf)
    work["ood_distance"] = pd.to_numeric(work["ood_distance"], errors="coerce").fillna(np.inf)
    work["n_mutations"] = pd.to_numeric(work["n_mutations"], errors="coerce").fillna(np.inf)

    work["_target_list"] = work.apply(_targets_for_row, axis=1)
    work["_mutation_list"] = work.apply(_mutations_for_row, axis=1)
    work["_target_set"] = work["_target_list"].apply(lambda xs: set(xs))
    work["_mutation_set"] = work["_mutation_list"].apply(lambda xs: set(xs))
    work["_rescue_set"] = work.apply(lambda r: set(r["_mutation_set"]) - set(r["_target_set"]), axis=1)
    work["_retains_targets"] = work.apply(
        lambda r: bool(r["_target_set"]) and set(r["_target_set"]).issubset(set(r["_mutation_set"])),
        axis=1,
    )
    work["_has_second_site"] = work["_rescue_set"].apply(lambda xs: len(xs) >= 1)

    # Strict retain+add filter.
    work = work[work["_retains_targets"] & work["_has_second_site"]].copy()
    if work.empty:
        cols = list(candidates_df.columns)
        for needed in ["selection_score", "diversity_novelty", "max_mutation_overlap", "selection_reason", "presentation_rank"]:
            if needed not in cols:
                cols.append(needed)
        return pd.DataFrame(columns=cols)

    ranking_cols = ["pareto_rank", "score", "clinical_score", "rescue_dms_mean", "uncertainty", "ood_distance", "n_mutations"]
    ranking_asc = [True, False, False, True, True, True, True]

    work = work.sort_values(by=ranking_cols, ascending=ranking_asc, kind="mergesort").reset_index(drop=True)
    work["_candidate_uid"] = work["candidate_uid"].astype(str)
    work["_sequence_key"] = work.apply(
        lambda r: str(r.get("sequence", "")).strip().upper() or f"uid:{r['_candidate_uid']}",
        axis=1,
    )
    work["_mutation_key"] = work.apply(
        lambda r: "|".join(sorted(r["_mutation_set"])) or f"uid:{r['_candidate_uid']}",
        axis=1,
    )
    work["_delivery_key"] = work["delivery_method"].astype(str).str.strip().str.lower()

    # Dedupe: keep highest-ranked row per (sequence, delivery), then per (mutation set, delivery).
    # Including delivery_method ensures each delivery method retains its own representative
    # even when the underlying protein sequence is identical (e.g. mrna vs protein therapy).
    work = work.drop_duplicates(subset=["_sequence_key", "_delivery_key"], keep="first")
    work = work.drop_duplicates(subset=["_mutation_key", "_delivery_key"], keep="first")
    work = work.sort_values(by=ranking_cols, ascending=ranking_asc, kind="mergesort").reset_index(drop=True)

    # Normalize scores to [0, 1] so diversity terms are on a comparable scale.
    finite_scores = work["score"][np.isfinite(work["score"])]
    if len(finite_scores) > 1:
        score_min = float(finite_scores.min())
        score_max = float(finite_scores.max())
    else:
        score_min, score_max = 0.0, 1.0
    score_range = max(score_max - score_min, 1e-6)
    work["_score_norm"] = ((work["score"] - score_min) / score_range).clip(0.0, 1.0)

    # DMS quality bonus: negative rescue_dms_mean = individually functional rescue mutations.
    # Normalize to [0, 1] where 1 = best DMS quality (most negative Z-scores).
    finite_dms = work["rescue_dms_mean"][np.isfinite(work["rescue_dms_mean"])]
    if len(finite_dms) > 1:
        dms_min = float(finite_dms.min())  # most negative = best
        dms_max = float(finite_dms.max())
        dms_range = max(dms_max - dms_min, 1e-6)
        # Invert: lower Z = higher bonus
        work["_dms_quality_norm"] = ((dms_max - work["rescue_dms_mean"]) / dms_range).clip(0.0, 1.0)
    else:
        work["_dms_quality_norm"] = 0.0
    work["_dms_quality_norm"] = work["_dms_quality_norm"].fillna(0.0)

    desired_delivery = [str(d).strip().lower() for d in (delivery_methods or DEFAULT_DELIVERY_METHODS)]
    selected_indices: List[int] = []
    selected_index_set: set[int] = set()
    stage_reason: Dict[int, str] = {}
    target_counts: Dict[str, int] = {}
    base_mut_counts: Dict[str, int] = {}

    def _can_take(idx: int) -> bool:
        if idx in selected_index_set:
            return False
        target = str(work.at[idx, "target_label"])
        if target_counts.get(target, 0) >= int(max_per_target):
            return False
        # Base-mutation cap: prevent any single hotspot from dominating via combos.
        target_set = work.at[idx, "_target_set"]
        for mut in target_set:
            if base_mut_counts.get(mut, 0) >= int(max_per_base_mutation):
                return False
        return True

    def _take(idx: int, reason: str) -> bool:
        if not _can_take(idx):
            return False
        selected_indices.append(idx)
        selected_index_set.add(idx)
        target = str(work.at[idx, "target_label"])
        target_counts[target] = target_counts.get(target, 0) + 1
        for mut in work.at[idx, "_target_set"]:
            base_mut_counts[mut] = base_mut_counts.get(mut, 0) + 1
        stage_reason[idx] = reason
        return True

    # Quota seed 1: at least one per delivery if available.
    delivery_series = work["delivery_method"].astype(str).str.lower()
    for delivery in desired_delivery:
        matches = work.index[delivery_series == delivery].tolist()
        for idx in matches:
            if _take(int(idx), f"quota_delivery_{delivery}"):
                break

    # Quota seed 2: at least one per profile if available.
    for profile in sorted(work["profile"].astype(str).unique()):
        matches = work.index[work["profile"].astype(str) == profile].tolist()
        for idx in matches:
            if _take(int(idx), f"quota_profile_{profile}"):
                break

    # Quota seed 3: target diversity — at least one per Big-8 hotspot.
    target_label_upper = work["target_label"].astype(str).str.upper()
    for hotspot in BIG8_HOTSPOTS:
        # Match any target_label containing this hotspot (single or combo).
        matches = work.index[target_label_upper.str.contains(hotspot, regex=False)].tolist()
        for idx in matches:
            if _take(int(idx), f"quota_target_{hotspot}"):
                break

    # Quota seed 4: delivery floor — ensure min_per_delivery per method.
    delivery_selected_counts: Dict[str, int] = {}
    for idx in selected_indices:
        d = str(work.at[idx, "delivery_method"]).strip().lower()
        delivery_selected_counts[d] = delivery_selected_counts.get(d, 0) + 1
    for delivery in desired_delivery:
        current = delivery_selected_counts.get(delivery, 0)
        if current >= int(min_per_delivery):
            continue
        matches = work.index[delivery_series == delivery].tolist()
        for idx in matches:
            if delivery_selected_counts.get(delivery, 0) >= int(min_per_delivery):
                break
            if _take(int(idx), f"quota_delivery_floor_{delivery}"):
                delivery_selected_counts[delivery] = delivery_selected_counts.get(delivery, 0) + 1

    novelty_weight = 0.35
    overlap_penalty = 0.20
    dms_quality_weight = 0.25

    def _greedy_utility(idx: int, selected: Sequence[int]) -> tuple[float, float, float]:
        rescue_set = set(work.at[idx, "_rescue_set"])
        if not selected:
            novelty = 1.0
            max_overlap = 0.0
        else:
            overlaps = [_jaccard_overlap(rescue_set, set(work.at[s, "_rescue_set"])) for s in selected]
            max_overlap = float(max(overlaps)) if overlaps else 0.0
            novelty = float(1.0 - np.mean(overlaps)) if overlaps else 1.0
        dms_bonus = float(work.at[idx, "_dms_quality_norm"]) * dms_quality_weight
        utility = float(work.at[idx, "_score_norm"]) + dms_bonus + (novelty_weight * novelty) - (overlap_penalty * max_overlap)
        return utility, novelty, max_overlap

    # Fill remaining slots with greedy diversity utility.
    while len(selected_indices) < int(top_n):
        remaining = [int(i) for i in work.index if i not in selected_index_set and _can_take(int(i))]
        if not remaining:
            break

        best_idx = -1
        best_utility = -float("inf")
        best_novelty = 0.0
        best_overlap = 0.0
        for idx in remaining:
            utility, novelty, max_overlap = _greedy_utility(idx, selected_indices)
            if utility > best_utility + 1e-12:
                best_idx = idx
                best_utility = utility
                best_novelty = novelty
                best_overlap = max_overlap
            elif abs(utility - best_utility) <= 1e-12:
                # Stable deterministic tie-break by rank order (lower index is better).
                if best_idx < 0 or idx < best_idx:
                    best_idx = idx
                    best_utility = utility
                    best_novelty = novelty
                    best_overlap = max_overlap

        if best_idx < 0:
            break
        if _take(best_idx, "diversity_fill"):
            stage_reason[best_idx] = (
                f"diversity_fill_utility={best_utility:.3f}_novelty={best_novelty:.3f}_overlap={best_overlap:.3f}"
            )

    if not selected_indices:
        cols = list(candidates_df.columns)
        for needed in ["selection_score", "diversity_novelty", "max_mutation_overlap", "selection_reason", "presentation_rank"]:
            if needed not in cols:
                cols.append(needed)
        return pd.DataFrame(columns=cols)

    # Recompute selection metrics in selected order for deterministic persisted metadata.
    selected_rows = []
    prior_selected: List[int] = []
    for rank, idx in enumerate(selected_indices[: int(top_n)], start=1):
        utility, novelty, max_overlap = _greedy_utility(idx, prior_selected)
        row = work.loc[idx].copy()
        row["selection_score"] = float(utility)
        row["diversity_novelty"] = float(novelty)
        row["max_mutation_overlap"] = float(max_overlap)
        row["selection_reason"] = str(stage_reason.get(idx, "trust_adjusted_rank"))
        row["presentation_rank"] = int(rank)
        selected_rows.append(row)
        prior_selected.append(idx)

    result = pd.DataFrame(selected_rows)
    drop_cols = [c for c in result.columns if c.startswith("_")]
    result = result.drop(columns=drop_cols, errors="ignore")

    # Keep predictable column order with new fields near the end.
    base_cols = [c for c in candidates_df.columns if c in result.columns]
    extra_cols = [c for c in ["selection_score", "diversity_novelty", "max_mutation_overlap", "selection_reason", "presentation_rank"] if c in result.columns]
    ordered_cols = base_cols + [c for c in extra_cols if c not in base_cols]
    result = result.loc[:, ordered_cols]
    return result.reset_index(drop=True)


In [ ]:
%%writefile p53cad/results/store.py
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
import json
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

import pandas as pd

from p53cad.core.logging import get_logger
from p53cad.results.schema import SCHEMA_VERSION


logger = get_logger(__name__)


@dataclass
class RunPaths:
    run_dir: Path
    manifest_path: Path
    scenarios_path: Path
    candidates_path: Path
    trajectories_path: Path
    clinical_path: Path
    top30_path: Path
    top30_csv_path: Path
    summary_path: Path
    checkpoint_state_path: Path
    checkpoint_scenarios_path: Path


class CampaignStore:
    def __init__(self, base_dir: Path | str = "data/campaigns"):
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)
        self.index_path = self.base_dir / "index.json"

    def run_paths(self, run_id: str) -> RunPaths:
        run_dir = self.base_dir / str(run_id)
        return RunPaths(
            run_dir=run_dir,
            manifest_path=run_dir / "manifest.json",
            scenarios_path=run_dir / "scenarios.parquet",
            candidates_path=run_dir / "candidates.parquet",
            trajectories_path=run_dir / "trajectories.parquet",
            clinical_path=run_dir / "clinical.parquet",
            top30_path=run_dir / "top30.parquet",
            top30_csv_path=run_dir / "top30.csv",
            summary_path=run_dir / "summary.md",
            checkpoint_state_path=run_dir / "checkpoints" / "state.json",
            checkpoint_scenarios_path=run_dir / "checkpoints" / "scenario_status.jsonl",
        )

    def init_run(
        self,
        run_id: str,
        config: Dict[str, Any],
        runtime_caps: Optional[Dict[str, Any]] = None,
        resume: bool = False,
    ) -> RunPaths:
        paths = self.run_paths(run_id)
        paths.run_dir.mkdir(parents=True, exist_ok=True)
        paths.checkpoint_state_path.parent.mkdir(parents=True, exist_ok=True)

        if paths.manifest_path.exists() and resume:
            return paths

        manifest = {
            "run_id": run_id,
            "schema_version": SCHEMA_VERSION,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
            "status": "initialized",
            "config": config,
            "runtime_capabilities": runtime_caps or {},
            "artifacts": {
                "scenarios": paths.scenarios_path.name,
                "candidates": paths.candidates_path.name,
                "trajectories": paths.trajectories_path.name,
                "clinical": paths.clinical_path.name,
                "top30": paths.top30_path.name,
                "top30_csv": paths.top30_csv_path.name,
                "summary": paths.summary_path.name,
            },
        }
        self._write_json(paths.manifest_path, manifest)
        self._update_index(run_id, status="initialized")
        return paths

    def update_manifest(self, run_id: str, patch: Dict[str, Any]) -> None:
        paths = self.run_paths(run_id)
        if not paths.manifest_path.exists():
            raise FileNotFoundError(f"Manifest not found for run {run_id}")
        manifest = self._read_json(paths.manifest_path)
        manifest.update(patch)
        manifest["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
        self._write_json(paths.manifest_path, manifest)
        if "status" in patch:
            self._update_index(run_id, status=str(patch["status"]))

    def append_scenario_checkpoint(self, run_id: str, payload: Dict[str, Any]) -> None:
        paths = self.run_paths(run_id)
        paths.checkpoint_scenarios_path.parent.mkdir(parents=True, exist_ok=True)
        with paths.checkpoint_scenarios_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(payload) + "\n")

    def load_scenario_checkpoints(self, run_id: str) -> pd.DataFrame:
        paths = self.run_paths(run_id)
        if not paths.checkpoint_scenarios_path.exists():
            return pd.DataFrame()
        rows = []
        with paths.checkpoint_scenarios_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    logger.warning("Skipping malformed checkpoint row in %s", paths.checkpoint_scenarios_path)
        return pd.DataFrame(rows)

    def write_tables(
        self,
        run_id: str,
        scenarios_df: pd.DataFrame,
        candidates_df: pd.DataFrame,
        trajectories_df: pd.DataFrame,
        clinical_df: pd.DataFrame,
    ) -> None:
        paths = self.run_paths(run_id)
        self._write_parquet(scenarios_df, paths.scenarios_path)
        self._write_parquet(candidates_df, paths.candidates_path)
        self._write_parquet(trajectories_df, paths.trajectories_path)
        self._write_parquet(clinical_df, paths.clinical_path)

    def write_top30(self, run_id: str, top30_df: pd.DataFrame) -> None:
        paths = self.run_paths(run_id)
        self._write_parquet(top30_df, paths.top30_path)
        top30_df.to_csv(paths.top30_csv_path, index=False)

    def write_summary(self, run_id: str, text: str) -> None:
        paths = self.run_paths(run_id)
        paths.summary_path.write_text(text, encoding="utf-8")

    def write_validation(self, run_id: str, filename: str, data: Any) -> Path:
        """Write a JSON validation artifact to the run directory."""
        paths = self.run_paths(run_id)
        out_path = paths.run_dir / filename
        out_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
        return out_path

    def load_run_bundle(self, run_id: str) -> Dict[str, Any]:
        paths = self.run_paths(run_id)
        manifest = self._read_json(paths.manifest_path)

        def _safe_read(path: Path) -> pd.DataFrame:
            if path.exists():
                return pd.read_parquet(path)
            return pd.DataFrame()

        return {
            "manifest": manifest,
            "scenarios": _safe_read(paths.scenarios_path),
            "candidates": _safe_read(paths.candidates_path),
            "trajectories": _safe_read(paths.trajectories_path),
            "clinical": _safe_read(paths.clinical_path),
            "top30": _safe_read(paths.top30_path),
            "run_dir": paths.run_dir,
        }

    def list_runs(self) -> pd.DataFrame:
        if not self.index_path.exists():
            return pd.DataFrame(columns=["run_id", "status", "updated_at_utc", "path"])
        payload = self._read_json(self.index_path)
        runs = payload.get("runs", []) if isinstance(payload, dict) else []
        if not runs:
            return pd.DataFrame(columns=["run_id", "status", "updated_at_utc", "path"])
        df = pd.DataFrame(runs)
        if "updated_at_utc" in df.columns:
            df = df.sort_values("updated_at_utc", ascending=False)
        return df

    def latest_run_id(self) -> Optional[str]:
        df = self.list_runs()
        if df.empty:
            return None
        return str(df.iloc[0]["run_id"])

    def _write_parquet(self, df: pd.DataFrame, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(path, index=False)

    def _read_json(self, path: Path) -> Dict[str, Any]:
        return json.loads(path.read_text(encoding="utf-8"))

    def _write_json(self, path: Path, payload: Dict[str, Any]) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    def _update_index(self, run_id: str, status: str) -> None:
        now = datetime.now(timezone.utc).isoformat()
        payload = {"runs": []}
        if self.index_path.exists():
            try:
                payload = self._read_json(self.index_path)
            except Exception:
                payload = {"runs": []}

        runs = payload.get("runs", []) if isinstance(payload, dict) else []
        runs = [r for r in runs if str(r.get("run_id")) != str(run_id)]
        runs.append(
            {
                "run_id": str(run_id),
                "status": str(status),
                "updated_at_utc": now,
                "path": str((self.base_dir / str(run_id)).resolve()),
            }
        )
        runs = sorted(runs, key=lambda r: str(r.get("updated_at_utc", "")), reverse=True)
        self._write_json(self.index_path, {"runs": runs})

    def generate_pymol_scripts(self, run_id: str, top_n: int = 5) -> list[Path]:
        """Auto-generate PyMOL PML scripts for a campaign's top candidates.

        This is intended to be called after :meth:`write_top30` (or any
        finalisation step) so that ready-to-use visualisation scripts are
        stored alongside the other campaign artefacts.

        Parameters
        ----------
        run_id : str
            The campaign run identifier.
        top_n : int
            Number of top candidates to generate scripts for (default 5).

        Returns
        -------
        list[Path]
            Paths of the generated ``.pml`` files, or an empty list if
            generation was skipped (e.g. no top-30 table found).
        """
        paths = self.run_paths(run_id)

        if not paths.top30_path.exists():
            logger.warning(
                "Top-30 table not found for run %s – skipping PyMOL generation",
                run_id,
            )
            return []

        top_df = pd.read_parquet(paths.top30_path)
        if top_df.empty:
            logger.warning("Top-30 table is empty for run %s", run_id)
            return []

        pymol_dir = paths.run_dir / "pymol"
        pymol_dir.mkdir(parents=True, exist_ok=True)

        try:
            from p53cad.viz.pymol import PyMolGenerator

            gen = PyMolGenerator()
            generated = gen.generate_campaign_pml(top_df, pymol_dir, top_n=top_n)
            logger.info(
                "Generated %d PyMOL scripts for run %s in %s",
                len(generated),
                run_id,
                pymol_dir,
            )

            # Record the artefact in the manifest so downstream tools can
            # discover it.
            self.update_manifest(run_id, {"artifacts.pymol_dir": str(pymol_dir)})
            return generated
        except Exception:
            logger.exception("Failed to generate PyMOL scripts for run %s", run_id)
            return []


In [ ]:
%%writefile p53cad/results/visualization.py
from __future__ import annotations

from typing import Any, Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.interpolate import griddata


def _parse_mutation_position(mut: str) -> int | None:
    text = str(mut).strip().upper()
    digits = "".join(ch for ch in text if ch.isdigit())
    if not digits:
        return None
    try:
        return int(digits)
    except ValueError:
        return None



def build_candidate_position_heatmap(
    candidates: Sequence[Dict[str, Any]],
    max_positions: int = 40,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      - matrix_df: rows candidates, cols informative positions, values 0/1 mutation presence
      - freq_df: position-level mutation frequency summary
    """
    rows: List[Dict[str, Any]] = []
    position_counts: Dict[int, int] = {}

    for cand in candidates:
        mut_positions = cand.get("mut_positions") or []
        if not mut_positions:
            mut_positions = [
                p for p in (_parse_mutation_position(m) for m in (cand.get("mutations") or [])) if p is not None
            ]
        mut_set = {int(p) for p in mut_positions}
        for pos in mut_set:
            position_counts[pos] = position_counts.get(pos, 0) + 1
        rows.append(
            {
                "candidate_label": f"#{cand.get('candidate_id', '?')} {cand.get('profile', 'Unknown')}",
                "score": float(cand.get("score", 0.0)),
                "positions": mut_set,
            }
        )

    if not rows:
        return pd.DataFrame(), pd.DataFrame(columns=["position", "count", "frequency"]) 

    n_candidates = len(rows)
    freq_df = pd.DataFrame(
        [
            {
                "position": int(pos),
                "count": int(count),
                "frequency": float(count / max(n_candidates, 1)),
            }
            for pos, count in sorted(position_counts.items(), key=lambda kv: (-kv[1], kv[0]))
        ]
    )

    if freq_df.empty:
        return pd.DataFrame(), freq_df

    selected_positions = freq_df["position"].tolist()[: max(1, int(max_positions))]
    matrix = []
    for row in rows:
        matrix.append(
            {
                "candidate_label": row["candidate_label"],
                **{f"pos_{pos}": (1 if pos in row["positions"] else 0) for pos in selected_positions},
            }
        )
    matrix_df = pd.DataFrame(matrix)
    return matrix_df, freq_df



def build_multi_metric_frame(
    traj_df: pd.DataFrame,
    *,
    normalize: bool,
    min_function: float,
    min_stability: float,
    min_binding: float,
    min_identity: float,
) -> pd.DataFrame:
    out = traj_df.copy()
    if out.empty:
        return out

    for col in ["score", "stability", "binding", "identity"]:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = pd.to_numeric(out[col], errors="coerce")

    if not normalize:
        return out

    # Threshold-anchored normalization (not per-trace min/max)
    score_hi = max(float(min_function) + 1.0, float(np.nanpercentile(out["score"], 95)) if out["score"].notna().any() else float(min_function) + 1.0)
    out["score"] = np.clip((out["score"] - float(min_function)) / max(score_hi - float(min_function), 1e-6), 0.0, 1.0)

    out["stability"] = np.clip((out["stability"] - float(min_stability)) / max(0.0 - float(min_stability), 1e-6), 0.0, 1.0)
    out["binding"] = np.clip((out["binding"] - float(min_binding)) / max(10.0 - float(min_binding), 1e-6), 0.0, 1.0)
    out["identity"] = np.clip((out["identity"] - float(min_identity)) / max(100.0 - float(min_identity), 1e-6), 0.0, 1.0)
    return out



def build_loss_mesh(
    traj_df: pd.DataFrame,
    grid_size: int = 35,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray] | None:
    """
    Build interpolated loss mesh from trajectory points.

    Returns (grid_x, grid_y, grid_z) or None if insufficient points.
    """
    needed = {"lx", "ly", "loss_total"}
    if traj_df is None or traj_df.empty or not needed.issubset(set(traj_df.columns)):
        return None

    work = traj_df[list(needed)].copy()
    work = work.apply(pd.to_numeric, errors="coerce").dropna()
    if len(work) < 12:
        return None

    # Need enough unique points for interpolation
    if work[["lx", "ly"]].drop_duplicates().shape[0] < 8:
        return None

    x = work["lx"].to_numpy()
    y = work["ly"].to_numpy()
    z = work["loss_total"].to_numpy()

    gx = np.linspace(float(np.min(x)), float(np.max(x)), int(max(grid_size, 12)))
    gy = np.linspace(float(np.min(y)), float(np.max(y)), int(max(grid_size, 12)))
    grid_x, grid_y = np.meshgrid(gx, gy)

    grid_z = griddata((x, y), z, (grid_x, grid_y), method="linear")
    if np.isnan(grid_z).all():
        return None

    # Fill sparse holes with nearest so surface is viewable.
    mask = np.isnan(grid_z)
    if np.any(mask):
        nearest = griddata((x, y), z, (grid_x, grid_y), method="nearest")
        grid_z[mask] = nearest[mask]

    return grid_x, grid_y, grid_z


### 0b. Upload Data Files

Upload the required data files when prompted:
1. **`p53_DMS_Giacomelli_2018.csv`** — DMS functional scores (~1.9 MB)
2. **`functional_oracle.pt`** — Trained oracle weights (~29 MB)
3. **`p53_wt.pdb`** — Wild-type PDB structure (optional, ~243 KB)

In [ ]:
import shutil

# Check if running in Colab
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    from google.colab import files
    print("Upload data files (DMS CSV, oracle weights, optionally PDB)...")
    print("You can select multiple files at once.\n")
    uploaded = files.upload()

    # Move files to expected locations
    for fname, content in uploaded.items():
        if fname.endswith('.csv'):
            dest = f'data/raw/{fname}'
            with open(dest, 'wb') as f:
                f.write(content)
            print(f"  -> {dest}")
        elif fname.endswith('.pt'):
            dest = f'data/models/{fname}'
            with open(dest, 'wb') as f:
                f.write(content)
            print(f"  -> {dest}")
        elif fname.endswith('.pdb'):
            dest = f'data/raw/{fname}'
            with open(dest, 'wb') as f:
                f.write(content)
            print(f"  -> {dest}")
        else:
            print(f"  Skipped unknown file: {fname}")
else:
    # Running locally — check if data files exist
    import os
    data_files = {
        'DMS CSV': 'data/raw/p53_DMS_Giacomelli_2018.csv',
        'Oracle weights': 'data/models/functional_oracle.pt',
        'WT PDB': 'data/raw/p53_wt.pdb',
    }
    for label, path in data_files.items():
        exists = os.path.exists(path)
        status = "found" if exists else "MISSING"
        print(f"  {label}: {path} [{status}]")

print("\nData setup complete!")

## 1. Environment Setup

In [ ]:
from p53cad.core.runtime import bootstrap_runtime, get_runtime_capabilities

bootstrap_runtime(seed=42)
caps = get_runtime_capabilities()

print("Runtime Capabilities")
print("=" * 50)
for key, val in caps.items():
    print(f"  {key:<35} {val}")

## 2. Inspect Data: Wild-Type p53 & DMS Dataset

In [ ]:
from p53cad.data.dms import P53_WT, get_dms_data

print(f"p53 Wild-Type Sequence ({len(P53_WT)} amino acids):")
# Print in blocks of 60 with position markers
for i in range(0, len(P53_WT), 60):
    chunk = P53_WT[i:i+60]
    print(f"  {i+1:>4}  {chunk}")

print(f"\nLoading DMS data (Giacomelli 2018)...")
dms_df = get_dms_data()
print(f"  Variants: {len(dms_df):,}")
print(f"  Columns: {list(dms_df.columns)}")
dms_df.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize DMS score distribution
score_col = [c for c in dms_df.columns if "Nutlin" in c and "Z" in c]
if score_col:
    scores = dms_df[score_col[0]].dropna()
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    ax.hist(scores, bins=80, color="#2563EB", alpha=0.7, edgecolor="white")
    ax.axvline(0, color="red", linestyle="--", label="Neutral (Z=0)")
    ax.axvline(-0.5, color="green", linestyle="--", alpha=0.7, label="Functional threshold (Z=-0.5)")
    ax.set_xlabel("Nutlin-3 Z-score")
    ax.set_ylabel("Count")
    ax.set_title("DMS Functional Score Distribution (Giacomelli 2018)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"  Mean Z-score: {scores.mean():.3f}")
    print(f"  Functional (Z<0): {(scores < 0).sum():,} / {len(scores):,} ({100*(scores < 0).mean():.1f}%)")
else:
    # Fallback for legacy format
    if 'score' in dms_df.columns:
        scores = dms_df['score'].dropna()
        fig, ax = plt.subplots(1, 1, figsize=(10, 4))
        ax.hist(scores, bins=80, color="#2563EB", alpha=0.7, edgecolor="white")
        ax.axvline(0, color="red", linestyle="--", label="Neutral (score=0)")
        ax.set_xlabel("Functional Score")
        ax.set_ylabel("Count")
        ax.set_title("DMS Functional Score Distribution")
        ax.legend()
        plt.tight_layout()
        plt.show()

## 3. Scenario Matrix

The campaign explores rescue mutations for 8 major p53 cancer hotspots (the "BIG8"),
plus pairwise combinations, across 3 delivery methods.

In [ ]:
from p53cad.results.schema import BIG8_HOTSPOTS, DEFAULT_DELIVERY_METHODS, build_scenario_matrix

print("BIG8 Cancer Hotspots:")
for i, h in enumerate(BIG8_HOTSPOTS, 1):
    print(f"  {i}. {h}")

print(f"\nDelivery Methods: {DEFAULT_DELIVERY_METHODS}")

# Build the full scenario matrix
scenarios = build_scenario_matrix(
    hotspots=BIG8_HOTSPOTS,
    delivery_methods=DEFAULT_DELIVERY_METHODS,
    include_pairs=True,
)
print(f"\nTotal scenarios: {len(scenarios)}")
print(f"  Single-hotspot: {sum(1 for s in scenarios if '+' not in s.target_label)}")
print(f"  Pair combos:    {sum(1 for s in scenarios if '+' in s.target_label)}")

# Show first few
print(f"\nFirst 10 scenarios:")
for s in scenarios[:10]:
    print(f"  {s.scenario_id:<40} targets={s.targets}  delivery={s.delivery_method}")

## 4. Run the Campaign

**Budget options:**
| Budget | Pass A Steps | Pass B Steps | Approx. Time |
|--------|-------------|-------------|---------------|
| `fast` | 40 steps × 1 restart | 120 steps × 1 restart | ~5 min (GPU) |
| `medium` | 60 steps × 1 restart | 200 steps × 2 restarts | ~30 min (GPU) |
| `high` | 80 steps × 1 restart | 280 steps × 2 restarts | ~2 hrs (GPU) |

Change `BUDGET` below to control run time. On Colab free tier (T4 GPU), `fast` is recommended.

In [ ]:
# ===== CONFIGURATION =====
BUDGET = "fast"          # "fast", "medium", or "high"
SEED = 42
INCLUDE_PAIRS = True     # Include pairwise hotspot combos
SHORTLIST_N = 30         # Number of top candidates to shortlist
# =========================

In [ ]:
import time
from p53cad.engine.campaign import CampaignRunner

print(f"Initializing CampaignRunner...")
runner = CampaignRunner()

# Report loaded model info
if runner.embedder is not None:
    print(f"  ESM-2 model: {runner.embedder.model_name}")
    print(f"  Hidden dim:  {runner.embedder.hidden_size}")
if runner.oracle is not None:
    print(f"  Oracle input_dim: {runner.oracle.input_dim}")
    arch = "attention_pooling" if hasattr(runner.oracle.model, 'attn') else "legacy_mlp"
    print(f"  Oracle architecture: {arch}")
print(f"  Pairwise DMS entries: {len(runner._pairwise_dms)}")
print(f"  Contact map entries:  {len(runner._wt_contacts)}")
print(f"\nReady to run.")

In [ ]:
print(f"Starting campaign (budget={BUDGET}, seed={SEED})...")
print(f"This may take a while depending on budget.\n")

t0 = time.time()

result = runner.run(
    budget=BUDGET,
    seed=SEED,
    include_pairs=INCLUDE_PAIRS,
    shortlist_n=SHORTLIST_N,
    with_clinical=True,
)

elapsed = time.time() - t0

print(f"\n{'=' * 60}")
print(f"  CAMPAIGN COMPLETE \u2014 {elapsed/60:.1f} min")
print(f"{'=' * 60}")
print(f"  Run ID:      {result['run_id']}")
print(f"  Scenarios:   {result['n_scenarios']}")
print(f"  Candidates:  {result['n_candidates']}")
print(f"  Shortlist:   {result['n_shortlist']}")
print(f"  Run dir:     {result['run_dir']}")

## 5. Results Analysis

In [ ]:
import json
import os
import pandas as pd
import numpy as np

run_dir = result["run_dir"]

# Load all candidates
cand = pd.read_parquet(os.path.join(run_dir, "candidates.parquet"))
print(f"Total candidates: {len(cand)}")
print(f"Columns: {list(cand.columns)}")

# Separate deep-refined candidates
if "pass_name" in cand.columns:
    deep = cand[cand["pass_name"] == "deep"].copy()
    print(f"Deep-refined candidates: {len(deep)}")
else:
    deep = cand.copy()
    print(f"All candidates (no pass separation): {len(deep)}")

cand.describe()

In [ ]:
# Target retention analysis
retains = 0
for _, row in deep.iterrows():
    targets = json.loads(str(row.get("targets_json", "[]")))
    muts = json.loads(str(row.get("mutations_json", "[]")))
    if set(targets).issubset(set(muts)):
        retains += 1

print(f"Target Retention: {retains}/{len(deep)} ({100*retains/max(len(deep),1):.1f}%)")

# Pareto ranking
if "pareto_rank" in deep.columns:
    rank1 = (deep["pareto_rank"] == 1).sum()
    max_rank = int(deep["pareto_rank"].replace([np.inf], np.nan).dropna().max())
    print(f"Pareto rank-1 (non-dominated): {rank1}")
    print(f"Total Pareto fronts: {max_rank}")

# DMS quality
if "rescue_dms_mean" in deep.columns:
    valid_dms = deep[deep["rescue_dms_mean"].notna() & (deep["rescue_dms_mean"] != 0)]
    print(f"\nCandidates with DMS data: {len(valid_dms)}")
    if len(valid_dms):
        print(f"  Mean rescue DMS Z: {valid_dms['rescue_dms_mean'].mean():.3f}")
        func = (valid_dms["rescue_dms_mean"] < 0).sum()
        print(f"  Functional rescues (Z<0): {func}/{len(valid_dms)} ({100*func/len(valid_dms):.0f}%)")

# Score stats
if "score" in deep.columns:
    print(f"\nOracle Score Stats:")
    print(f"  Mean: {deep['score'].mean():.4f}")
    print(f"  Max:  {deep['score'].max():.4f}")
    print(f"  Min:  {deep['score'].min():.4f}")

### 5a. Top-30 Shortlist

In [ ]:
from p53cad.results.schema import select_presentation_shortlist

top = select_presentation_shortlist(deep, top_n=SHORTLIST_N)
print(f"Shortlist: {len(top)} candidates\n")

# Build display table
display_rows = []
for _, row in top.iterrows():
    targets = json.loads(str(row.get("targets_json", "[]")))
    muts = json.loads(str(row.get("mutations_json", "[]")))
    rescue = [m for m in muts if m not in targets]
    dms_z = row.get("rescue_dms_mean", None)
    dms_str = f"{dms_z:+.2f}" if dms_z and dms_z != 0 else "N/A"
    pr = row.get("pareto_rank", None)
    pr_str = f"{int(pr)}" if pr and pr != np.inf else "?"
    display_rows.append({
        "Rank": int(row.get("presentation_rank", 0)),
        "Target": row["target_label"],
        "Rescue Mutations": "+".join(rescue),
        "Oracle Score": f"{float(row['score']):.3f}",
        "DMS Z-score": dms_str,
        "Pareto Rank": pr_str,
        "Delivery": row["delivery_method"],
    })

shortlist_df = pd.DataFrame(display_rows)
shortlist_df

In [ ]:
# Delivery method distribution
if len(top):
    delivery_counts = top["delivery_method"].value_counts()
    print("Delivery Distribution:")
    for d, c in delivery_counts.items():
        print(f"  {d}: {c}")

    unique_targets = top["target_label"].nunique()
    print(f"\nUnique target combos: {unique_targets}")

### 5b. Score Distribution Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Oracle score distribution
if "score" in deep.columns:
    axes[0].hist(deep["score"].dropna(), bins=40, color="#2563EB", alpha=0.7, edgecolor="white")
    axes[0].set_title("Oracle Score Distribution")
    axes[0].set_xlabel("Oracle Score")
    axes[0].set_ylabel("Count")

# 2. DMS rescue quality
if "rescue_dms_mean" in deep.columns:
    valid = deep["rescue_dms_mean"].dropna()
    valid = valid[valid != 0]
    if len(valid):
        axes[1].hist(valid, bins=40, color="#10B981", alpha=0.7, edgecolor="white")
        axes[1].axvline(0, color="red", linestyle="--", alpha=0.7)
        axes[1].set_title("DMS Rescue Z-score")
        axes[1].set_xlabel("Z-score")
        axes[1].set_ylabel("Count")

# 3. Scores by target
if "target_label" in deep.columns and "score" in deep.columns:
    # Get single-hotspot targets only for readability
    singles = deep[~deep["target_label"].str.contains(r"\+", na=False)].copy()
    if len(singles):
        target_order = singles.groupby("target_label")["score"].median().sort_values(ascending=False).index
        sns.boxplot(data=singles, x="target_label", y="score", order=target_order,
                    ax=axes[2], palette="Blues_d")
        axes[2].set_title("Score by Hotspot")
        axes[2].set_xlabel("")
        axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 5c. Mutation Position Heatmap

In [ ]:
from p53cad.results.visualization import build_candidate_position_heatmap

# Prepare candidate dicts for heatmap
cand_dicts = []
for _, row in top.iterrows():
    muts = json.loads(str(row.get("mutations_json", "[]")))
    cand_dicts.append({
        "candidate_id": row.get("presentation_rank", 0),
        "profile": row.get("profile", "Unknown"),
        "score": float(row.get("score", 0)),
        "mutations": muts,
    })

matrix_df, freq_df = build_candidate_position_heatmap(cand_dicts)

if not matrix_df.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [3, 1]})

    # Heatmap
    pos_cols = [c for c in matrix_df.columns if c.startswith("pos_")]
    heatmap_data = matrix_df[pos_cols].values
    pos_labels = [c.replace("pos_", "") for c in pos_cols]

    axes[0].imshow(heatmap_data, aspect="auto", cmap="YlOrRd", interpolation="nearest")
    axes[0].set_xticks(range(len(pos_labels)))
    axes[0].set_xticklabels(pos_labels, rotation=90, fontsize=8)
    axes[0].set_ylabel("Candidate")
    axes[0].set_title("Mutation Position Heatmap (Top-30 Shortlist)")

    # Frequency bar chart
    if not freq_df.empty:
        axes[1].bar(range(len(freq_df)), freq_df["frequency"], color="#2563EB", alpha=0.7)
        axes[1].set_xticks(range(len(freq_df)))
        axes[1].set_xticklabels(freq_df["position"].astype(str), rotation=90, fontsize=8)
        axes[1].set_ylabel("Frequency")
        axes[1].set_xlabel("Position")
        axes[1].set_title("Mutation Frequency by Position")

    plt.tight_layout()
    plt.show()
else:
    print("No mutation position data available for heatmap")

### 5d. Clinical Impact

In [ ]:
from p53cad.analysis.clinical_impact import TCGA_P53_MUTATIONS, CANCER_INCIDENCE, P53_MUTATION_RATE
from p53cad.results.schema import BIG8_HOTSPOTS

# Show TCGA hotspot frequencies
print("TCGA p53 Hotspot Mutation Frequencies")
print("=" * 55)
for mut in BIG8_HOTSPOTS:
    info = TCGA_P53_MUTATIONS.get(mut, {})
    freq = info.get("frequency", 0)
    cancers = info.get("cancer_types", [])
    print(f"  {mut:<8}  {freq:>4.1f}%  ->  {', '.join(cancers)}")

# Estimate affected patients
print(f"\nEstimated Annual Patient Impact (US):")
print(f"  {'Cancer':<15} {'Incidence':>10} {'p53 mut rate':>12} {'p53 mutant':>12}")
print(f"  {'-'*15} {'-'*10} {'-'*12} {'-'*12}")
total_p53 = 0
for cancer, incidence in sorted(CANCER_INCIDENCE.items(), key=lambda x: -x[1]):
    rate = P53_MUTATION_RATE.get(cancer, 0)
    p53_cases = int(incidence * rate / 100)
    total_p53 += p53_cases
    print(f"  {cancer:<15} {incidence:>10,} {rate:>11}% {p53_cases:>12,}")
print(f"  {'TOTAL':<15} {'':<10} {'':<12} {total_p53:>12,}")

### 5e. Clinical Impact per Shortlisted Candidate

In [ ]:
# Load clinical impact data if available
clinical_path = os.path.join(run_dir, "clinical.parquet")
if os.path.exists(clinical_path):
    clinical_df = pd.read_parquet(clinical_path)
    print(f"Clinical impact records: {len(clinical_df)}")
    clinical_df.head(10)
else:
    print("Clinical impact data not generated in this run.")
    print("(Set with_clinical=True and budget >= 'medium' for clinical analysis)")

## 6. Export Results

In [ ]:
# Export shortlist to CSV
csv_path = os.path.join(run_dir, "top30.csv")
if os.path.exists(csv_path):
    print(f"Shortlist CSV already saved: {csv_path}")
else:
    shortlist_df.to_csv(csv_path, index=False)
    print(f"Shortlist saved to: {csv_path}")

# Summary
summary_path = os.path.join(run_dir, "summary.md")
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())
else:
    print("\nNo summary.md generated. Key results:")
    print(f"  Run dir: {run_dir}")
    print(f"  Candidates: {result['n_candidates']}")
    print(f"  Shortlist: {result['n_shortlist']}")

In [ ]:
# List all output artifacts
print(f"\nAll artifacts in {run_dir}:")
for f in sorted(os.listdir(run_dir)):
    fpath = os.path.join(run_dir, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        unit = "KB" if size > 1024 else "B"
        size_str = f"{size/1024:.1f} KB" if size > 1024 else f"{size} B"
        print(f"  {f:<35} {size_str}")
    elif os.path.isdir(fpath):
        n_files = len(os.listdir(fpath))
        print(f"  {f + '/':<35} ({n_files} files)")

In [ ]:
# Download results (Colab only)
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    from google.colab import files
    import shutil

    # Create a zip of all results
    results_zip = shutil.make_archive('p53_campaign_results', 'zip', run_dir)
    print(f"\nDownloading results archive: {results_zip}")
    files.download(results_zip)
else:
    print(f"\nResults saved to: {run_dir}")

---

**Done!** The campaign has completed. Key outputs:
- `candidates.parquet` — All evaluated candidates
- `top30.parquet` / `top30.csv` — Shortlisted rescue mutations
- `clinical.parquet` — Patient impact estimates
- `trajectories.parquet` — Optimization trajectories
- `summary.md` — Human-readable report

On Colab, results are automatically zipped and downloaded.